# Core discovery, chronic mouse, and human Müller atlas

Contains the GSE328966 early-mouse discovery analysis, GSE313176 longitudinal mouse validation, and 2026 human Müller atlas analysis. Failed exploratory cells and all execution outputs were removed for repository presentation. The locked signatures and inferential limitations are preserved.

**Important:** These notebooks document the finalized analysis logic. Public raw data are downloaded from the cited repositories; large intermediate objects and outputs are intentionally not embedded in the GitHub version. Where a dataset lacks condition-level biological replication, results are explicitly treated as descriptive.

## Part I — GSE328966 early mouse Müller discovery

In [ ]:
# ============================================================
# DR MÜLLER PROJECT
# Notebook 01 — GSE328966 integrity audit
# ============================================================

!pip -q install scanpy anndata igraph leidenalg

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/DR_Muller_AdaptiveReactive")
CACHE = Path("/content/DR_Muller_cache")

dirs = [
    ROOT / "00_manifest",
    ROOT / "01_GSE328966_early_mouse",
    ROOT / "02_Human_scRNA_2026",
    ROOT / "03_GSE160306_human_validation",
    ROOT / "04_GSE313176_mouse_longitudinal",
    ROOT / "05_GSE290024_organoid",
    ROOT / "signatures",
    ROOT / "figures",
    ROOT / "tables",
    ROOT / "logs",
    CACHE / "GSE328966",
]

for directory in dirs:
    directory.mkdir(parents=True, exist_ok=True)

manifest = pd.DataFrame([
    {
        "gsm": "GSM9695016",
        "sample": "Control5D",
        "condition": "Control",
        "day": 5,
        "pooled_mice": 3,
        "independent_library_n": 1,
        "file_prefix": "GSM9695016_Control5D"
    },
    {
        "gsm": "GSM9695017",
        "sample": "Control15D",
        "condition": "Control",
        "day": 15,
        "pooled_mice": 3,
        "independent_library_n": 1,
        "file_prefix": "GSM9695017_Control15D"
    },
    {
        "gsm": "GSM9695018",
        "sample": "DM5D",
        "condition": "DM",
        "day": 5,
        "pooled_mice": 3,
        "independent_library_n": 1,
        "file_prefix": "GSM9695018_DM5D"
    },
    {
        "gsm": "GSM9695019",
        "sample": "DM15D",
        "condition": "DM",
        "day": 15,
        "pooled_mice": 3,
        "independent_library_n": 1,
        "file_prefix": "GSM9695019_DM15D"
    },
])

manifest_path = ROOT / "00_manifest" / "GSE328966_samples.csv"
manifest.to_csv(manifest_path, index=False)

display(manifest)

print("\nIMPORTANT:")
print(
    "Each condition contains ONE pooled scRNA library derived from 3 mice. "
    "Cells are NOT independent animal replicates."
)

In [ ]:
import subprocess
import tarfile

DOWNLOAD_DIR = CACHE / "GSE328966"
EXTRACT_DIR = DOWNLOAD_DIR / "raw"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

tar_path = DOWNLOAD_DIR / "GSE328966_RAW.tar"

url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE328nnn/GSE328966/suppl/GSE328966_RAW.tar"
)

if not tar_path.exists():
    print("Downloading GSE328966 processed matrices...")
    subprocess.run(
        ["wget", "-c", "-O", str(tar_path), url],
        check=True
    )
else:
    print("Archive already exists:", tar_path)

if len(list(EXTRACT_DIR.glob("*matrix.mtx.gz"))) == 0:
    print("Extracting archive...")
    with tarfile.open(tar_path, "r") as tar:
        tar.extractall(EXTRACT_DIR)
else:
    print("Files already extracted.")

matrix_files = sorted(EXTRACT_DIR.glob("*matrix.mtx.gz"))
feature_files = sorted(EXTRACT_DIR.glob("*features.tsv.gz"))
barcode_files = sorted(EXTRACT_DIR.glob("*barcodes.tsv.gz"))

print("\nMatrix files:", len(matrix_files))
print("Feature files:", len(feature_files))
print("Barcode files:", len(barcode_files))

for f in sorted(EXTRACT_DIR.iterdir()):
    print(f.name)

assert len(matrix_files) == 4, "Expected 4 matrices"
assert len(feature_files) == 4, "Expected 4 feature files"
assert len(barcode_files) == 4, "Expected 4 barcode files"

print("\nFILE AUDIT PASSED.")

In [ ]:
import gzip
import scanpy as sc
import anndata as ad

from scipy.io import mmread

def read_geo_10x(prefix, sample, condition, day):

    matrix_path = EXTRACT_DIR / f"{prefix}_matrix.mtx.gz"
    features_path = EXTRACT_DIR / f"{prefix}_features.tsv.gz"
    barcodes_path = EXTRACT_DIR / f"{prefix}_barcodes.tsv.gz"

    # Matrix Market files are genes x cells -> transpose to cells x genes
    with gzip.open(matrix_path, "rb") as fh:
        X = mmread(fh).T.tocsr()

    features = pd.read_csv(
        features_path,
        sep="\t",
        header=None
    )

    barcodes = pd.read_csv(
        barcodes_path,
        sep="\t",
        header=None
    )[0].astype(str).values

    # Cell Ranger features.tsv usually contains:
    # column 0 = gene ID
    # column 1 = gene symbol
    gene_ids = features.iloc[:, 0].astype(str).values

    if features.shape[1] >= 2:
        gene_symbols = features.iloc[:, 1].astype(str).values
    else:
        gene_symbols = gene_ids

    adata = ad.AnnData(X=X)

    adata.var["gene_id"] = gene_ids
    adata.var_names = gene_symbols
    adata.var_names_make_unique()

    # Make cell names globally unique
    adata.obs_names = [
        f"{sample}:{barcode}"
        for barcode in barcodes
    ]

    adata.obs["sample"] = sample
    adata.obs["condition"] = condition
    adata.obs["day"] = str(day)

    return adata


adatas = []

for _, row in manifest.iterrows():

    x = read_geo_10x(
        prefix=row["file_prefix"],
        sample=row["sample"],
        condition=row["condition"],
        day=int(row["day"])
    )

    print(
        row["sample"],
        "cells =", x.n_obs,
        "genes =", x.n_vars
    )

    adatas.append(x)

adata = ad.concat(
    adatas,
    join="inner",
    merge="same"
)

print("\nCOMBINED DATA")
print(adata)
print("\nTotal cells:", adata.n_obs)
print("Total genes:", adata.n_vars)

In [ ]:
# Identify mitochondrial genes
adata.var["mt"] = (
    adata.var_names.str.startswith("mt-") |
    adata.var_names.str.startswith("MT-")
)

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    percent_top=None,
    log1p=False,
    inplace=True
)

# Counts before QC
before = (
    adata.obs
    .groupby("sample", observed=True)
    .size()
    .rename("raw_cells")
)

# Paper-compatible filtering
keep = (
    (adata.obs["n_genes_by_counts"] >= 200) &
    (adata.obs["n_genes_by_counts"] <= 8000) &
    (adata.obs["total_counts"] >= 500) &
    (adata.obs["total_counts"] <= 25000) &
    (adata.obs["pct_counts_mt"] < 5)
)

adata_qc = adata[keep].copy()

after = (
    adata_qc.obs
    .groupby("sample", observed=True)
    .size()
    .rename("post_qc_cells")
)

qc_audit = pd.concat(
    [before, after],
    axis=1
)

qc_audit["removed_cells"] = (
    qc_audit["raw_cells"] -
    qc_audit["post_qc_cells"]
)

qc_audit["retained_pct"] = (
    100 *
    qc_audit["post_qc_cells"] /
    qc_audit["raw_cells"]
)

display(qc_audit)

print("\nTOTAL RAW CELLS:", f"{adata.n_obs:,}")
print("TOTAL POST-QC CELLS:", f"{adata_qc.n_obs:,}")

qc_audit.to_csv(
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_QC_audit.csv"
)

adata_qc.write_h5ad(
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_QC.h5ad",
    compression="gzip"
)

print("\nSaved QC object.")

In [ ]:
# Preserve raw count matrix
adata_qc.layers["counts"] = adata_qc.X.copy()

# Standard normalization
sc.pp.normalize_total(
    adata_qc,
    target_sum=1e4
)

sc.pp.log1p(adata_qc)

# Highly variable genes
sc.pp.highly_variable_genes(
    adata_qc,
    n_top_genes=3000,
    flavor="seurat"
)

# PCA
sc.tl.pca(
    adata_qc,
    n_comps=50,
    use_highly_variable=True
)

# Neighborhood graph
sc.pp.neighbors(
    adata_qc,
    n_neighbors=15,
    n_pcs=20
)

# Match the published clustering resolution initially
sc.tl.leiden(
    adata_qc,
    resolution=0.6,
    key_added="leiden"
)

sc.tl.umap(adata_qc)

sc.pl.umap(
    adata_qc,
    color=[
        "sample",
        "condition",
        "day",
        "leiden"
    ],
    ncols=2,
    frameon=False
)

In [ ]:
marker_sets = {

    "Muller": [
        "Rlbp1",
        "Glul",
        "Slc1a3",
        "Aqp4",
        "Kcnj10"
    ],

    "Rod": [
        "Rho",
        "Nrl",
        "Gnat1",
        "Pde6a"
    ],

    "Cone": [
        "Arr3",
        "Gnat2",
        "Opn1sw",
        "Opn1mw"
    ],

    "Bipolar": [
        "Vsx2",
        "Grm6",
        "Prkca"
    ],

    "RGC": [
        "Rbpms",
        "Pou4f1",
        "Pou4f2",
        "Slc17a6"
    ],

    "Amacrine": [
        "Gad1",
        "Gad2",
        "Tfap2a"
    ],

    "Microglia": [
        "P2ry12",
        "Cx3cr1",
        "Aif1",
        "C1qa"
    ],

    "Astrocyte": [
        "Gfap",
        "Aldoc",
        "S100b"
    ]
}

usable_markers = {}

for celltype, genes in marker_sets.items():

    present = [
        gene
        for gene in genes
        if gene in adata_qc.var_names
    ]

    usable_markers[celltype] = present

    print(
        celltype,
        ":",
        present
    )

In [ ]:
for celltype, genes in usable_markers.items():

    if len(genes) >= 2:
        sc.tl.score_genes(
            adata_qc,
            gene_list=genes,
            score_name=f"{celltype}_score"
        )

score_cols = [
    x
    for x in adata_qc.obs.columns
    if x.endswith("_score")
]

cluster_scores = (
    adata_qc.obs
    .groupby("leiden", observed=True)[score_cols]
    .mean()
)

display(cluster_scores)

cluster_scores.to_csv(
    ROOT /
    "01_GSE328966_early_mouse" /
    "cluster_marker_scores.csv"
)

In [ ]:
sc.pl.dotplot(
    adata_qc,
    var_names=usable_markers,
    groupby="leiden",
    standard_scale="var"
)

In [ ]:
# ============================================================
# CELL 9 — QC + MÜLLER IDENTITY REPRODUCIBILITY AUDIT
# ============================================================

print("=" * 70)
print("A. OUR DATA VS PUBLISHED STUDY")
print("=" * 70)

print(f"Raw cells:       {adata.n_obs:,}")
print(f"Our post-QC:     {adata_qc.n_obs:,}")
print("Published QC:    20,759")
print(f"Difference:      {adata_qc.n_obs - 20759:+,}")

print("\nOur retained fraction:")
print(f"{100 * adata_qc.n_obs / adata.n_obs:.2f}%")

print("\nPublished retained fraction:")
print(f"{100 * 20759 / 23901:.2f}%")


# ------------------------------------------------------------
# B. CLUSTER SIZE + SAMPLE COMPOSITION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("B. LEIDEN CLUSTER COUNTS")
print("=" * 70)

cluster_counts = (
    adata_qc.obs["leiden"]
    .value_counts()
    .sort_index(key=lambda x: x.astype(int))
    .rename("n_cells")
)

cluster_pct = (
    100 * cluster_counts / adata_qc.n_obs
).rename("pct_total")

cluster_summary = pd.concat(
    [cluster_counts, cluster_pct],
    axis=1
)

display(cluster_summary)


print("\n" + "=" * 70)
print("C. CELLS PER SAMPLE WITHIN EACH CLUSTER")
print("=" * 70)

cluster_sample = pd.crosstab(
    adata_qc.obs["leiden"],
    adata_qc.obs["sample"]
)

# Sort clusters numerically
cluster_sample = cluster_sample.loc[
    sorted(cluster_sample.index, key=int)
]

display(cluster_sample)


print("\n" + "=" * 70)
print("D. CANDIDATE MÜLLER CLUSTERS: 5 AND 19")
print("=" * 70)

candidate_table = cluster_sample.loc[
    [x for x in ["5", "19"] if x in cluster_sample.index]
].copy()

candidate_table["TOTAL"] = candidate_table.sum(axis=1)

display(candidate_table)

for cluster in ["5", "19"]:
    if cluster in adata_qc.obs["leiden"].unique():
        n = (adata_qc.obs["leiden"] == cluster).sum()
        pct = 100 * n / adata_qc.n_obs

        print(
            f"Cluster {cluster}: "
            f"{n:,} cells = {pct:.2f}% of post-QC retina"
        )


# ------------------------------------------------------------
# E. QC CHARACTERISTICS OF CLUSTERS 5 AND 19
# ------------------------------------------------------------

qc_cols = [
    "n_genes_by_counts",
    "total_counts",
    "pct_counts_mt"
]

candidate_qc = (
    adata_qc.obs[
        adata_qc.obs["leiden"].isin(["5", "19"])
    ]
    .groupby("leiden", observed=True)[qc_cols]
    .median()
)

print("\nMedian QC metrics:")
display(candidate_qc)


# ------------------------------------------------------------
# F. EXISTING BIOLOGICAL SCORES
# ------------------------------------------------------------

candidate_scores = (
    adata_qc.obs[
        adata_qc.obs["leiden"].isin(["5", "19"])
    ]
    .groupby("leiden", observed=True)[
        [
            "Muller_score",
            "Rod_score",
            "Cone_score",
            "Bipolar_score",
            "Microglia_score",
            "Astrocyte_score"
        ]
    ]
    .mean()
)

print("\nMean identity scores:")
display(candidate_scores)


# ------------------------------------------------------------
# G. VISUALIZE MÜLLER SCORE AND CANDIDATE CLUSTERS
# ------------------------------------------------------------

sc.pl.umap(
    adata_qc,
    color=[
        "leiden",
        "Muller_score",
        "Rod_score"
    ],
    ncols=3,
    frameon=False
)

print("\nAudit complete.")

In [ ]:
# ============================================================
# CELL 10 — DATA-DRIVEN IDENTITY CHECK FOR CLUSTERS 5 AND 19
# ============================================================

# Exploratory only:
# these statistics are used for CELL-TYPE IDENTIFICATION,
# not for DM-vs-control biological inference.

sc.tl.rank_genes_groups(
    adata_qc,
    groupby="leiden",
    groups=["5", "19"],
    reference="rest",
    method="wilcoxon",
    key_added="cluster_identity_rank"
)

for cluster in ["5", "19"]:

    df = sc.get.rank_genes_groups_df(
        adata_qc,
        group=cluster,
        key="cluster_identity_rank"
    )

    print("\n" + "=" * 70)
    print(f"TOP MARKERS — CLUSTER {cluster}")
    print("=" * 70)

    display(
        df[
            [
                "names",
                "logfoldchanges",
                "pvals_adj"
            ]
        ].head(30)
    )


# Direct visualization of the two candidate populations
genes_to_check = [
    "Rlbp1",
    "Glul",
    "Slc1a3",
    "Aqp4",
    "Kcnj10",
    "Rho",
    "Nrl",
    "Gnat1"
]

genes_to_check = [
    g for g in genes_to_check
    if g in adata_qc.var_names
]

sc.pl.violin(
    adata_qc[
        adata_qc.obs["leiden"].isin(["5", "19"])
    ],
    keys=genes_to_check,
    groupby="leiden",
    stripplot=False,
    multi_panel=True
)

In [ ]:
# ============================================================
# NEXT CELL — FREEZE CLEAN MÜLLER POPULATION
# ============================================================

# Müller glia = Leiden cluster 5 only
adata_muller = adata_qc[
    adata_qc.obs["leiden"] == "5"
].copy()

print("=" * 70)
print("FROZEN MÜLLER DATASET")
print("=" * 70)

print(f"\nCells: {adata_muller.n_obs:,}")
print(f"Genes: {adata_muller.n_vars:,}")

# ------------------------------------------------------------
# Sample composition
# ------------------------------------------------------------

sample_counts = (
    adata_muller.obs["sample"]
    .value_counts()
    .reindex([
        "Control5D",
        "DM5D",
        "Control15D",
        "DM15D"
    ])
)

sample_pct = 100 * sample_counts / sample_counts.sum()

muller_composition = pd.DataFrame({
    "n_cells": sample_counts,
    "pct_muller_cells": sample_pct
})

print("\nMüller cells by experimental group:")
display(muller_composition)

# ------------------------------------------------------------
# Confirm raw counts were preserved
# ------------------------------------------------------------

print("\nAvailable layers:", list(adata_muller.layers.keys()))

assert "counts" in adata_muller.layers.keys(), (
    "Raw counts layer is missing — STOP before continuing."
)

# ------------------------------------------------------------
# Sanity-check Müller markers
# ------------------------------------------------------------

muller_markers = [
    "Rlbp1",
    "Glul",
    "Slc1a3",
    "Aqp4",
    "Kcnj10",
    "Clu",
    "Dkk3"
]

present_markers = [
    g for g in muller_markers
    if g in adata_muller.var_names
]

print("\nCanonical Müller markers present:")
print(present_markers)

# ------------------------------------------------------------
# Save frozen object
# ------------------------------------------------------------

out_path = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_cluster5_FROZEN.h5ad"
)

adata_muller.write_h5ad(
    out_path,
    compression="gzip"
)

print("\nSaved:")
print(out_path)

print("\nMÜLLER OBJECT FROZEN SUCCESSFULLY.")

In [ ]:
# ============================================================
# NEXT CELL — MÜLLER PSEUDOBULK EFFECT-SIZE TABLE
# Descriptive only: one pooled library per condition
# ============================================================

import numpy as np
import pandas as pd
from scipy import sparse

sample_order = [
    "Control5D",
    "DM5D",
    "Control15D",
    "DM15D"
]

genes = adata_muller.var_names.to_numpy()
X_counts = adata_muller.layers["counts"]

# ------------------------------------------------------------
# 1. Aggregate raw counts within each pooled library
# ------------------------------------------------------------

pb_counts = {}
det_frac = {}

for sample in sample_order:

    mask = (
        adata_muller.obs["sample"]
        .astype(str)
        .to_numpy() == sample
    )

    Xs = X_counts[mask]

    # summed counts per gene
    pb_counts[sample] = np.asarray(
        Xs.sum(axis=0)
    ).ravel()

    # fraction of Müller cells expressing each gene
    if sparse.issparse(Xs):
        det_frac[sample] = np.asarray(
            (Xs > 0).mean(axis=0)
        ).ravel()
    else:
        det_frac[sample] = (
            Xs > 0
        ).mean(axis=0)

pb_counts = pd.DataFrame(
    pb_counts,
    index=genes
)

det_frac = pd.DataFrame(
    det_frac,
    index=genes
)

# ------------------------------------------------------------
# 2. Library-size audit
# ------------------------------------------------------------

library_sizes = pb_counts.sum(axis=0)

print("=" * 70)
print("PSEUDOBULK LIBRARY SIZES")
print("=" * 70)

display(
    pd.DataFrame({
        "total_UMI": library_sizes,
        "muller_cells": [
            (adata_muller.obs["sample"].astype(str) == s).sum()
            for s in sample_order
        ]
    })
)

# ------------------------------------------------------------
# 3. CPM normalization
# ------------------------------------------------------------

pb_cpm = (
    pb_counts
    .div(library_sizes, axis=1)
    * 1_000_000
)

# Small prior only to avoid infinite fold changes
prior = 0.5

early_log2fc = np.log2(
    (pb_cpm["DM5D"] + prior) /
    (pb_cpm["Control5D"] + prior)
)

late_log2fc = np.log2(
    (pb_cpm["DM15D"] + prior) /
    (pb_cpm["Control15D"] + prior)
)

# ------------------------------------------------------------
# 4. Build complete descriptive effect table
# ------------------------------------------------------------

effect_table = pd.DataFrame(index=genes)

effect_table["early_log2FC"] = early_log2fc
effect_table["late_log2FC"] = late_log2fc

for sample in sample_order:
    effect_table[f"{sample}_CPM"] = pb_cpm[sample]
    effect_table[f"{sample}_detect"] = det_frac[sample]

# Expression filter for later ranking:
# gene must be detected in >=10% of Müller cells
# in at least one experimental group
max_detection = det_frac.max(axis=1)

effect_table["max_detection"] = max_detection
effect_table["eligible_for_ranking"] = (
    effect_table["max_detection"] >= 0.10
)

print("\nGenes total:", f"{len(effect_table):,}")
print(
    "Genes eligible for ranking:",
    f"{effect_table['eligible_for_ranking'].sum():,}"
)

# ------------------------------------------------------------
# 5. Inspect known biological anchors
# ------------------------------------------------------------

anchors = [
    "Zfp36",
    "Mt1",
    "Slc14a1",
    "Gfap",
    "Vim",
    "Glul",
    "Rlbp1",
    "Slc1a3"
]

anchors_present = [
    g for g in anchors
    if g in effect_table.index
]

print("\n" + "=" * 70)
print("KNOWN / REFERENCE MÜLLER GENES")
print("=" * 70)

display(
    effect_table.loc[
        anchors_present,
        [
            "early_log2FC",
            "late_log2FC",
            "Control5D_detect",
            "DM5D_detect",
            "Control15D_detect",
            "DM15D_detect"
        ]
    ].round(3)
)

# ------------------------------------------------------------
# 6. Show strongest descriptive early changes
# ------------------------------------------------------------

rankable = effect_table[
    effect_table["eligible_for_ranking"]
].copy()

print("\n" + "=" * 70)
print("TOP EARLY INCREASES — DM5D vs Control5D")
print("=" * 70)

display(
    rankable
    .sort_values("early_log2FC", ascending=False)
    [
        [
            "early_log2FC",
            "late_log2FC",
            "Control5D_detect",
            "DM5D_detect"
        ]
    ]
    .head(15)
    .round(3)
)

print("\n" + "=" * 70)
print("TOP EARLY DECREASES — DM5D vs Control5D")
print("=" * 70)

display(
    rankable
    .sort_values("early_log2FC", ascending=True)
    [
        [
            "early_log2FC",
            "late_log2FC",
            "Control5D_detect",
            "DM5D_detect"
        ]
    ]
    .head(15)
    .round(3)
)

# ------------------------------------------------------------
# 7. Save complete table
# ------------------------------------------------------------

out_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_descriptive_effects.csv"
)

effect_table.to_csv(out_file)

print("\nSaved:")
print(out_file)

print("\nIMPORTANT:")
print(
    "These are descriptive pooled-library effect sizes, "
    "NOT replicated differential-expression statistics."
)

In [ ]:
# ============================================================
# NEXT CELL — ANCHOR EXPRESSION DECOMPOSITION
# prevalence vs expression intensity
# ============================================================

import numpy as np
import pandas as pd
from scipy import sparse

anchors = [
    "Zfp36",
    "Mt1",
    "Slc14a1",
    "Gfap",
    "Vim",
    "Glul",
    "Rlbp1",
    "Slc1a3"
]

sample_order = [
    "Control5D",
    "DM5D",
    "Control15D",
    "DM15D"
]

rows = []

for gene in anchors:

    if gene not in adata_muller.var_names:
        continue

    gene_idx = adata_muller.var_names.get_loc(gene)

    for sample in sample_order:

        mask = (
            adata_muller.obs["sample"]
            .astype(str)
            .to_numpy() == sample
        )

        # Raw counts
        x_raw = adata_muller.layers["counts"][mask, gene_idx]

        if sparse.issparse(x_raw):
            x_raw = np.asarray(x_raw.toarray()).ravel()
        else:
            x_raw = np.asarray(x_raw).ravel()

        # Normalized log1p expression stored in X
        x_norm = adata_muller.X[mask, gene_idx]

        if sparse.issparse(x_norm):
            x_norm = np.asarray(x_norm.toarray()).ravel()
        else:
            x_norm = np.asarray(x_norm).ravel()

        positive = x_raw > 0

        rows.append({
            "gene": gene,
            "sample": sample,
            "n_cells": len(x_raw),
            "n_positive": positive.sum(),
            "detect_frac": positive.mean(),
            "raw_counts_total": x_raw.sum(),
            "raw_mean_per_cell": x_raw.mean(),
            "raw_mean_positive_cells": (
                x_raw[positive].mean()
                if positive.sum() > 0 else np.nan
            ),
            "norm_mean_all_cells": x_norm.mean(),
            "norm_mean_positive_cells": (
                x_norm[positive].mean()
                if positive.sum() > 0 else np.nan
            )
        })

anchor_detail = pd.DataFrame(rows)

print("=" * 80)
print("ANCHOR EXPRESSION DECOMPOSITION")
print("=" * 80)

display(
    anchor_detail.round(4)
)

# Easier visual comparison
pivot_detect = anchor_detail.pivot(
    index="gene",
    columns="sample",
    values="detect_frac"
)[sample_order]

pivot_intensity = anchor_detail.pivot(
    index="gene",
    columns="sample",
    values="norm_mean_positive_cells"
)[sample_order]

print("\n" + "=" * 80)
print("A. FRACTION OF MÜLLER CELLS EXPRESSING EACH GENE")
print("=" * 80)

display(pivot_detect.round(3))

print("\n" + "=" * 80)
print("B. NORMALIZED EXPRESSION AMONG POSITIVE CELLS")
print("=" * 80)

display(pivot_intensity.round(3))

# Save
out_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_anchor_decomposition.csv"
)

anchor_detail.to_csv(out_file, index=False)

print("\nSaved:")
print(out_file)

In [ ]:
# ============================================================
# NEXT CELL — WHOLE-TRANSCRIPTOME RESPONSE DECOMPOSITION
# Separate abundance, mean expression, and prevalence effects
# ============================================================

import numpy as np
import pandas as pd
from scipy import sparse

sample_order = [
    "Control5D",
    "DM5D",
    "Control15D",
    "DM15D"
]

genes = adata_muller.var_names.to_numpy()

# ------------------------------------------------------------
# 1. Calculate mean normalized expression and prevalence
#    for EVERY gene in each condition
# ------------------------------------------------------------

mean_norm = {}
detect_frac_all = {}

for sample in sample_order:

    mask = (
        adata_muller.obs["sample"]
        .astype(str)
        .to_numpy() == sample
    )

    # normalized/log1p expression
    Xn = adata_muller.X[mask]

    # raw counts
    Xr = adata_muller.layers["counts"][mask]

    if sparse.issparse(Xn):
        mean_norm[sample] = np.asarray(
            Xn.mean(axis=0)
        ).ravel()
    else:
        mean_norm[sample] = np.asarray(
            Xn.mean(axis=0)
        ).ravel()

    if sparse.issparse(Xr):
        detect_frac_all[sample] = np.asarray(
            (Xr > 0).mean(axis=0)
        ).ravel()
    else:
        detect_frac_all[sample] = (
            Xr > 0
        ).mean(axis=0)

mean_norm = pd.DataFrame(
    mean_norm,
    index=genes
)

detect_frac_all = pd.DataFrame(
    detect_frac_all,
    index=genes
)

# ------------------------------------------------------------
# 2. Add complementary response metrics
# ------------------------------------------------------------

response = effect_table.copy()

# Mean normalized-expression differences
response["early_norm_delta"] = (
    mean_norm["DM5D"] -
    mean_norm["Control5D"]
)

response["late_norm_delta"] = (
    mean_norm["DM15D"] -
    mean_norm["Control15D"]
)

# Prevalence differences
response["early_detect_delta"] = (
    detect_frac_all["DM5D"] -
    detect_frac_all["Control5D"]
)

response["late_detect_delta"] = (
    detect_frac_all["DM15D"] -
    detect_frac_all["Control15D"]
)

# ------------------------------------------------------------
# 3. Direction agreement
# ------------------------------------------------------------

def direction(x, tol=1e-12):
    if x > tol:
        return 1
    if x < -tol:
        return -1
    return 0

response["early_fc_dir"] = (
    response["early_log2FC"]
    .apply(direction)
)

response["early_norm_dir"] = (
    response["early_norm_delta"]
    .apply(direction)
)

response["early_detect_dir"] = (
    response["early_detect_delta"]
    .apply(direction)
)

response["late_fc_dir"] = (
    response["late_log2FC"]
    .apply(direction)
)

response["late_norm_dir"] = (
    response["late_norm_delta"]
    .apply(direction)
)

response["late_detect_dir"] = (
    response["late_detect_delta"]
    .apply(direction)
)

# All three metrics point in same direction
response["early_all_concordant"] = (
    (response["early_fc_dir"] == response["early_norm_dir"]) &
    (response["early_fc_dir"] == response["early_detect_dir"]) &
    (response["early_fc_dir"] != 0)
)

response["late_all_concordant"] = (
    (response["late_fc_dir"] == response["late_norm_dir"]) &
    (response["late_fc_dir"] == response["late_detect_dir"]) &
    (response["late_fc_dir"] != 0)
)

# ------------------------------------------------------------
# 4. Prevalence-driven response flag
#
# Strong change in fraction of expressing cells,
# but relatively small change in mean normalized expression.
# This is descriptive, NOT a biological classification yet.
# ------------------------------------------------------------

response["early_prevalence_shift"] = (
    response["early_detect_delta"].abs() >= 0.10
)

response["late_prevalence_shift"] = (
    response["late_detect_delta"].abs() >= 0.10
)

# ------------------------------------------------------------
# 5. Audit known anchors
# ------------------------------------------------------------

anchors = [
    "Zfp36",
    "Mt1",
    "Slc14a1",
    "Gfap",
    "Vim",
    "Glul",
    "Rlbp1",
    "Slc1a3"
]

anchors_present = [
    g for g in anchors
    if g in response.index
]

cols = [
    "early_log2FC",
    "early_norm_delta",
    "early_detect_delta",
    "early_all_concordant",
    "late_log2FC",
    "late_norm_delta",
    "late_detect_delta",
    "late_all_concordant"
]

print("=" * 90)
print("ANCHOR RESPONSE ACROSS THREE METRICS")
print("=" * 90)

display(
    response.loc[
        anchors_present,
        cols
    ].round(4)
)

# ------------------------------------------------------------
# 6. Restrict inspection to reasonably detected genes
# ------------------------------------------------------------

rankable = response[
    response["eligible_for_ranking"]
].copy()

print("\n" + "=" * 90)
print("EARLY RESPONSE — ALL 3 METRICS AGREE ON INCREASE")
print("=" * 90)

early_up = rankable[
    rankable["early_all_concordant"] &
    (rankable["early_fc_dir"] == 1)
].sort_values(
    "early_log2FC",
    ascending=False
)

display(
    early_up[
        [
            "early_log2FC",
            "early_norm_delta",
            "early_detect_delta",
            "late_log2FC",
            "late_norm_delta",
            "late_detect_delta"
        ]
    ].head(20).round(4)
)

print("\nNumber of concordant early increases:", len(early_up))


print("\n" + "=" * 90)
print("EARLY RESPONSE — ALL 3 METRICS AGREE ON DECREASE")
print("=" * 90)

early_down = rankable[
    rankable["early_all_concordant"] &
    (rankable["early_fc_dir"] == -1)
].sort_values(
    "early_log2FC",
    ascending=True
)

display(
    early_down[
        [
            "early_log2FC",
            "early_norm_delta",
            "early_detect_delta",
            "late_log2FC",
            "late_norm_delta",
            "late_detect_delta"
        ]
    ].head(20).round(4)
)

print("\nNumber of concordant early decreases:", len(early_down))


# ------------------------------------------------------------
# 7. Largest early prevalence shifts
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("LARGEST EARLY GAINS IN FRACTION OF EXPRESSING MÜLLER CELLS")
print("=" * 90)

display(
    rankable.sort_values(
        "early_detect_delta",
        ascending=False
    )[
        [
            "early_detect_delta",
            "early_norm_delta",
            "early_log2FC",
            "late_detect_delta",
            "late_log2FC"
        ]
    ].head(20).round(4)
)

# ------------------------------------------------------------
# 8. Save
# ------------------------------------------------------------

out_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_response_decomposition.csv"
)

response.to_csv(out_file)

print("\nSaved:")
print(out_file)

print(
    "\nNo statistical significance is inferred here; "
    "this is descriptive response decomposition."
)

In [ ]:
# ============================================================
# NEXT CELL — MÜLLER SAMPLE-LEVEL DEPTH / COMPLEXITY AUDIT
# Check whether early prevalence shifts are driven by
# sequencing depth or RNA complexity differences
# ============================================================

import pandas as pd
import numpy as np

qc_vars = [
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt"
]

sample_order = [
    "Control5D",
    "DM5D",
    "Control15D",
    "DM15D"
]

# ------------------------------------------------------------
# 1. Per-sample descriptive QC
# ------------------------------------------------------------

rows = []

for sample in sample_order:

    sub = adata_muller.obs[
        adata_muller.obs["sample"].astype(str) == sample
    ]

    rows.append({
        "sample": sample,
        "n_cells": len(sub),

        "UMI_mean": sub["total_counts"].mean(),
        "UMI_median": sub["total_counts"].median(),
        "UMI_q25": sub["total_counts"].quantile(0.25),
        "UMI_q75": sub["total_counts"].quantile(0.75),

        "genes_mean": sub["n_genes_by_counts"].mean(),
        "genes_median": sub["n_genes_by_counts"].median(),
        "genes_q25": sub["n_genes_by_counts"].quantile(0.25),
        "genes_q75": sub["n_genes_by_counts"].quantile(0.75),

        "mt_mean": sub["pct_counts_mt"].mean(),
        "mt_median": sub["pct_counts_mt"].median(),
    })

qc_by_sample = pd.DataFrame(rows).set_index("sample")

print("=" * 85)
print("MÜLLER QC BY SAMPLE")
print("=" * 85)

display(qc_by_sample.round(2))


# ------------------------------------------------------------
# 2. Relative depth differences
# ------------------------------------------------------------

ctrl5_median = qc_by_sample.loc["Control5D", "UMI_median"]
dm5_median = qc_by_sample.loc["DM5D", "UMI_median"]

ctrl15_median = qc_by_sample.loc["Control15D", "UMI_median"]
dm15_median = qc_by_sample.loc["DM15D", "UMI_median"]

print("\n" + "=" * 85)
print("RELATIVE MEDIAN UMI DEPTH")
print("=" * 85)

print(
    f"DM5D / Control5D median UMI ratio: "
    f"{dm5_median / ctrl5_median:.3f}"
)

print(
    f"DM15D / Control15D median UMI ratio: "
    f"{dm15_median / ctrl15_median:.3f}"
)


# ------------------------------------------------------------
# 3. Median detected-gene ratios
# ------------------------------------------------------------

ctrl5_genes = qc_by_sample.loc["Control5D", "genes_median"]
dm5_genes = qc_by_sample.loc["DM5D", "genes_median"]

ctrl15_genes = qc_by_sample.loc["Control15D", "genes_median"]
dm15_genes = qc_by_sample.loc["DM15D", "genes_median"]

print("\n" + "=" * 85)
print("RELATIVE MEDIAN GENE COMPLEXITY")
print("=" * 85)

print(
    f"DM5D / Control5D median detected-gene ratio: "
    f"{dm5_genes / ctrl5_genes:.3f}"
)

print(
    f"DM15D / Control15D median detected-gene ratio: "
    f"{dm15_genes / ctrl15_genes:.3f}"
)


# ------------------------------------------------------------
# 4. Distribution plots
# ------------------------------------------------------------

sc.pl.violin(
    adata_muller,
    keys=[
        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt"
    ],
    groupby="sample",
    rotation=45,
    stripplot=False,
    multi_panel=True
)


# ------------------------------------------------------------
# 5. Save audit
# ------------------------------------------------------------

out_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_sample_QC_audit.csv"
)

qc_by_sample.to_csv(out_file)

print("\nSaved:")
print(out_file)

In [ ]:
# ============================================================
# NEXT CELL — DEPTH-STANDARDIZED EARLY (5D) SENSITIVITY ANALYSIS
# Every retained Müller cell is downsampled to exactly 5,000 UMIs
# ============================================================

import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

TARGET_UMI = 5000
RANDOM_SEED = 42

# ------------------------------------------------------------
# 1. Take only 5-day Müller cells
# ------------------------------------------------------------

early_mask = (
    adata_muller.obs["sample"]
    .astype(str)
    .isin(["Control5D", "DM5D"])
)

early = adata_muller[early_mask].copy()

# Use raw counts as X for downsampling
early.X = early.layers["counts"].copy()

# ------------------------------------------------------------
# 2. Keep only cells capable of reaching the common depth
# ------------------------------------------------------------

raw_depth = np.asarray(
    early.X.sum(axis=1)
).ravel()

eligible = raw_depth >= TARGET_UMI

early_bal = early[eligible].copy()

print("=" * 85)
print("EARLY DEPTH-STANDARDIZATION")
print("=" * 85)

print(f"\nTarget depth: {TARGET_UMI:,} UMIs/cell")

before_counts = (
    early.obs["sample"]
    .astype(str)
    .value_counts()
    .reindex(["Control5D", "DM5D"])
)

after_counts = (
    early_bal.obs["sample"]
    .astype(str)
    .value_counts()
    .reindex(["Control5D", "DM5D"])
)

retention = pd.DataFrame({
    "before": before_counts,
    "retained_for_matching": after_counts
})

retention["retained_pct"] = (
    100 *
    retention["retained_for_matching"] /
    retention["before"]
)

display(retention.round(2))

# ------------------------------------------------------------
# 3. Downsample every retained cell to exactly 5,000 counts
# ------------------------------------------------------------

sc.pp.downsample_counts(
    early_bal,
    counts_per_cell=TARGET_UMI,
    random_state=RANDOM_SEED
)

depth_after = np.asarray(
    early_bal.X.sum(axis=1)
).ravel()

print("\nPost-downsampling depth:")
print(
    pd.Series(depth_after)
    .describe()
    .round(2)
)

# Sanity check
assert depth_after.min() == TARGET_UMI
assert depth_after.max() == TARGET_UMI

print("\nAll retained cells now have exactly 5,000 UMIs.")

# ------------------------------------------------------------
# 4. Recalculate prevalence + aggregate counts
# ------------------------------------------------------------

genes = early_bal.var_names.to_numpy()

balanced_counts = {}
balanced_detect = {}

for sample in ["Control5D", "DM5D"]:

    mask = (
        early_bal.obs["sample"]
        .astype(str)
        .to_numpy() == sample
    )

    Xs = early_bal.X[mask]

    balanced_counts[sample] = np.asarray(
        Xs.sum(axis=0)
    ).ravel()

    if sparse.issparse(Xs):
        balanced_detect[sample] = np.asarray(
            (Xs > 0).mean(axis=0)
        ).ravel()
    else:
        balanced_detect[sample] = (
            Xs > 0
        ).mean(axis=0)

balanced_counts = pd.DataFrame(
    balanced_counts,
    index=genes
)

balanced_detect = pd.DataFrame(
    balanced_detect,
    index=genes
)

# ------------------------------------------------------------
# 5. Depth-balanced pseudobulk CPM + log2FC
# ------------------------------------------------------------

balanced_libsize = balanced_counts.sum(axis=0)

balanced_cpm = (
    balanced_counts
    .div(balanced_libsize, axis=1)
    * 1_000_000
)

prior = 0.5

balanced_log2fc = np.log2(
    (balanced_cpm["DM5D"] + prior) /
    (balanced_cpm["Control5D"] + prior)
)

balanced_detect_delta = (
    balanced_detect["DM5D"] -
    balanced_detect["Control5D"]
)

# ------------------------------------------------------------
# 6. Compare original vs depth-standardized results
# ------------------------------------------------------------

depth_test = pd.DataFrame(index=genes)

depth_test["original_log2FC"] = response.loc[
    genes, "early_log2FC"
]

depth_test["balanced_log2FC"] = balanced_log2fc

depth_test["original_detect_delta"] = response.loc[
    genes, "early_detect_delta"
]

depth_test["balanced_detect_delta"] = balanced_detect_delta

depth_test["log2FC_change_after_balance"] = (
    depth_test["balanced_log2FC"] -
    depth_test["original_log2FC"]
)

depth_test["detect_change_after_balance"] = (
    depth_test["balanced_detect_delta"] -
    depth_test["original_detect_delta"]
)

# ------------------------------------------------------------
# 7. Inspect anchors
# ------------------------------------------------------------

anchors = [
    "Zfp36",
    "Mt1",
    "Slc14a1",
    "Gfap",
    "Vim",
    "Glul",
    "Rlbp1",
    "Slc1a3"
]

anchors_present = [
    g for g in anchors
    if g in depth_test.index
]

print("\n" + "=" * 85)
print("ANCHORS: ORIGINAL VS DEPTH-STANDARDIZED")
print("=" * 85)

display(
    depth_test.loc[
        anchors_present,
        [
            "original_log2FC",
            "balanced_log2FC",
            "original_detect_delta",
            "balanced_detect_delta"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 8. Correlation across genes
# ------------------------------------------------------------

eligible_genes = response.loc[
    genes, "eligible_for_ranking"
].fillna(False)

corr_fc = depth_test.loc[
    eligible_genes,
    ["original_log2FC", "balanced_log2FC"]
].corr().iloc[0, 1]

corr_detect = depth_test.loc[
    eligible_genes,
    ["original_detect_delta", "balanced_detect_delta"]
].corr().iloc[0, 1]

print("\n" + "=" * 85)
print("GLOBAL ROBUSTNESS")
print("=" * 85)

print(
    f"Original vs balanced log2FC correlation: "
    f"{corr_fc:.3f}"
)

print(
    f"Original vs balanced detection-delta correlation: "
    f"{corr_detect:.3f}"
)

# ------------------------------------------------------------
# 9. Strongest balanced prevalence gains
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("TOP DEPTH-STANDARDIZED EARLY PREVALENCE GAINS")
print("=" * 85)

display(
    depth_test.loc[eligible_genes]
    .sort_values(
        "balanced_detect_delta",
        ascending=False
    )[
        [
            "balanced_detect_delta",
            "original_detect_delta",
            "balanced_log2FC",
            "original_log2FC"
        ]
    ]
    .head(20)
    .round(4)
)

# ------------------------------------------------------------
# 10. Save
# ------------------------------------------------------------

out_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_5D_depth_standardized.csv"
)

depth_test.to_csv(out_file)

print("\nSaved:")
print(out_file)

print(
    "\nThis is a depth-standardized sensitivity analysis, "
    "not independent biological replication."
)

In [ ]:
# ============================================================
# NEXT CELL — REPEATED DEPTH-STANDARDIZATION ROBUSTNESS
# 25 independent downsampling runs at 5,000 UMIs/cell
# ============================================================

import scanpy as sc
import numpy as np
import pandas as pd
from scipy import sparse

TARGET_UMI = 5000
N_REPEATS = 25

# ------------------------------------------------------------
# 1. Reconstruct the untouched 5-day object from raw counts
# ------------------------------------------------------------

mask_5d = (
    adata_muller.obs["sample"]
    .astype(str)
    .isin(["Control5D", "DM5D"])
)

early_base = adata_muller[mask_5d].copy()

# raw counts become X
early_base.X = early_base.layers["counts"].copy()

raw_depth = np.asarray(
    early_base.X.sum(axis=1)
).ravel()

eligible_cells = raw_depth >= TARGET_UMI

early_base = early_base[eligible_cells].copy()

print("=" * 90)
print("REPEATED 5D DEPTH-STANDARDIZATION")
print("=" * 90)

print(f"\nTarget: {TARGET_UMI:,} UMIs/cell")
print(f"Repeats: {N_REPEATS}")

print("\nEligible cells:")

display(
    early_base.obs["sample"]
    .astype(str)
    .value_counts()
    .reindex(["Control5D", "DM5D"])
    .rename("n_cells")
    .to_frame()
)

genes = early_base.var_names.to_numpy()

ctrl_mask = (
    early_base.obs["sample"]
    .astype(str)
    .to_numpy() == "Control5D"
)

dm_mask = (
    early_base.obs["sample"]
    .astype(str)
    .to_numpy() == "DM5D"
)

# ------------------------------------------------------------
# 2. Repeated downsampling
# ------------------------------------------------------------

all_log2fc = []
all_detect_delta = []

for seed in range(N_REPEATS):

    tmp = early_base.copy()

    # ensure fresh raw-count matrix each run
    tmp.X = tmp.layers["counts"].copy()

    sc.pp.downsample_counts(
        tmp,
        counts_per_cell=TARGET_UMI,
        random_state=seed
    )

    Xc = tmp.X[ctrl_mask]
    Xd = tmp.X[dm_mask]

    # aggregate counts
    ctrl_sum = np.asarray(
        Xc.sum(axis=0)
    ).ravel()

    dm_sum = np.asarray(
        Xd.sum(axis=0)
    ).ravel()

    # library-normalized CPM
    ctrl_cpm = (
        ctrl_sum /
        ctrl_sum.sum()
    ) * 1_000_000

    dm_cpm = (
        dm_sum /
        dm_sum.sum()
    ) * 1_000_000

    # same small prior used previously
    prior = 0.5

    log2fc = np.log2(
        (dm_cpm + prior) /
        (ctrl_cpm + prior)
    )

    # prevalence
    if sparse.issparse(Xc):
        ctrl_detect = np.asarray(
            (Xc > 0).mean(axis=0)
        ).ravel()

        dm_detect = np.asarray(
            (Xd > 0).mean(axis=0)
        ).ravel()
    else:
        ctrl_detect = (
            Xc > 0
        ).mean(axis=0)

        dm_detect = (
            Xd > 0
        ).mean(axis=0)

    detect_delta = dm_detect - ctrl_detect

    all_log2fc.append(log2fc)
    all_detect_delta.append(detect_delta)

    if (seed + 1) % 5 == 0:
        print(
            f"Completed {seed + 1}/{N_REPEATS} repeats"
        )

all_log2fc = np.vstack(all_log2fc)
all_detect_delta = np.vstack(all_detect_delta)

# ------------------------------------------------------------
# 3. Stability summary
# ------------------------------------------------------------

stability = pd.DataFrame(index=genes)

stability["median_balanced_log2FC"] = np.median(
    all_log2fc,
    axis=0
)

stability["log2FC_q10"] = np.quantile(
    all_log2fc,
    0.10,
    axis=0
)

stability["log2FC_q90"] = np.quantile(
    all_log2fc,
    0.90,
    axis=0
)

stability["median_balanced_detect_delta"] = np.median(
    all_detect_delta,
    axis=0
)

stability["detect_q10"] = np.quantile(
    all_detect_delta,
    0.10,
    axis=0
)

stability["detect_q90"] = np.quantile(
    all_detect_delta,
    0.90,
    axis=0
)

# fraction of runs with positive / negative expression effect
stability["frac_positive_log2FC"] = (
    all_log2fc > 0
).mean(axis=0)

stability["frac_negative_log2FC"] = (
    all_log2fc < 0
).mean(axis=0)

stability["direction_stability"] = np.maximum(
    stability["frac_positive_log2FC"],
    stability["frac_negative_log2FC"]
)

# original result for comparison
stability["original_log2FC"] = response.loc[
    genes,
    "early_log2FC"
]

stability["original_detect_delta"] = response.loc[
    genes,
    "early_detect_delta"
]

stability["eligible_for_ranking"] = response.loc[
    genes,
    "eligible_for_ranking"
].fillna(False)

# ------------------------------------------------------------
# 4. Define robustness ONLY, not biological meaning
#
# Robust-up:
# at least 90% of downsampling runs positive
#
# Robust-down:
# at least 90% negative
# ------------------------------------------------------------

stability["robust_direction"] = "unstable"

stability.loc[
    stability["frac_positive_log2FC"] >= 0.90,
    "robust_direction"
] = "up"

stability.loc[
    stability["frac_negative_log2FC"] >= 0.90,
    "robust_direction"
] = "down"

# ------------------------------------------------------------
# 5. Inspect anchors + previously interesting genes
# ------------------------------------------------------------

genes_of_interest = [
    "Zfp36",
    "Mt1",
    "Slc14a1",
    "Gfap",
    "Vim",
    "Glul",
    "Rlbp1",
    "Slc1a3",

    "Ccl21a",
    "Rsad2",
    "Gbp9",
    "Ccl19",
    "Bst2",

    "Shisal2b",
    "Abca8b",
    "Dhcr7",
    "Fmnl2",
    "Nr3c1",
    "Ide"
]

genes_present = [
    g for g in genes_of_interest
    if g in stability.index
]

print("\n" + "=" * 90)
print("ANCHORS / CANDIDATES — 25-RUN ROBUSTNESS")
print("=" * 90)

display(
    stability.loc[
        genes_present,
        [
            "original_log2FC",
            "median_balanced_log2FC",
            "log2FC_q10",
            "log2FC_q90",
            "direction_stability",
            "robust_direction",
            "original_detect_delta",
            "median_balanced_detect_delta"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 6. Robust early increases
# ------------------------------------------------------------

rankable = stability[
    stability["eligible_for_ranking"]
].copy()

robust_up = (
    rankable[
        rankable["robust_direction"] == "up"
    ]
    .sort_values(
        "median_balanced_log2FC",
        ascending=False
    )
)

robust_down = (
    rankable[
        rankable["robust_direction"] == "down"
    ]
    .sort_values(
        "median_balanced_log2FC",
        ascending=True
    )
)

print("\n" + "=" * 90)
print("TOP ROBUST EARLY INCREASES")
print("=" * 90)

display(
    robust_up[
        [
            "median_balanced_log2FC",
            "log2FC_q10",
            "log2FC_q90",
            "direction_stability",
            "original_log2FC",
            "median_balanced_detect_delta"
        ]
    ]
    .head(25)
    .round(4)
)

print(
    "\nNumber robust-up:",
    len(robust_up)
)

print("\n" + "=" * 90)
print("TOP ROBUST EARLY DECREASES")
print("=" * 90)

display(
    robust_down[
        [
            "median_balanced_log2FC",
            "log2FC_q10",
            "log2FC_q90",
            "direction_stability",
            "original_log2FC",
            "median_balanced_detect_delta"
        ]
    ]
    .head(25)
    .round(4)
)

print(
    "\nNumber robust-down:",
    len(robust_down)
)

# ------------------------------------------------------------
# 7. Save
# ------------------------------------------------------------

out_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_5D_repeated_depth_robustness.csv"
)

stability.to_csv(out_file)

print("\nSaved:")
print(out_file)

print(
    "\nThese labels mean technically stable direction "
    "under depth standardization only. "
    "They do NOT yet mean adaptive/protective."
)

In [ ]:
# ============================================================
# NEXT CELL — MÜLLER SPECIFICITY / AMBIENT-EXPRESSION AUDIT
#
# Question:
# Are our strongest early-response genes genuinely expressed
# in Müller glia, or mainly markers of other retinal populations?
# ============================================================

import numpy as np
import pandas as pd
from scipy import sparse

# ------------------------------------------------------------
# 1. Choose strongest robust early-response genes
# ------------------------------------------------------------

rankable = stability[
    stability["eligible_for_ranking"]
].copy()

top_up = (
    rankable[
        rankable["robust_direction"] == "up"
    ]
    .sort_values(
        "median_balanced_log2FC",
        ascending=False
    )
    .head(25)
    .index
    .tolist()
)

top_down = (
    rankable[
        rankable["robust_direction"] == "down"
    ]
    .sort_values(
        "median_balanced_log2FC",
        ascending=True
    )
    .head(25)
    .index
    .tolist()
)

# Add important biological anchors
extra_genes = [
    "Zfp36",
    "Mt1",
    "Slc14a1",
    "Vim",
    "Glul",
    "Slc1a3",
    "Ccl21a",
    "Rsad2",
    "Bst2",
    "Ccl19",
    "Gbp9",
    "Col1a1",
    "Vegfc",
    "Ntn3"
]

candidate_genes = list(
    dict.fromkeys(
        top_up +
        top_down +
        extra_genes
    )
)

candidate_genes = [
    g for g in candidate_genes
    if g in adata_qc.var_names
]

print("=" * 90)
print("MÜLLER SPECIFICITY / AMBIENT-EXPRESSION AUDIT")
print("=" * 90)

print(
    f"\nCandidate genes examined: "
    f"{len(candidate_genes)}"
)

# ------------------------------------------------------------
# 2. Identify Müller vs all other retinal cells
# ------------------------------------------------------------

is_muller = (
    adata_qc.obs["leiden"]
    .astype(str)
    .to_numpy() == "5"
)

is_other = ~is_muller

gene_idx = [
    adata_qc.var_names.get_loc(g)
    for g in candidate_genes
]

# normalized log1p expression
X_norm = adata_qc[:, candidate_genes].X

# raw counts
X_raw = adata_qc[:, candidate_genes].layers["counts"]

# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def col_mean(X):
    return np.asarray(
        X.mean(axis=0)
    ).ravel()

def detection_fraction(X):
    if sparse.issparse(X):
        return np.asarray(
            (X > 0).mean(axis=0)
        ).ravel()
    else:
        return (
            X > 0
        ).mean(axis=0)

# ------------------------------------------------------------
# 4. Müller vs non-Müller expression
# ------------------------------------------------------------

muller_mean = col_mean(
    X_norm[is_muller]
)

other_mean = col_mean(
    X_norm[is_other]
)

muller_detect = detection_fraction(
    X_raw[is_muller]
)

other_detect = detection_fraction(
    X_raw[is_other]
)

audit = pd.DataFrame(
    index=candidate_genes
)

audit["muller_mean_expr"] = muller_mean
audit["other_mean_expr"] = other_mean

audit["muller_detect"] = muller_detect
audit["other_detect"] = other_detect

# Müller enrichment relative to all other cells
eps = 1e-4

audit["muller_vs_other_log2ratio"] = np.log2(
    (audit["muller_mean_expr"] + eps) /
    (audit["other_mean_expr"] + eps)
)

# ------------------------------------------------------------
# 5. Determine which Leiden cluster expresses each gene most
# ------------------------------------------------------------

clusters = sorted(
    adata_qc.obs["leiden"]
    .astype(str)
    .unique(),
    key=int
)

cluster_mean_dict = {}

for cluster in clusters:

    mask = (
        adata_qc.obs["leiden"]
        .astype(str)
        .to_numpy() == cluster
    )

    cluster_mean_dict[cluster] = col_mean(
        X_norm[mask]
    )

cluster_means = pd.DataFrame(
    cluster_mean_dict,
    index=candidate_genes
)

audit["highest_expression_cluster"] = (
    cluster_means.idxmax(axis=1)
)

# rank cluster 5 for each gene
def cluster5_rank(row):
    ranked = (
        row
        .sort_values(
            ascending=False
        )
        .index
        .tolist()
    )

    return ranked.index("5") + 1

audit["muller_cluster_rank"] = (
    cluster_means.apply(
        cluster5_rank,
        axis=1
    )
)

# ------------------------------------------------------------
# 6. Add early-response robustness metrics
# ------------------------------------------------------------

for col in [
    "median_balanced_log2FC",
    "log2FC_q10",
    "log2FC_q90",
    "direction_stability",
    "robust_direction"
]:

    audit[col] = stability.loc[
        audit.index,
        col
    ]

# ------------------------------------------------------------
# 7. A conservative warning flag
#
# This is NOT automatic exclusion.
#
# Flag genes where:
# - Müller detection is low (<15%)
# AND
# - expression is higher outside Müller than inside Müller
# ------------------------------------------------------------

audit["possible_non_muller_signal"] = (
    (audit["muller_detect"] < 0.15) &
    (audit["muller_vs_other_log2ratio"] < 0)
)

# ------------------------------------------------------------
# 8. Print strongest UP genes
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("TOP ROBUST EARLY INCREASES — CELL-TYPE AUDIT")
print("=" * 90)

display(
    audit.loc[
        [g for g in top_up if g in audit.index],
        [
            "median_balanced_log2FC",
            "muller_detect",
            "other_detect",
            "muller_vs_other_log2ratio",
            "highest_expression_cluster",
            "muller_cluster_rank",
            "possible_non_muller_signal"
        ]
    ]
    .head(25)
    .round(4)
)

# ------------------------------------------------------------
# 9. Print strongest DOWN genes
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("TOP ROBUST EARLY DECREASES — CELL-TYPE AUDIT")
print("=" * 90)

display(
    audit.loc[
        [g for g in top_down if g in audit.index],
        [
            "median_balanced_log2FC",
            "muller_detect",
            "other_detect",
            "muller_vs_other_log2ratio",
            "highest_expression_cluster",
            "muller_cluster_rank",
            "possible_non_muller_signal"
        ]
    ]
    .head(25)
    .round(4)
)

# ------------------------------------------------------------
# 10. Key candidate genes
# ------------------------------------------------------------

key_present = [
    g for g in extra_genes
    if g in audit.index
]

print("\n" + "=" * 90)
print("BIOLOGICAL ANCHORS / CANDIDATES")
print("=" * 90)

display(
    audit.loc[
        key_present,
        [
            "median_balanced_log2FC",
            "muller_detect",
            "other_detect",
            "muller_vs_other_log2ratio",
            "highest_expression_cluster",
            "muller_cluster_rank",
            "possible_non_muller_signal"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 11. Summary
# ------------------------------------------------------------

n_flagged = audit[
    "possible_non_muller_signal"
].sum()

print("\n" + "=" * 90)
print("SUMMARY")
print("=" * 90)

print(
    f"Candidate genes flagged as possible "
    f"non-Müller/ambient signal: "
    f"{n_flagged}/{len(audit)}"
)

# ------------------------------------------------------------
# 12. Save
# ------------------------------------------------------------

out_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_candidate_specificity_audit.csv"
)

audit.to_csv(out_file)

print("\nSaved:")
print(out_file)

print(
    "\nFlags are QC warnings only. "
    "No genes have been excluded yet."
)

In [ ]:
# ============================================================
# NEXT CELL — WHOLE-TRANSCRIPTOME MÜLLER-SUPPORTED RESPONSE SET
#
# Combine:
# 1) repeated depth-standardization robustness
# 2) effect-size magnitude
# 3) actual Müller expression
# 4) Müller-vs-other retinal specificity
#
# This defines technically credible Müller RESPONSE genes.
# It does NOT define adaptive/protective genes yet.
# ============================================================

import numpy as np
import pandas as pd
from scipy import sparse

genes = adata_qc.var_names.to_numpy()

# ------------------------------------------------------------
# 1. Müller vs all non-Müller cells
# ------------------------------------------------------------

is_muller = (
    adata_qc.obs["leiden"]
    .astype(str)
    .to_numpy() == "5"
)

is_other = ~is_muller

X_norm = adata_qc.X
X_raw = adata_qc.layers["counts"]

def matrix_mean(X):
    return np.asarray(
        X.mean(axis=0)
    ).ravel()

def detect_fraction(X):
    if sparse.issparse(X):
        return np.asarray(
            (X > 0).mean(axis=0)
        ).ravel()
    else:
        return (
            X > 0
        ).mean(axis=0)

muller_mean = matrix_mean(
    X_norm[is_muller]
)

other_mean = matrix_mean(
    X_norm[is_other]
)

muller_detect = detect_fraction(
    X_raw[is_muller]
)

other_detect = detect_fraction(
    X_raw[is_other]
)

specificity_all = pd.DataFrame(
    index=genes
)

specificity_all["muller_mean_expr"] = muller_mean
specificity_all["other_mean_expr"] = other_mean

specificity_all["muller_detect"] = muller_detect
specificity_all["other_detect"] = other_detect

eps = 1e-4

specificity_all["muller_vs_other_log2ratio"] = np.log2(
    (specificity_all["muller_mean_expr"] + eps) /
    (specificity_all["other_mean_expr"] + eps)
)

# ------------------------------------------------------------
# 2. Find expression rank of Müller cluster for every gene
# ------------------------------------------------------------

clusters = sorted(
    adata_qc.obs["leiden"]
    .astype(str)
    .unique(),
    key=int
)

cluster_means = {}

for cluster in clusters:

    mask = (
        adata_qc.obs["leiden"]
        .astype(str)
        .to_numpy() == cluster
    )

    cluster_means[cluster] = matrix_mean(
        X_norm[mask]
    )

cluster_means = pd.DataFrame(
    cluster_means,
    index=genes
)

specificity_all["highest_expression_cluster"] = (
    cluster_means.idxmax(axis=1)
)

# rank of cluster 5 among all clusters
ranks = (
    cluster_means
    .rank(
        axis=1,
        method="min",
        ascending=False
    )
)

specificity_all["muller_cluster_rank"] = (
    ranks["5"]
    .astype(int)
)

# ------------------------------------------------------------
# 3. Merge with technical robustness
# ------------------------------------------------------------

master = stability.join(
    specificity_all,
    how="left"
)

# ------------------------------------------------------------
# 4. Conservative Müller-support criteria
#
# We require:
# - gene was eligible in original analysis
# - stable direction in >=90% downsampling runs
# - |median balanced log2FC| >= 0.5
# - detected in >=5% of Müller cells
#
# AND at least ONE Müller-support criterion:
# - mean expression higher in Müller than all other cells
# OR
# - Müller is among top 3 expressing clusters
#
# This avoids throwing away shared glial/stress genes.
# ------------------------------------------------------------

master["effect_size_pass"] = (
    master["median_balanced_log2FC"]
    .abs() >= 0.5
)

master["muller_expression_pass"] = (
    master["muller_detect"] >= 0.05
)

master["muller_specificity_support"] = (
    (master["muller_vs_other_log2ratio"] > 0) |
    (master["muller_cluster_rank"] <= 3)
)

master["muller_supported_response"] = (
    master["eligible_for_ranking"].fillna(False) &
    master["robust_direction"].isin(["up", "down"]) &
    master["effect_size_pass"] &
    master["muller_expression_pass"] &
    master["muller_specificity_support"]
)

# ------------------------------------------------------------
# 5. Separate UP and DOWN
# ------------------------------------------------------------

supported = master[
    master["muller_supported_response"]
].copy()

supported_up = supported[
    supported["robust_direction"] == "up"
].sort_values(
    "median_balanced_log2FC",
    ascending=False
)

supported_down = supported[
    supported["robust_direction"] == "down"
].sort_values(
    "median_balanced_log2FC",
    ascending=True
)

print("=" * 95)
print("WHOLE-TRANSCRIPTOME MÜLLER-SUPPORTED EARLY RESPONSE")
print("=" * 95)

print(
    f"\nSupported UP genes:   {len(supported_up):,}"
)

print(
    f"Supported DOWN genes: {len(supported_down):,}"
)

print(
    f"Total supported:      {len(supported):,}"
)

# ------------------------------------------------------------
# 6. Top supported increases
# ------------------------------------------------------------

cols = [
    "median_balanced_log2FC",
    "log2FC_q10",
    "log2FC_q90",
    "muller_detect",
    "other_detect",
    "muller_vs_other_log2ratio",
    "highest_expression_cluster",
    "muller_cluster_rank"
]

print("\n" + "=" * 95)
print("TOP MÜLLER-SUPPORTED EARLY INCREASES")
print("=" * 95)

display(
    supported_up[
        cols
    ]
    .head(30)
    .round(4)
)

# ------------------------------------------------------------
# 7. Top supported decreases
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("TOP MÜLLER-SUPPORTED EARLY DECREASES")
print("=" * 95)

display(
    supported_down[
        cols
    ]
    .head(30)
    .round(4)
)

# ------------------------------------------------------------
# 8. Key biology check
# ------------------------------------------------------------

key_genes = [
    "Zfp36",
    "Mt1",
    "Slc14a1",
    "Vim",
    "Glul",
    "Slc1a3",
    "Ccl21a",
    "Rsad2",
    "Bst2",
    "Ccl19",
    "Gbp9",
    "Phgdh",
    "Ntn3",
    "Vegfc",
    "Col1a1",
    "Aqp5"
]

key_genes = [
    g for g in key_genes
    if g in master.index
]

print("\n" + "=" * 95)
print("KEY GENES — FINAL TECHNICAL FILTER STATUS")
print("=" * 95)

display(
    master.loc[
        key_genes,
        [
            "median_balanced_log2FC",
            "direction_stability",
            "robust_direction",
            "muller_detect",
            "muller_vs_other_log2ratio",
            "muller_cluster_rank",
            "effect_size_pass",
            "muller_expression_pass",
            "muller_specificity_support",
            "muller_supported_response"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 9. Save master table
# ------------------------------------------------------------

out_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_MASTER_early_response.csv"
)

master.to_csv(out_file)

print("\nSaved:")
print(out_file)

print(
    "\nIMPORTANT: "
    "'Müller-supported response' means technically credible "
    "and sufficiently Müller-expressed. "
    "It does NOT mean adaptive, protective, or causal."
)

In [ ]:
# ============================================================
# NEXT CELL — PRERANKED PATHWAY ANALYSIS OF EARLY MÜLLER RESPONSE
#
# Uses the broad technically credible Müller transcriptome,
# NOT only the thresholded 333 UP / 461 DOWN genes.
#
# Positive NES = enriched toward DM5D increase
# Negative NES = enriched toward DM5D decrease
# ============================================================

!pip -q install gseapy

import gseapy as gp
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Define ranked Müller-supported universe
#
# Keep genes that:
# - were adequately detected
# - are sufficiently Müller-expressed
# - have Müller-specific/supportive expression
# - show reasonably stable direction under depth balancing
#
# We deliberately DO NOT require |log2FC| >= 0.5 here.
# ------------------------------------------------------------

gsea_input = master[
    master["eligible_for_ranking"].fillna(False) &
    master["muller_expression_pass"].fillna(False) &
    master["muller_specificity_support"].fillna(False) &
    (master["direction_stability"] >= 0.80)
].copy()

gsea_input = gsea_input[
    gsea_input["median_balanced_log2FC"].notna()
].copy()

# Remove exact zero effects
gsea_input = gsea_input[
    gsea_input["median_balanced_log2FC"] != 0
].copy()

# Rank strongest increase -> strongest decrease
rnk = (
    gsea_input[["median_balanced_log2FC"]]
    .sort_values(
        "median_balanced_log2FC",
        ascending=False
    )
    .reset_index()
)

rnk.columns = [
    "gene",
    "score"
]

print("=" * 90)
print("PRERANKED MÜLLER PATHWAY ANALYSIS")
print("=" * 90)

print(
    f"\nGenes entering ranked analysis: "
    f"{len(rnk):,}"
)

print("\nTop ranked genes:")
display(rnk.head(10))

print("\nBottom ranked genes:")
display(rnk.tail(10))


# ------------------------------------------------------------
# 2. Run GO Biological Process
# ------------------------------------------------------------

print("\nRunning GO Biological Process GSEA...")

go_res = gp.prerank(
    rnk=rnk,
    gene_sets="GO_Biological_Process_2023",
    organism="Mouse",
    min_size=10,
    max_size=500,
    permutation_num=500,
    seed=42,
    threads=4,
    verbose=False
)

go_table = go_res.res2d.copy()


# ------------------------------------------------------------
# 3. Run Reactome
# ------------------------------------------------------------

print("Running Reactome GSEA...")

reactome_res = gp.prerank(
    rnk=rnk,
    gene_sets="Reactome_2022",
    organism="Mouse",
    min_size=10,
    max_size=500,
    permutation_num=500,
    seed=42,
    threads=4,
    verbose=False
)

reactome_table = reactome_res.res2d.copy()


# ------------------------------------------------------------
# 4. Standardize numeric columns
# ------------------------------------------------------------

for df in [go_table, reactome_table]:

    for col in ["NES", "NOM p-val", "FDR q-val"]:

        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )


# ------------------------------------------------------------
# 5. Helper: print significant positive / negative pathways
# ------------------------------------------------------------

def show_results(df, title, n=20):

    sig = df[
        df["FDR q-val"] < 0.25
    ].copy()

    positive = (
        sig[sig["NES"] > 0]
        .sort_values(
            "NES",
            ascending=False
        )
        .head(n)
    )

    negative = (
        sig[sig["NES"] < 0]
        .sort_values(
            "NES",
            ascending=True
        )
        .head(n)
    )

    cols = [
        "Term",
        "ES",
        "NES",
        "NOM p-val",
        "FDR q-val",
        "Lead_genes"
    ]

    cols = [
        c for c in cols
        if c in df.columns
    ]

    print("\n" + "=" * 90)
    print(f"{title} — ENRICHED TOWARD DM5D")
    print("=" * 90)

    display(
        positive[cols]
        .round(4)
    )

    print("\n" + "=" * 90)
    print(f"{title} — ENRICHED TOWARD CONTROL5D")
    print("=" * 90)

    display(
        negative[cols]
        .round(4)
    )

    print(
        f"\nSignificant pathways at FDR < 0.25: "
        f"{len(sig)}"
    )

    return sig


go_sig = show_results(
    go_table,
    "GO BIOLOGICAL PROCESS"
)

reactome_sig = show_results(
    reactome_table,
    "REACTOME"
)


# ------------------------------------------------------------
# 6. Save complete results
# ------------------------------------------------------------

go_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_GSEA_GO_BP.csv"
)

reactome_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_GSEA_Reactome.csv"
)

rank_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_GSEA_ranked_genes.csv"
)

go_table.to_csv(
    go_file,
    index=False
)

reactome_table.to_csv(
    reactome_file,
    index=False
)

rnk.to_csv(
    rank_file,
    index=False
)

print("\nSaved:")
print(go_file)
print(reactome_file)
print(rank_file)

print(
    "\nIMPORTANT: Positive NES means the pathway is shifted "
    "toward DM5D; negative NES means it is shifted toward Control5D."
)

print(
    "These are pathway-level descriptive results from one pooled "
    "library per condition, not replicated animal-level inference."
)

In [ ]:
# ============================================================
# NEXT CELL — TIE-ROBUST GSEA SENSITIVITY ANALYSIS
#
# Problem:
# Median effects produced 28.43% tied ranking values.
#
# Solution:
# 1. use MEAN depth-balanced log2FC across 25 runs
# 2. add a negligible deterministic secondary tie-breaker
# 3. rerun GO + Reactome
# 4. compare pathway stability with original GSEA
# ============================================================

import numpy as np
import pandas as pd
import gseapy as gp

# ------------------------------------------------------------
# 1. Mean effect across the 25 depth-standardization runs
# ------------------------------------------------------------

mean_balanced_fc = np.mean(
    all_log2fc,
    axis=0
)

sd_balanced_fc = np.std(
    all_log2fc,
    axis=0,
    ddof=1
)

mean_effect = pd.DataFrame(
    {
        "mean_balanced_log2FC": mean_balanced_fc,
        "sd_balanced_log2FC": sd_balanced_fc
    },
    index=early_base.var_names
)

# ------------------------------------------------------------
# 2. Add to master table
# ------------------------------------------------------------

gsea_master = master.join(
    mean_effect,
    how="left"
)

# Use the SAME biological/technical eligibility rules
gsea2 = gsea_master[
    gsea_master["eligible_for_ranking"].fillna(False) &
    gsea_master["muller_expression_pass"].fillna(False) &
    gsea_master["muller_specificity_support"].fillna(False) &
    (gsea_master["direction_stability"] >= 0.80) &
    gsea_master["mean_balanced_log2FC"].notna()
].copy()

# ------------------------------------------------------------
# 3. Diagnose ties before correction
# ------------------------------------------------------------

n_total = len(gsea2)

n_unique = (
    gsea2["mean_balanced_log2FC"]
    .nunique()
)

tie_fraction = 1 - (n_unique / n_total)

print("=" * 90)
print("TIE-ROBUST GSEA")
print("=" * 90)

print(f"\nGenes entering analysis: {n_total:,}")
print(
    f"Unique mean scores: {n_unique:,}"
)

print(
    f"Approximate tied-score fraction before "
    f"tie-breaking: {100 * tie_fraction:.2f}%"
)

# ------------------------------------------------------------
# 4. Deterministic negligible tie-breaker
#
# Primary ranking remains mean balanced log2FC.
# Original log2FC is used only to order exact ties.
# Alphabetical gene order resolves any remaining exact ties.
# The perturbation is tiny enough not to alter biological
# effect magnitudes.
# ------------------------------------------------------------

tmp = gsea2[
    [
        "mean_balanced_log2FC",
        "original_log2FC"
    ]
].copy()

tmp["gene"] = tmp.index

tmp = tmp.sort_values(
    [
        "mean_balanced_log2FC",
        "original_log2FC",
        "gene"
    ],
    ascending=[
        False,
        False,
        True
    ]
)

# deterministic unique tiny offsets
offset = np.linspace(
    1e-9,
    -1e-9,
    len(tmp)
)

tmp["tie_free_score"] = (
    tmp["mean_balanced_log2FC"].to_numpy()
    + offset
)

rnk2 = (
    tmp[
        ["gene", "tie_free_score"]
    ]
    .rename(
        columns={
            "tie_free_score": "score"
        }
    )
)

print(
    "Unique final ranking scores:",
    rnk2["score"].nunique()
)

assert (
    rnk2["score"].nunique()
    == len(rnk2)
), "Tie-breaking failed."

print("\nTop ranked genes:")
display(
    rnk2.head(10)
)

print("\nBottom ranked genes:")
display(
    rnk2.tail(10)
)

# ------------------------------------------------------------
# 5. Rerun GO Biological Process
# ------------------------------------------------------------

print("\nRunning tie-robust GO GSEA...")

go2 = gp.prerank(
    rnk=rnk2,
    gene_sets="GO_Biological_Process_2023",
    organism="Mouse",
    min_size=10,
    max_size=500,
    permutation_num=1000,
    seed=42,
    threads=4,
    verbose=False
)

go2_table = go2.res2d.copy()

# ------------------------------------------------------------
# 6. Rerun Reactome
# ------------------------------------------------------------

print("Running tie-robust Reactome GSEA...")

reactome2 = gp.prerank(
    rnk=rnk2,
    gene_sets="Reactome_2022",
    organism="Mouse",
    min_size=10,
    max_size=500,
    permutation_num=1000,
    seed=42,
    threads=4,
    verbose=False
)

reactome2_table = reactome2.res2d.copy()

# ------------------------------------------------------------
# 7. Numeric conversion
# ------------------------------------------------------------

for df in [
    go2_table,
    reactome2_table
]:

    for col in [
        "NES",
        "NOM p-val",
        "FDR q-val"
    ]:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

# ------------------------------------------------------------
# 8. Display helper
# ------------------------------------------------------------

def display_gsea(df, name, n=20):

    sig = df[
        df["FDR q-val"] < 0.25
    ].copy()

    up = (
        sig[sig["NES"] > 0]
        .sort_values(
            "NES",
            ascending=False
        )
        .head(n)
    )

    down = (
        sig[sig["NES"] < 0]
        .sort_values(
            "NES",
            ascending=True
        )
        .head(n)
    )

    cols = [
        "Term",
        "NES",
        "NOM p-val",
        "FDR q-val",
        "Lead_genes"
    ]

    print("\n" + "=" * 90)
    print(
        f"{name} — DM5D ENRICHED"
    )
    print("=" * 90)

    display(
        up[cols].round(4)
    )

    print("\n" + "=" * 90)
    print(
        f"{name} — CONTROL5D ENRICHED"
    )
    print("=" * 90)

    display(
        down[cols].round(4)
    )

    print(
        f"\nTotal FDR < 0.25: "
        f"{len(sig)}"
    )

    return sig


go2_sig = display_gsea(
    go2_table,
    "GO BP"
)

reactome2_sig = display_gsea(
    reactome2_table,
    "REACTOME"
)

# ------------------------------------------------------------
# 9. Compare pathway overlap with original analysis
# ------------------------------------------------------------

old_go_terms = set(
    go_sig["Term"]
)

new_go_terms = set(
    go2_sig["Term"]
)

old_reactome_terms = set(
    reactome_sig["Term"]
)

new_reactome_terms = set(
    reactome2_sig["Term"]
)

print("\n" + "=" * 90)
print("PATHWAY ROBUSTNESS")
print("=" * 90)

print(
    "GO significant terms:"
)

print(
    f"Original = {len(old_go_terms)}, "
    f"Tie-robust = {len(new_go_terms)}, "
    f"Shared = {len(old_go_terms & new_go_terms)}"
)

print(
    "\nReactome significant terms:"
)

print(
    f"Original = {len(old_reactome_terms)}, "
    f"Tie-robust = {len(new_reactome_terms)}, "
    f"Shared = {len(old_reactome_terms & new_reactome_terms)}"
)

# ------------------------------------------------------------
# 10. Specifically audit pathways central to our hypothesis
# ------------------------------------------------------------

keywords = [
    "Interferon",
    "Retinoid",
    "Vitamin",
    "Chondroitin",
    "Glycosaminoglycan",
    "Extracellular Matrix",
    "Netrin",
    "Potassium",
    "Endocytosis",
    "Autophagy"
]

focus = reactome2_table[
    reactome2_table["Term"]
    .str.contains(
        "|".join(keywords),
        case=False,
        regex=True
    )
].copy()

focus = focus.sort_values(
    "FDR q-val",
    ascending=True
)

print("\n" + "=" * 90)
print("HYPOTHESIS-RELEVANT REACTOME PATHWAYS")
print("=" * 90)

display(
    focus[
        [
            "Term",
            "NES",
            "NOM p-val",
            "FDR q-val",
            "Lead_genes"
        ]
    ]
    .head(30)
    .round(4)
)

# ------------------------------------------------------------
# 11. Save
# ------------------------------------------------------------

rank_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_GSEA_TIE_ROBUST_rank.csv"
)

go_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_GSEA_TIE_ROBUST_GO.csv"
)

reactome_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_GSEA_TIE_ROBUST_Reactome.csv"
)

rnk2.to_csv(
    rank_file,
    index=False
)

go2_table.to_csv(
    go_file,
    index=False
)

reactome2_table.to_csv(
    reactome_file,
    index=False
)

print("\nSaved:")
print(rank_file)
print(go_file)
print(reactome_file)

print(
    "\nDo not interpret pathways as final biology "
    "until stability versus the original ranking is assessed."
)

In [ ]:
# ============================================================
# NEXT CELL — EARLY vs LATE MÜLLER TRAJECTORY CLASSIFICATION
#
# Compare:
# early = mean depth-balanced DM5D vs Control5D effect
# late  = direct pseudobulk DM15D vs Control15D effect
#
# Descriptive only: one pooled library per condition.
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Build trajectory table
# ------------------------------------------------------------

trajectory = master.copy()

# Early effect:
# mean across the 25 depth-standardized 5D analyses
trajectory["early_effect"] = gsea_master[
    "mean_balanced_log2FC"
]

# Late effect:
# original pseudobulk CPM effect is acceptable because
# Control15D and DM15D were depth/complexity matched
trajectory["late_effect"] = response[
    "late_log2FC"
]

# ------------------------------------------------------------
# 2. Restrict to genes with credible Müller expression
# ------------------------------------------------------------

trajectory["trajectory_eligible"] = (
    trajectory["eligible_for_ranking"].fillna(False) &
    trajectory["muller_expression_pass"].fillna(False) &
    trajectory["muller_specificity_support"].fillna(False) &
    trajectory["early_effect"].notna() &
    trajectory["late_effect"].notna()
)

traj = trajectory[
    trajectory["trajectory_eligible"]
].copy()

# ------------------------------------------------------------
# 3. Descriptive thresholds
#
# Strong early response = |log2FC| >= 0.5
# Late "returned"       = |log2FC| < 0.25
# Persistent            = same direction and |late| >= 0.5
# Reversed              = opposite direction and |late| >= 0.5
#
# These are descriptive bins, not significance thresholds.
# ------------------------------------------------------------

EARLY_THR = 0.50
RETURN_THR = 0.25
LATE_THR = 0.50

traj["trajectory_class"] = "other"

# ---- early UP ----

early_up = traj["early_effect"] >= EARLY_THR

traj.loc[
    early_up &
    (traj["late_effect"].abs() < RETURN_THR),
    "trajectory_class"
] = "early_up_transient"

traj.loc[
    early_up &
    (traj["late_effect"] >= LATE_THR),
    "trajectory_class"
] = "persistent_up"

traj.loc[
    early_up &
    (traj["late_effect"] <= -LATE_THR),
    "trajectory_class"
] = "up_to_down_reversal"

# ---- early DOWN ----

early_down = traj["early_effect"] <= -EARLY_THR

traj.loc[
    early_down &
    (traj["late_effect"].abs() < RETURN_THR),
    "trajectory_class"
] = "early_down_transient"

traj.loc[
    early_down &
    (traj["late_effect"] <= -LATE_THR),
    "trajectory_class"
] = "persistent_down"

traj.loc[
    early_down &
    (traj["late_effect"] >= LATE_THR),
    "trajectory_class"
] = "down_to_up_reversal"

# ------------------------------------------------------------
# 4. Count trajectories
# ------------------------------------------------------------

trajectory_counts = (
    traj["trajectory_class"]
    .value_counts()
)

print("=" * 95)
print("EARLY → LATE MÜLLER TRAJECTORIES")
print("=" * 95)

display(
    trajectory_counts
    .rename("n_genes")
    .to_frame()
)

# ------------------------------------------------------------
# 5. Early vs late correlation
# ------------------------------------------------------------

corr_all = traj[
    ["early_effect", "late_effect"]
].corr().iloc[0, 1]

print(
    f"\nEarly vs late effect correlation: "
    f"{corr_all:.3f}"
)

# ------------------------------------------------------------
# 6. Show most important trajectory classes
# ------------------------------------------------------------

def show_class(name, n=25, sort_col="early_effect", ascending=False):

    x = traj[
        traj["trajectory_class"] == name
    ].copy()

    x = x.sort_values(
        sort_col,
        ascending=ascending
    )

    print("\n" + "=" * 95)
    print(name.upper())
    print("=" * 95)

    display(
        x[
            [
                "early_effect",
                "late_effect",
                "muller_detect",
                "muller_vs_other_log2ratio",
                "muller_cluster_rank"
            ]
        ]
        .head(n)
        .round(4)
    )

    print(
        f"\nTotal: {len(x):,}"
    )


show_class(
    "early_up_transient",
    sort_col="early_effect",
    ascending=False
)

show_class(
    "persistent_up",
    sort_col="early_effect",
    ascending=False
)

show_class(
    "up_to_down_reversal",
    sort_col="early_effect",
    ascending=False
)

show_class(
    "early_down_transient",
    sort_col="early_effect",
    ascending=True
)

show_class(
    "persistent_down",
    sort_col="early_effect",
    ascending=True
)

show_class(
    "down_to_up_reversal",
    sort_col="early_effect",
    ascending=True
)

# ------------------------------------------------------------
# 7. Audit biologically interesting genes
# ------------------------------------------------------------

focus_genes = [
    "Ccl21a",
    "Rsad2",
    "Bst2",
    "Ccl19",
    "Gbp9",
    "Phgdh",
    "Ntn3",
    "Vegfc",
    "Aqp5",
    "Ncan",
    "Chsy3",
    "Kcnj10",
    "Glul",
    "Slc1a3",
    "Mt1",
    "Zfp36",
    "Slc14a1"
]

focus_present = [
    g for g in focus_genes
    if g in traj.index
]

print("\n" + "=" * 95)
print("FOCUS GENES — EARLY → LATE TRAJECTORY")
print("=" * 95)

display(
    traj.loc[
        focus_present,
        [
            "early_effect",
            "late_effect",
            "trajectory_class",
            "muller_detect",
            "muller_cluster_rank"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 8. Save
# ------------------------------------------------------------

out_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_early_late_trajectory.csv"
)

trajectory.to_csv(out_file)

print("\nSaved:")
print(out_file)

print(
    "\nTrajectory classes are descriptive patterns only. "
    "They are not significance calls or evidence of causality."
)

In [ ]:
# ============================================================
# NEXT CELL — DIAGNOSE EMPTY LOCAL ORA
# Check exact vs case-insensitive overlap between:
#   Müller background
#   trajectory genes
#   downloaded Reactome / GO libraries
# ============================================================

import pandas as pd

print("=" * 95)
print("GENE-SYMBOL OVERLAP DIAGNOSTIC")
print("=" * 95)

# ------------------------------------------------------------
# 1. Background symbols
# ------------------------------------------------------------

bg_original = set(
    traj.index.astype(str)
)

bg_upper = {
    g.upper()
    for g in bg_original
}

print("\nBackground example:")
print(
    sorted(bg_original)[:20]
)


# ------------------------------------------------------------
# 2. Inspect raw library gene symbols
# ------------------------------------------------------------

reactome_all = set()

for genes in reactome_sets.values():
    reactome_all.update(
        map(str, genes)
    )

go_all = set()

for genes in go_sets.values():
    go_all.update(
        map(str, genes)
    )

print("\nReactome gene examples:")
print(
    sorted(reactome_all)[:30]
)

print("\nGO gene examples:")
print(
    sorted(go_all)[:30]
)


# ------------------------------------------------------------
# 3. Exact versus case-insensitive overlap
# ------------------------------------------------------------

reactome_exact = (
    bg_original &
    reactome_all
)

go_exact = (
    bg_original &
    go_all
)

reactome_upper = (
    bg_upper &
    {g.upper() for g in reactome_all}
)

go_upper = (
    bg_upper &
    {g.upper() for g in go_all}
)

summary = pd.DataFrame(
    {
        "exact_overlap": [
            len(reactome_exact),
            len(go_exact)
        ],
        "case_insensitive_overlap": [
            len(reactome_upper),
            len(go_upper)
        ]
    },
    index=[
        "Reactome_2022",
        "GO_Biological_Process_2023"
    ]
)

print("\n" + "=" * 95)
print("BACKGROUND ↔ LIBRARY OVERLAP")
print("=" * 95)

display(summary)


# ------------------------------------------------------------
# 4. Check trajectory gene overlap too
# ------------------------------------------------------------

rows = []

for label, genes in gene_sets_by_class.items():

    genes_original = set(
        map(str, genes)
    )

    genes_upper = {
        g.upper()
        for g in genes_original
    }

    rows.append({
        "trajectory": label,
        "n_genes": len(genes_original),

        "Reactome_exact": len(
            genes_original &
            reactome_all
        ),

        "Reactome_case_insensitive": len(
            genes_upper &
            {g.upper() for g in reactome_all}
        ),

        "GO_exact": len(
            genes_original &
            go_all
        ),

        "GO_case_insensitive": len(
            genes_upper &
            {g.upper() for g in go_all}
        )
    })

trajectory_overlap = pd.DataFrame(rows)

print("\n" + "=" * 95)
print("TRAJECTORY ↔ LIBRARY OVERLAP")
print("=" * 95)

display(
    trajectory_overlap
)


# ------------------------------------------------------------
# 5. Specifically check known genes
# ------------------------------------------------------------

test_genes = [
    "Ccl21a",
    "Rsad2",
    "Bst2",
    "Ccl19",
    "Phgdh",
    "Ntn3",
    "Vegfc",
    "Ncan",
    "Chsy3"
]

rows = []

for gene in test_genes:

    rows.append({
        "gene": gene,

        "Reactome_exact": (
            gene in reactome_all
        ),

        "Reactome_upper": (
            gene.upper()
            in {
                x.upper()
                for x in reactome_all
            }
        ),

        "GO_exact": (
            gene in go_all
        ),

        "GO_upper": (
            gene.upper()
            in {
                x.upper()
                for x in go_all
            }
        )
    })

print("\n" + "=" * 95)
print("KNOWN GENE AUDIT")
print("=" * 95)

display(
    pd.DataFrame(rows)
)

In [ ]:
# ============================================================
# NEXT CELL — CORRECTED LOCAL TRAJECTORY ORA
#
# Fixes:
# 1. case-insensitive gene matching
# 2. library-specific annotated Müller background
# 3. original mouse symbols preserved in outputs
#
# No Enrichr enrichment API is used.
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import hypergeom


# ------------------------------------------------------------
# 1. Original Müller trajectory background
# ------------------------------------------------------------

background_original = set(
    traj.index.astype(str)
)

trajectory_sets_original = {
    "early_up_transient": set(
        traj[
            traj["trajectory_class"] == "early_up_transient"
        ].index.astype(str)
    ),

    "persistent_up": set(
        traj[
            traj["trajectory_class"] == "persistent_up"
        ].index.astype(str)
    ),

    "up_to_down_reversal": set(
        traj[
            traj["trajectory_class"] == "up_to_down_reversal"
        ].index.astype(str)
    ),
}


# ------------------------------------------------------------
# 2. Uppercase matching map while retaining mouse symbols
# ------------------------------------------------------------

def build_upper_map(genes):

    mapping = {}

    for gene in genes:

        key = str(gene).upper()

        if key not in mapping:
            mapping[key] = []

        mapping[key].append(str(gene))

    return mapping


background_map = build_upper_map(
    background_original
)

background_upper = set(
    background_map.keys()
)


# ------------------------------------------------------------
# 3. Normalize pathway libraries to uppercase
# ------------------------------------------------------------

def normalize_library(library_dict):

    normalized = {}

    for term, genes in library_dict.items():

        normalized[term] = {
            str(g).upper()
            for g in genes
        }

    return normalized


reactome_upper = normalize_library(
    reactome_sets
)

go_upper = normalize_library(
    go_sets
)


# ------------------------------------------------------------
# 4. BH adjustment
# ------------------------------------------------------------

def bh_adjust(pvalues):

    p = np.asarray(
        pvalues,
        dtype=float
    )

    m = len(p)

    if m == 0:
        return np.array([])

    order = np.argsort(p)

    ranked = p[order]

    adjusted_ranked = (
        ranked *
        m /
        np.arange(1, m + 1)
    )

    adjusted_ranked = np.minimum.accumulate(
        adjusted_ranked[::-1]
    )[::-1]

    adjusted_ranked = np.clip(
        adjusted_ranked,
        0,
        1
    )

    adjusted = np.empty(m)

    adjusted[order] = adjusted_ranked

    return adjusted


# ------------------------------------------------------------
# 5. Correct local ORA
#
# Critical point:
# Universe = Müller background genes represented somewhere
# in that specific pathway library.
# ------------------------------------------------------------

def corrected_local_ora(
    query_original,
    library_upper,
    trajectory_name,
    library_name,
    min_pathway_size=5,
    max_pathway_size=500
):

    # union of genes represented by this library
    library_gene_union = set()

    for genes in library_upper.values():
        library_gene_union.update(genes)

    # library-annotated Müller universe
    annotated_background = (
        background_upper &
        library_gene_union
    )

    # query genes normalized to uppercase
    query_upper_all = {
        g.upper()
        for g in query_original
    }

    # query genes actually represented in this library
    query_upper = (
        query_upper_all &
        annotated_background
    )

    M = len(
        annotated_background
    )

    N = len(
        query_upper
    )

    rows = []

    for term, pathway_all in library_upper.items():

        pathway = (
            pathway_all &
            annotated_background
        )

        K = len(
            pathway
        )

        if K < min_pathway_size:
            continue

        if K > max_pathway_size:
            continue

        overlap_upper = (
            query_upper &
            pathway
        )

        x = len(
            overlap_upper
        )

        if x == 0:
            continue

        # P(X >= x)
        pval = hypergeom.sf(
            x - 1,
            M,
            K,
            N
        )

        # odds ratio with Haldane-Anscombe correction
        a = x
        b = N - x
        c = K - x
        d = M - K - N + x

        odds_ratio = (
            (a + 0.5) *
            (d + 0.5)
        ) / (
            (b + 0.5) *
            (c + 0.5)
        )

        # convert overlap back to original mouse symbols
        original_overlap = []

        for symbol_upper in sorted(
            overlap_upper
        ):

            original_overlap.extend(
                background_map.get(
                    symbol_upper,
                    [symbol_upper]
                )
            )

        original_overlap = sorted(
            set(original_overlap)
        )

        rows.append({
            "trajectory_class": trajectory_name,
            "library": library_name,

            "Term": term,

            "Overlap": f"{x}/{K}",

            "overlap_n": x,
            "pathway_background_n": K,

            "query_mapped_n": N,
            "query_original_n": len(
                query_original
            ),

            "background_annotated_n": M,
            "background_original_n": len(
                background_original
            ),

            "query_mapping_pct": (
                100 * N /
                len(query_original)
                if len(query_original) > 0
                else np.nan
            ),

            "Odds Ratio": odds_ratio,
            "P-value": pval,

            "Genes": ";".join(
                original_overlap
            )
        })

    result = pd.DataFrame(
        rows
    )

    if len(result) == 0:
        return result

    result["Adjusted P-value"] = bh_adjust(
        result["P-value"].values
    )

    result = result.sort_values(
        [
            "Adjusted P-value",
            "P-value"
        ]
    )

    return result


# ------------------------------------------------------------
# 6. Run all trajectory classes
# ------------------------------------------------------------

all_results = []
coverage_rows = []

libraries = [
    (
        "Reactome_2022",
        reactome_upper
    ),
    (
        "GO_Biological_Process_2023",
        go_upper
    )
]

for trajectory_name, query_genes in (
    trajectory_sets_original.items()
):

    for library_name, library_data in libraries:

        # calculate coverage first
        library_union = set()

        for genes in library_data.values():
            library_union.update(genes)

        annotated_bg = (
            background_upper &
            library_union
        )

        query_upper = {
            g.upper()
            for g in query_genes
        }

        mapped_query = (
            query_upper &
            annotated_bg
        )

        coverage_rows.append({
            "trajectory_class": trajectory_name,
            "library": library_name,
            "original_query_genes": len(
                query_genes
            ),
            "mapped_query_genes": len(
                mapped_query
            ),
            "mapping_pct": (
                100 *
                len(mapped_query) /
                len(query_genes)
            ),
            "annotated_background": len(
                annotated_bg
            )
        })

        result = corrected_local_ora(
            query_original=query_genes,
            library_upper=library_data,
            trajectory_name=trajectory_name,
            library_name=library_name
        )

        if len(result) > 0:
            all_results.append(
                result
            )


coverage = pd.DataFrame(
    coverage_rows
)

print("=" * 100)
print("LIBRARY MAPPING COVERAGE")
print("=" * 100)

display(
    coverage.round(2)
)


# ------------------------------------------------------------
# 7. Combine
# ------------------------------------------------------------

if len(all_results) == 0:

    raise RuntimeError(
        "No ORA results generated even after "
        "case-insensitive matching."
    )

enrichment_fixed = pd.concat(
    all_results,
    ignore_index=True
)


# ------------------------------------------------------------
# 8. Print significant pathway sections
# ------------------------------------------------------------

for trajectory_name in (
    trajectory_sets_original.keys()
):

    for library_name, _ in libraries:

        sub = enrichment_fixed[
            (
                enrichment_fixed[
                    "trajectory_class"
                ] == trajectory_name
            )
            &
            (
                enrichment_fixed[
                    "library"
                ] == library_name
            )
        ].copy()

        sig = (
            sub[
                sub[
                    "Adjusted P-value"
                ] < 0.10
            ]
            .sort_values(
                "Adjusted P-value"
            )
        )

        print(
            "\n" + "=" * 100
        )

        print(
            f"{trajectory_name.upper()} "
            f"— {library_name}"
        )

        print("=" * 100)

        print(
            f"Terms tested with >=1 overlap: "
            f"{len(sub):,}"
        )

        print(
            f"FDR < 0.10 pathways: "
            f"{len(sig):,}"
        )

        if len(sig) > 0:

            display(
                sig[
                    [
                        "Term",
                        "Overlap",
                        "Odds Ratio",
                        "Adjusted P-value",
                        "Genes"
                    ]
                ]
                .head(25)
                .round(5)
            )

        else:

            print(
                "No pathways at FDR < 0.10."
            )


# ------------------------------------------------------------
# 9. Hypothesis-focused terms
# ------------------------------------------------------------

keywords = [
    "interferon",
    "immune",
    "retinoid",
    "vitamin",
    "glycosaminoglycan",
    "chondroitin",
    "extracellular",
    "matrix",
    "netrin",
    "potassium",
    "vascular",
    "angiogenesis",
    "endothelial",
    "metabolism",
    "metabolic",
    "serine",
    "lipid",
    "autophagy"
]

focus_fixed = enrichment_fixed[
    enrichment_fixed["Term"]
    .str.contains(
        "|".join(keywords),
        case=False,
        regex=True,
        na=False
    )
].copy()

focus_fixed = focus_fixed.sort_values(
    [
        "trajectory_class",
        "Adjusted P-value"
    ]
)

print(
    "\n" + "=" * 100
)

print(
    "HYPOTHESIS-RELEVANT TRAJECTORY PATHWAYS"
)

print("=" * 100)

display(
    focus_fixed[
        [
            "trajectory_class",
            "library",
            "Term",
            "Overlap",
            "query_mapped_n",
            "Odds Ratio",
            "Adjusted P-value",
            "Genes"
        ]
    ]
    .head(80)
    .round(5)
)


# ------------------------------------------------------------
# 10. Save
# ------------------------------------------------------------

out_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_trajectory_LOCAL_ORA_CASE_FIXED.csv"
)

coverage_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_trajectory_library_coverage.csv"
)

enrichment_fixed.to_csv(
    out_file,
    index=False
)

coverage.to_csv(
    coverage_file,
    index=False
)

print("\nSaved:")
print(out_file)
print(coverage_file)

print(
    "\nIMPORTANT: enrichment uses only genes "
    "annotated in each respective library."
)

print(
    "Mouse genes absent from a library "
    "(for example Ccl21a here) were not mapped "
    "to invented human orthologs."
)

In [ ]:
# ============================================================
# NEXT CELL — STATISTICALLY CORRECT LOCAL TRAJECTORY ORA
#
# Critical correction:
# BH multiple-testing adjustment is performed across ALL
# eligible pathway tests, including pathways with zero overlap.
#
# Same:
# - case-insensitive matching
# - library-specific annotated Müller background
# - original mouse symbols retained
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import hypergeom


# ------------------------------------------------------------
# 1. BH correction
# ------------------------------------------------------------

def bh_adjust_all(pvalues):

    p = np.asarray(pvalues, dtype=float)
    m = len(p)

    order = np.argsort(p)
    ranked = p[order]

    q = ranked * m / np.arange(1, m + 1)

    q = np.minimum.accumulate(
        q[::-1]
    )[::-1]

    q = np.clip(q, 0, 1)

    out = np.empty(m)
    out[order] = q

    return out


# ------------------------------------------------------------
# 2. Correct ORA function
# ------------------------------------------------------------

def corrected_local_ora_v2(
    query_original,
    library_upper,
    trajectory_name,
    library_name,
    min_pathway_size=5,
    max_pathway_size=500
):

    # union of library genes
    library_union = set()

    for genes in library_upper.values():
        library_union.update(genes)

    # Müller genes actually represented in this library
    annotated_background = (
        background_upper &
        library_union
    )

    # mapped query
    query_upper_all = {
        g.upper()
        for g in query_original
    }

    query_upper = (
        query_upper_all &
        annotated_background
    )

    M = len(annotated_background)
    N = len(query_upper)

    rows = []

    n_eligible_pathways = 0

    for term, pathway_all in library_upper.items():

        pathway = (
            pathway_all &
            annotated_background
        )

        K = len(pathway)

        if K < min_pathway_size:
            continue

        if K > max_pathway_size:
            continue

        # THIS pathway counts as a statistical test
        n_eligible_pathways += 1

        overlap_upper = (
            query_upper &
            pathway
        )

        x = len(overlap_upper)

        # zero overlap is still a tested hypothesis
        if x == 0:

            pval = 1.0
            odds_ratio = 0.0
            original_overlap = []

        else:

            # upper-tail hypergeometric
            pval = hypergeom.sf(
                x - 1,
                M,
                K,
                N
            )

            # Haldane-Anscombe corrected OR
            a = x
            b = N - x
            c = K - x
            d = M - K - N + x

            odds_ratio = (
                (a + 0.5) *
                (d + 0.5)
            ) / (
                (b + 0.5) *
                (c + 0.5)
            )

            original_overlap = []

            for symbol_upper in sorted(overlap_upper):

                original_overlap.extend(
                    background_map.get(
                        symbol_upper,
                        [symbol_upper]
                    )
                )

            original_overlap = sorted(
                set(original_overlap)
            )

        rows.append({
            "trajectory_class": trajectory_name,
            "library": library_name,

            "Term": term,

            "Overlap": f"{x}/{K}",
            "overlap_n": x,

            "pathway_background_n": K,

            "query_mapped_n": N,
            "query_original_n": len(query_original),

            "background_annotated_n": M,
            "background_original_n": len(background_original),

            "Odds Ratio": odds_ratio,
            "P-value": pval,

            "Genes": ";".join(original_overlap)
        })

    result = pd.DataFrame(rows)

    # IMPORTANT:
    # BH now includes every eligible pathway,
    # including zero-overlap pathways.
    result["Adjusted P-value"] = bh_adjust_all(
        result["P-value"].values
    )

    result["n_pathways_tested"] = (
        n_eligible_pathways
    )

    result = result.sort_values(
        [
            "Adjusted P-value",
            "P-value"
        ]
    )

    return result


# ------------------------------------------------------------
# 3. Run all 3 trajectories × 2 libraries
# ------------------------------------------------------------

all_results_v2 = []

for trajectory_name, query_genes in (
    trajectory_sets_original.items()
):

    for library_name, library_data in libraries:

        result = corrected_local_ora_v2(
            query_original=query_genes,
            library_upper=library_data,
            trajectory_name=trajectory_name,
            library_name=library_name
        )

        all_results_v2.append(result)


enrichment_v2 = pd.concat(
    all_results_v2,
    ignore_index=True
)


# ------------------------------------------------------------
# 4. Summary table
# ------------------------------------------------------------

summary_rows = []

for trajectory_name in trajectory_sets_original.keys():

    for library_name, _ in libraries:

        sub = enrichment_v2[
            (
                enrichment_v2["trajectory_class"]
                == trajectory_name
            )
            &
            (
                enrichment_v2["library"]
                == library_name
            )
        ]

        sig10 = sub[
            sub["Adjusted P-value"] < 0.10
        ]

        sig25 = sub[
            sub["Adjusted P-value"] < 0.25
        ]

        summary_rows.append({
            "trajectory_class": trajectory_name,
            "library": library_name,
            "pathways_tested": len(sub),
            "pathways_with_overlap": (
                sub["overlap_n"] > 0
            ).sum(),
            "FDR_lt_0.10": len(sig10),
            "FDR_lt_0.25": len(sig25)
        })


summary_v2 = pd.DataFrame(summary_rows)

print("=" * 100)
print("CORRECTED ORA SUMMARY")
print("=" * 100)

display(summary_v2)


# ------------------------------------------------------------
# 5. Print FDR < 0.10 results
# ------------------------------------------------------------

for trajectory_name in trajectory_sets_original.keys():

    for library_name, _ in libraries:

        sub = enrichment_v2[
            (
                enrichment_v2["trajectory_class"]
                == trajectory_name
            )
            &
            (
                enrichment_v2["library"]
                == library_name
            )
        ].copy()

        sig = (
            sub[
                sub["Adjusted P-value"] < 0.10
            ]
            .sort_values(
                "Adjusted P-value"
            )
        )

        print("\n" + "=" * 100)

        print(
            f"{trajectory_name.upper()} "
            f"— {library_name}"
        )

        print("=" * 100)

        if len(sig) == 0:

            print(
                "No pathways at FDR < 0.10."
            )

        else:

            display(
                sig[
                    [
                        "Term",
                        "Overlap",
                        "Odds Ratio",
                        "P-value",
                        "Adjusted P-value",
                        "Genes"
                    ]
                ]
                .head(25)
                .round(6)
            )


# ------------------------------------------------------------
# 6. Also show FDR < 0.25 for exploratory interpretation
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("EXPLORATORY FDR < 0.25 RESULTS")
print("=" * 100)

for trajectory_name in trajectory_sets_original.keys():

    for library_name, _ in libraries:

        sub = enrichment_v2[
            (
                enrichment_v2["trajectory_class"]
                == trajectory_name
            )
            &
            (
                enrichment_v2["library"]
                == library_name
            )
            &
            (
                enrichment_v2["Adjusted P-value"]
                < 0.25
            )
        ].copy()

        sub = sub.sort_values(
            "Adjusted P-value"
        )

        print(
            f"\n{trajectory_name} | "
            f"{library_name}: "
            f"{len(sub)} pathways"
        )

        if len(sub) > 0:

            display(
                sub[
                    [
                        "Term",
                        "Overlap",
                        "Odds Ratio",
                        "Adjusted P-value",
                        "Genes"
                    ]
                ]
                .head(15)
                .round(6)
            )


# ------------------------------------------------------------
# 7. Hypothesis-specific pathways
# ------------------------------------------------------------

keywords = [
    "interferon",
    "retinoid",
    "vitamin",
    "glycosaminoglycan",
    "chondroitin",
    "extracellular",
    "matrix",
    "collagen",
    "netrin",
    "potassium",
    "vascular",
    "angiogenesis",
    "endothelial",
    "metabolism",
    "serine",
    "lipid",
    "autophagy"
]

focus_v2 = enrichment_v2[
    enrichment_v2["Term"]
    .str.contains(
        "|".join(keywords),
        case=False,
        regex=True,
        na=False
    )
    &
    (
        enrichment_v2["overlap_n"] > 0
    )
].copy()

focus_v2 = focus_v2.sort_values(
    [
        "trajectory_class",
        "Adjusted P-value"
    ]
)

print("\n" + "=" * 100)
print("CORRECTED HYPOTHESIS-RELEVANT PATHWAYS")
print("=" * 100)

display(
    focus_v2[
        [
            "trajectory_class",
            "library",
            "Term",
            "Overlap",
            "Odds Ratio",
            "Adjusted P-value",
            "Genes"
        ]
    ]
    .head(80)
    .round(6)
)


# ------------------------------------------------------------
# 8. Save corrected result
# ------------------------------------------------------------

out_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_trajectory_LOCAL_ORA_FINAL.csv"
)

summary_file = (
    ROOT /
    "01_GSE328966_early_mouse" /
    "GSE328966_Muller_trajectory_LOCAL_ORA_FINAL_summary.csv"
)

enrichment_v2.to_csv(
    out_file,
    index=False
)

summary_v2.to_csv(
    summary_file,
    index=False
)

print("\nSaved:")
print(out_file)
print(summary_file)

print(
    "\nBH correction now includes ALL eligible pathway tests, "
    "including pathways with zero overlap."
)

In [ ]:
# ============================================================
# NEXT CELL — FREEZE GSE328966 DISCOVERY SIGNATURES
#
# Locks discovery definitions BEFORE independent validation.
#
# Primary:
#   1. full early-up transient trajectory
#   2. IFN-alpha/beta submodule
#   3. retinoid metabolism submodule
#   4. fat-soluble-vitamin submodule
#
# Secondary/exploratory:
#   ECM/GAG-related transient genes at Reactome FDR < 0.25
#
# Also freezes persistent/reversal gene lists descriptively,
# WITHOUT calling them enriched pathway programs.
# ============================================================

import json
import pandas as pd
from pathlib import Path

OUTDIR = (
    ROOT /
    "signatures"
)

OUTDIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Freeze full trajectory classes
# ------------------------------------------------------------

trajectory_signatures = {
    "early_up_transient_all": sorted(
        traj[
            traj["trajectory_class"] == "early_up_transient"
        ].index.astype(str).tolist()
    ),

    "persistent_up_all_descriptive": sorted(
        traj[
            traj["trajectory_class"] == "persistent_up"
        ].index.astype(str).tolist()
    ),

    "up_to_down_reversal_all_descriptive": sorted(
        traj[
            traj["trajectory_class"] == "up_to_down_reversal"
        ].index.astype(str).tolist()
    )
}


# ------------------------------------------------------------
# 2. Helper to extract genes from corrected Reactome ORA
# ------------------------------------------------------------

def genes_from_term(term_exact):

    hit = enrichment_v2[
        (enrichment_v2["trajectory_class"] == "early_up_transient") &
        (enrichment_v2["library"] == "Reactome_2022") &
        (enrichment_v2["Term"] == term_exact)
    ]

    if len(hit) == 0:
        raise ValueError(
            f"Reactome term not found: {term_exact}"
        )

    gene_string = hit.iloc[0]["Genes"]

    genes = [
        g.strip()
        for g in str(gene_string).split(";")
        if g.strip()
    ]

    return sorted(
        set(genes)
    )


# ------------------------------------------------------------
# 3. PRIMARY pathway-derived modules
#    FDR < 0.10 in corrected ORA
# ------------------------------------------------------------

primary_modules = {
    "early_IFN_alpha_beta": genes_from_term(
        "Interferon Alpha/Beta Signaling R-HSA-909733"
    ),

    "early_retinoid_metabolism": genes_from_term(
        "Retinoid Metabolism And Transport R-HSA-975634"
    ),

    "early_fat_soluble_vitamin_metabolism": genes_from_term(
        "Metabolism Of Fat-Soluble Vitamins R-HSA-6806667"
    )
}


# ------------------------------------------------------------
# 4. SECONDARY exploratory ECM/GAG module
#
# Only terms:
# - early_up_transient
# - Reactome
# - FDR < 0.25
# - ECM/GAG/collagen related
# ------------------------------------------------------------

ecm_keywords = [
    "glycosaminoglycan",
    "gag",
    "heparan sulfate",
    "extracellular matrix",
    "collagen"
]

ecm_mask = (
    (enrichment_v2["trajectory_class"] == "early_up_transient") &
    (enrichment_v2["library"] == "Reactome_2022") &
    (enrichment_v2["Adjusted P-value"] < 0.25) &
    (
        enrichment_v2["Term"]
        .str.contains(
            "|".join(ecm_keywords),
            case=False,
            regex=True,
            na=False
        )
    )
)

ecm_terms = enrichment_v2[
    ecm_mask
].copy()

ecm_genes = set()

for gene_string in ecm_terms["Genes"]:

    for g in str(gene_string).split(";"):

        g = g.strip()

        if g:
            ecm_genes.add(g)

secondary_modules = {
    "early_ECM_GAG_exploratory": sorted(
        ecm_genes
    )
}


# ------------------------------------------------------------
# 5. Combine frozen signatures
# ------------------------------------------------------------

locked_signatures = {}

locked_signatures.update(
    trajectory_signatures
)

locked_signatures.update(
    primary_modules
)

locked_signatures.update(
    secondary_modules
)


# ------------------------------------------------------------
# 6. Long-format export
# ------------------------------------------------------------

rows = []

for signature_name, genes in locked_signatures.items():

    if signature_name in primary_modules:
        evidence_level = "PRIMARY_FDR_lt_0.10"

    elif signature_name == "early_ECM_GAG_exploratory":
        evidence_level = "SECONDARY_FDR_lt_0.25"

    else:
        evidence_level = "DESCRIPTIVE_TRAJECTORY"

    for gene in genes:

        rows.append({
            "signature": signature_name,
            "gene": gene,
            "evidence_level": evidence_level,
            "discovery_dataset": "GSE328966"
        })

signature_table = pd.DataFrame(
    rows
)


# ------------------------------------------------------------
# 7. Print locked signature summary
# ------------------------------------------------------------

summary = (
    signature_table
    .groupby(
        [
            "signature",
            "evidence_level"
        ]
    )
    .size()
    .rename("n_genes")
    .reset_index()
)

print("=" * 100)
print("LOCKED GSE328966 DISCOVERY SIGNATURES")
print("=" * 100)

display(summary)


print("\n" + "=" * 100)
print("PRIMARY: INTERFERON ALPHA/BETA")
print("=" * 100)

print(
    primary_modules[
        "early_IFN_alpha_beta"
    ]
)


print("\n" + "=" * 100)
print("PRIMARY: RETINOID METABOLISM")
print("=" * 100)

print(
    primary_modules[
        "early_retinoid_metabolism"
    ]
)


print("\n" + "=" * 100)
print("PRIMARY: FAT-SOLUBLE VITAMIN METABOLISM")
print("=" * 100)

print(
    primary_modules[
        "early_fat_soluble_vitamin_metabolism"
    ]
)


print("\n" + "=" * 100)
print("SECONDARY: ECM/GAG EXPLORATORY")
print("=" * 100)

print(
    secondary_modules[
        "early_ECM_GAG_exploratory"
    ]
)

print("\nSource exploratory Reactome terms:")

display(
    ecm_terms[
        [
            "Term",
            "Adjusted P-value",
            "Genes"
        ]
    ]
    .sort_values(
        "Adjusted P-value"
    )
    .round(6)
)


# ------------------------------------------------------------
# 8. Save CSV
# ------------------------------------------------------------

csv_file = (
    OUTDIR /
    "GSE328966_LOCKED_discovery_signatures.csv"
)

signature_table.to_csv(
    csv_file,
    index=False
)


# ------------------------------------------------------------
# 9. Save JSON for easy reuse in later datasets
# ------------------------------------------------------------

json_file = (
    OUTDIR /
    "GSE328966_LOCKED_discovery_signatures.json"
)

with open(
    json_file,
    "w"
) as f:

    json.dump(
        locked_signatures,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 10. Freeze a small provenance record
# ------------------------------------------------------------

provenance = {
    "dataset": "GSE328966",
    "cell_type": "Muller_glia_cluster5",
    "early_comparison": "DM5D_vs_Control5D",
    "late_comparison": "DM15D_vs_Control15D",

    "primary_pathway_rule":
        "early_up_transient Reactome corrected local ORA FDR < 0.10",

    "secondary_pathway_rule":
        "early_up_transient Reactome corrected local ORA FDR < 0.25",

    "notes": [
        "one pooled library per condition",
        "5D effects depth-standardized using repeated downsampling",
        "trajectory classes are descriptive",
        "no causal/protective interpretation assigned",
        "persistent and reversal classes showed no pathway enrichment at FDR < 0.25"
    ]
}

provenance_file = (
    OUTDIR /
    "GSE328966_LOCKED_discovery_provenance.json"
)

with open(
    provenance_file,
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=2
    )


print("\nSaved:")
print(csv_file)
print(json_file)
print(provenance_file)

print(
    "\nThese signatures are now frozen for independent validation. "
    "Do not alter them based on GSE313176 or human results."
)

## Part II — GSE313176 longitudinal mouse validation

In [ ]:
# ============================================================
# GSE313176 — CELL 1
# GEO SAMPLE-DESIGN / FILE AUDIT
#
# Purpose:
# - retrieve GEO metadata
# - enumerate every sample
# - distinguish bulk vs scRNA
# - identify condition, timepoint, compartment
# - inspect supplementary files
#
# NO EXPRESSION ANALYSIS YET.
# ============================================================

import gzip
import re
import urllib.request
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

GSE = "GSE313176"

OUTDIR = (
    ROOT /
    "04_GSE313176_mouse_longitudinal"
)

OUTDIR.mkdir(
    parents=True,
    exist_ok=True
)

soft_url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE313nnn/GSE313176/soft/"
    "GSE313176_family.soft.gz"
)

soft_file = (
    OUTDIR /
    "GSE313176_family.soft.gz"
)

# ------------------------------------------------------------
# 2. Download GEO family SOFT metadata
# ------------------------------------------------------------

if not soft_file.exists():

    print("Downloading GEO metadata...")

    urllib.request.urlretrieve(
        soft_url,
        soft_file
    )

else:

    print("Metadata already downloaded.")


# ------------------------------------------------------------
# 3. Parse sample sections
# ------------------------------------------------------------

samples = []

current = None

with gzip.open(
    soft_file,
    "rt",
    errors="replace"
) as f:

    for line in f:

        line = line.rstrip("\n")

        # New GSM section
        if line.startswith("^SAMPLE = "):

            if current is not None:
                samples.append(current)

            current = {
                "gsm": line.split("=", 1)[1].strip(),
                "title": None,
                "source": None,
                "platform": None,
                "characteristics": [],
                "supplementary_files": []
            }

            continue

        if current is None:
            continue

        if line.startswith("!Sample_title = "):

            current["title"] = (
                line.split("=", 1)[1].strip()
            )

        elif line.startswith(
            "!Sample_source_name_ch1 = "
        ):

            current["source"] = (
                line.split("=", 1)[1].strip()
            )

        elif line.startswith(
            "!Sample_platform_id = "
        ):

            current["platform"] = (
                line.split("=", 1)[1].strip()
            )

        elif line.startswith(
            "!Sample_characteristics_ch1 = "
        ):

            current["characteristics"].append(
                line.split("=", 1)[1].strip()
            )

        elif line.startswith(
            "!Sample_supplementary_file"
        ):

            current["supplementary_files"].append(
                line.split("=", 1)[1].strip()
            )

# append final sample
if current is not None:
    samples.append(current)


# ------------------------------------------------------------
# 4. Convert to table + classify assay/design
# ------------------------------------------------------------

rows = []

for s in samples:

    title = str(s["title"] or "")
    source = str(s["source"] or "")

    supp = ";".join(
        s["supplementary_files"]
    )

    combined = " ".join(
        [
            title,
            source,
            " ".join(s["characteristics"]),
            supp
        ]
    )

    low = combined.lower()

    # ---------------- assay ----------------

    is_scrna = (
        "matrix.mtx" in low
        or "barcodes.tsv" in low
        or s["platform"] == "GPL24247"
    )

    assay = (
        "scRNA"
        if is_scrna
        else "bulk_RNA"
    )

    # ---------------- condition ----------------

    if (
        "stz" in low
        or "diabetic" in low
        or "diabetes" in low
    ):
        condition = "STZ"

    elif (
        "sham" in low
        or "control" in low
    ):
        condition = "Sham"

    else:
        condition = "UNKNOWN"

    # ---------------- time ----------------

    timepoint = "UNKNOWN"

    time_patterns = [
        (r"\b1\s*mo(?:nth)?s?\b", "1mo"),
        (r"\b3\s*mo(?:nth)?s?\b", "3mo"),
        (r"\b6\s*mo(?:nth)?s?\b", "6mo"),
        (r"\b1m\b", "1mo"),
        (r"\b3m\b", "3mo"),
        (r"\b6m\b", "6mo")
    ]

    for pattern, label in time_patterns:

        if re.search(
            pattern,
            low
        ):

            timepoint = label
            break

    # scRNA study is explicitly 3 months
    if assay == "scRNA" and timepoint == "UNKNOWN":
        timepoint = "3mo"

    # ---------------- compartment ----------------

    if assay == "scRNA":

        compartment = "whole_retina_scRNA"

    elif "endo" in low or "endothelial" in low:

        compartment = "endothelial"

    elif "retina" in low or "retinal" in low:

        compartment = "retina"

    else:

        compartment = "UNKNOWN"

    rows.append({
        "gsm": s["gsm"],
        "title": title,
        "assay": assay,
        "condition": condition,
        "timepoint": timepoint,
        "compartment": compartment,
        "platform": s["platform"],
        "source": source,
        "characteristics": " | ".join(
            s["characteristics"]
        ),
        "n_supp_files": len(
            s["supplementary_files"]
        ),
        "supplementary_files": supp
    })


meta313176 = pd.DataFrame(rows)


# ------------------------------------------------------------
# 5. Sort logically
# ------------------------------------------------------------

time_order = {
    "1mo": 1,
    "3mo": 3,
    "6mo": 6,
    "UNKNOWN": 99
}

condition_order = {
    "Sham": 0,
    "STZ": 1,
    "UNKNOWN": 9
}

comp_order = {
    "retina": 0,
    "endothelial": 1,
    "whole_retina_scRNA": 2,
    "UNKNOWN": 9
}

meta313176["_time"] = (
    meta313176["timepoint"]
    .map(time_order)
    .fillna(99)
)

meta313176["_cond"] = (
    meta313176["condition"]
    .map(condition_order)
    .fillna(9)
)

meta313176["_comp"] = (
    meta313176["compartment"]
    .map(comp_order)
    .fillna(9)
)

meta313176 = (
    meta313176
    .sort_values(
        [
            "assay",
            "_time",
            "_comp",
            "_cond",
            "gsm"
        ]
    )
    .drop(
        columns=[
            "_time",
            "_cond",
            "_comp"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 6. Full sample design
# ------------------------------------------------------------

print("=" * 100)
print("GSE313176 — SAMPLE DESIGN")
print("=" * 100)

display(
    meta313176[
        [
            "gsm",
            "title",
            "assay",
            "condition",
            "timepoint",
            "compartment",
            "platform",
            "n_supp_files"
        ]
    ]
)


# ------------------------------------------------------------
# 7. Experimental-design counts
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("BULK RNA-SEQ DESIGN")
print("=" * 100)

bulk_design = (
    meta313176[
        meta313176["assay"] == "bulk_RNA"
    ]
    .groupby(
        [
            "timepoint",
            "compartment",
            "condition"
        ],
        dropna=False
    )
    .size()
    .rename("n_samples")
    .reset_index()
)

display(
    bulk_design
)


print("\n" + "=" * 100)
print("scRNA-SEQ DESIGN")
print("=" * 100)

scrna_design = (
    meta313176[
        meta313176["assay"] == "scRNA"
    ]
    .groupby(
        [
            "timepoint",
            "condition"
        ],
        dropna=False
    )
    .size()
    .rename("n_libraries")
    .reset_index()
)

display(
    scrna_design
)


# ------------------------------------------------------------
# 8. Supplementary-file audit
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SUPPLEMENTARY FILES")
print("=" * 100)

for _, row in meta313176.iterrows():

    print(
        f"\n{row['gsm']} | "
        f"{row['title']} | "
        f"{row['assay']}"
    )

    files = [
        x
        for x in str(
            row["supplementary_files"]
        ).split(";")
        if x
    ]

    for x in files:
        print("   ", x)


# ------------------------------------------------------------
# 9. Sanity checks
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SANITY CHECK")
print("=" * 100)

print(
    "Total GEO samples:",
    len(meta313176)
)

print(
    "Bulk samples:",
    (meta313176["assay"] == "bulk_RNA").sum()
)

print(
    "scRNA libraries:",
    (meta313176["assay"] == "scRNA").sum()
)

print(
    "Unclassified condition:",
    (meta313176["condition"] == "UNKNOWN").sum()
)

print(
    "Unclassified timepoint:",
    (meta313176["timepoint"] == "UNKNOWN").sum()
)

print(
    "Unclassified compartment:",
    (meta313176["compartment"] == "UNKNOWN").sum()
)


# ------------------------------------------------------------
# 10. Save manifest
# ------------------------------------------------------------

manifest_file = (
    OUTDIR /
    "GSE313176_sample_manifest.csv"
)

meta313176.to_csv(
    manifest_file,
    index=False
)

print("\nSaved:")
print(manifest_file)

In [ ]:
# ============================================================
# GSE313176 — CELL 2
# DOWNLOAD + FORMAT AUDIT OF 18 LONGITUDINAL RETINA BULK FILES
#
# No differential expression yet.
# No signature scoring yet.
#
# Goal:
# - download all 18 replicated retina bulk tables
# - confirm file integrity
# - inspect header/schema
# - determine whether values are counts / normalized expression
# ============================================================

import gzip
import os
import urllib.request
from urllib.parse import urlparse
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------
# 1. Select ONLY whole-retina bulk samples
# ------------------------------------------------------------

retina_bulk_meta = meta313176[
    (meta313176["assay"] == "bulk_RNA") &
    (meta313176["compartment"] == "retina")
].copy()

retina_bulk_meta = retina_bulk_meta.reset_index(
    drop=True
)

print("=" * 100)
print("GSE313176 — RETINA BULK DOWNLOAD AUDIT")
print("=" * 100)

print(
    f"\nRetina bulk samples selected: "
    f"{len(retina_bulk_meta)}"
)

display(
    retina_bulk_meta[
        [
            "gsm",
            "title",
            "condition",
            "timepoint",
            "platform"
        ]
    ]
)

assert len(retina_bulk_meta) == 18, (
    f"Expected 18 retinal bulk samples, "
    f"found {len(retina_bulk_meta)}"
)


# ------------------------------------------------------------
# 2. Download directory
# ------------------------------------------------------------

BULK_DIR = (
    OUTDIR /
    "bulk_retina_processed"
)

BULK_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 3. Download each processed table
# ------------------------------------------------------------

download_rows = []

for _, row in retina_bulk_meta.iterrows():

    urls = [
        x
        for x in str(
            row["supplementary_files"]
        ).split(";")
        if x
    ]

    if len(urls) != 1:

        raise ValueError(
            f"{row['gsm']} expected 1 supplementary "
            f"file, found {len(urls)}"
        )

    url = urls[0]

    # NCBI FTP URLs work more reliably in Colab via HTTPS
    if url.startswith(
        "ftp://ftp.ncbi.nlm.nih.gov/"
    ):

        url = url.replace(
            "ftp://ftp.ncbi.nlm.nih.gov/",
            "https://ftp.ncbi.nlm.nih.gov/"
        )

    filename = Path(
        urlparse(url).path
    ).name

    destination = (
        BULK_DIR /
        filename
    )

    if not destination.exists():

        print(
            f"Downloading {row['gsm']} ..."
        )

        urllib.request.urlretrieve(
            url,
            destination
        )

    size_bytes = destination.stat().st_size

    download_rows.append({
        "gsm": row["gsm"],
        "title": row["title"],
        "condition": row["condition"],
        "timepoint": row["timepoint"],
        "filename": filename,
        "size_MB": size_bytes / (1024 ** 2),
        "local_path": str(destination)
    })


download_manifest = pd.DataFrame(
    download_rows
)

print("\n" + "=" * 100)
print("DOWNLOAD MANIFEST")
print("=" * 100)

display(
    download_manifest[
        [
            "gsm",
            "timepoint",
            "condition",
            "filename",
            "size_MB"
        ]
    ].round(3)
)

print(
    "\nTotal downloaded size: "
    f"{download_manifest['size_MB'].sum():.2f} MB"
)


# ------------------------------------------------------------
# 4. Read raw header + first lines from every file
# ------------------------------------------------------------

schema_rows = []

for _, row in download_manifest.iterrows():

    path = Path(
        row["local_path"]
    )

    with gzip.open(
        path,
        "rt",
        errors="replace"
    ) as f:

        first_line = f.readline().rstrip(
            "\n"
        )

        second_line = f.readline().rstrip(
            "\n"
        )

    # first try tab-separated header
    header_fields = first_line.split(
        "\t"
    )

    schema_rows.append({
        "gsm": row["gsm"],
        "timepoint": row["timepoint"],
        "condition": row["condition"],
        "n_header_fields": len(
            header_fields
        ),
        "header": first_line,
        "first_data_line": second_line
    })


schema_audit = pd.DataFrame(
    schema_rows
)


# ------------------------------------------------------------
# 5. Check whether all 18 have identical headers
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("HEADER / SCHEMA AUDIT")
print("=" * 100)

print(
    "Unique header strings:",
    schema_audit["header"].nunique()
)

print(
    "Unique numbers of header fields:",
    sorted(
        schema_audit[
            "n_header_fields"
        ].unique()
    )
)

print("\nHeader(s):")

for i, header in enumerate(
    schema_audit["header"].drop_duplicates(),
    start=1
):

    print(
        f"\nHEADER {i}:"
    )

    print(header)


# ------------------------------------------------------------
# 6. Parse first 5 rows of one representative file
#
# First try TAB.
# If parsing gives a single column, fall back to automatic
# delimiter detection.
# ------------------------------------------------------------

example_path = Path(
    download_manifest.iloc[0][
        "local_path"
    ]
)

try:

    preview = pd.read_csv(
        example_path,
        sep="\t",
        compression="gzip",
        nrows=5
    )

    if preview.shape[1] == 1:

        preview = pd.read_csv(
            example_path,
            sep=None,
            engine="python",
            compression="gzip",
            nrows=5
        )

except Exception:

    preview = pd.read_csv(
        example_path,
        sep=None,
        engine="python",
        compression="gzip",
        nrows=5
    )


print("\n" + "=" * 100)
print("REPRESENTATIVE FILE PREVIEW")
print("=" * 100)

print(
    download_manifest.iloc[0][
        "filename"
    ]
)

print(
    "\nParsed columns:"
)

print(
    preview.columns.tolist()
)

print(
    "\nParsed shape of preview:",
    preview.shape
)

display(
    preview
)


# ------------------------------------------------------------
# 7. Raw first two lines from one sample at each timepoint
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("RAW FILE EXAMPLES BY TIMEPOINT")
print("=" * 100)

for tp in [
    "1mo",
    "3mo",
    "6mo"
]:

    sub = schema_audit[
        (
            schema_audit["timepoint"]
            == tp
        )
        &
        (
            schema_audit["condition"]
            == "Sham"
        )
    ]

    if len(sub) == 0:
        continue

    r = sub.iloc[0]

    print(
        f"\n--- {tp} | {r['gsm']} ---"
    )

    print(
        "HEADER:"
    )

    print(
        r["header"]
    )

    print(
        "FIRST DATA LINE:"
    )

    print(
        r["first_data_line"]
    )


# ------------------------------------------------------------
# 8. Save download manifest
# ------------------------------------------------------------

download_manifest_file = (
    OUTDIR /
    "GSE313176_retina_bulk_download_manifest.csv"
)

schema_file = (
    OUTDIR /
    "GSE313176_retina_bulk_schema_audit.csv"
)

download_manifest.to_csv(
    download_manifest_file,
    index=False
)

schema_audit.to_csv(
    schema_file,
    index=False
)

print("\nSaved:")
print(download_manifest_file)
print(schema_file)

print(
    "\nSTOP HERE. "
    "We need to identify the expression unit/schema "
    "before combining samples."
)

In [ ]:
# ============================================================
# GSE313176 — CELL 3
# BUILD + QC AUDIT OF 18-SAMPLE RETINA RAW COUNT MATRIX
#
# NO differential expression.
# NO normalization for inference.
# NO signature testing.
#
# Goal:
# - merge all raw-count files
# - prove rows/IDs are consistent
# - audit library size / complexity
# - inspect replicate structure and gross outliers
# ============================================================

import gzip
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr


# ------------------------------------------------------------
# 1. Read all 18 retinal bulk files
# ------------------------------------------------------------

sample_tables = {}
gene_orders = {}

for _, row in download_manifest.iterrows():

    gsm = row["gsm"]
    path = Path(row["local_path"])

    df = pd.read_csv(
        path,
        sep="\t",
        compression="gzip"
    )

    # basic schema check
    assert list(df.columns) == ["Gene", "Counts"], (
        f"{gsm}: unexpected columns {df.columns.tolist()}"
    )

    # enforce strings for IDs
    df["Gene"] = df["Gene"].astype(str)

    # counts must be numeric
    df["Counts"] = pd.to_numeric(
        df["Counts"],
        errors="raise"
    )

    sample_tables[gsm] = df
    gene_orders[gsm] = df["Gene"].tolist()


# ------------------------------------------------------------
# 2. Row/gene-universe consistency
# ------------------------------------------------------------

reference_gsm = download_manifest.iloc[0]["gsm"]

reference_genes = gene_orders[reference_gsm]

same_gene_order = {
    gsm: (
        genes == reference_genes
    )
    for gsm, genes in gene_orders.items()
}

print("=" * 100)
print("COUNT MATRIX STRUCTURE AUDIT")
print("=" * 100)

print(
    "\nAll samples have identical gene order:",
    all(same_gene_order.values())
)

print(
    "Number of genes in reference sample:",
    len(reference_genes)
)

# unique IDs
print(
    "Unique gene IDs in reference:",
    len(set(reference_genes))
)

print(
    "Duplicated gene IDs in reference:",
    len(reference_genes)
    - len(set(reference_genes))
)

if not all(same_gene_order.values()):

    print("\nSamples with non-identical row order:")

    for gsm, ok in same_gene_order.items():

        if not ok:
            print(" ", gsm)

    raise RuntimeError(
        "Gene rows are not identical across samples. "
        "Stop before merging."
    )


# ------------------------------------------------------------
# 3. Construct gene × sample raw-count matrix
# ------------------------------------------------------------

count_matrix = pd.DataFrame(
    {
        gsm: sample_tables[gsm]["Counts"].to_numpy()
        for gsm in download_manifest["gsm"]
    },
    index=reference_genes
)

count_matrix.index.name = "ensembl_gene_id"


# ------------------------------------------------------------
# 4. Validate raw-count properties
# ------------------------------------------------------------

all_nonnegative = (
    count_matrix.to_numpy() >= 0
).all()

all_integer = np.all(
    np.equal(
        count_matrix.to_numpy(),
        np.floor(count_matrix.to_numpy())
    )
)

any_nan = count_matrix.isna().any().any()

print("\n" + "=" * 100)
print("RAW COUNT VALIDATION")
print("=" * 100)

print(
    "Matrix shape:",
    count_matrix.shape
)

print(
    "All counts non-negative:",
    all_nonnegative
)

print(
    "All counts integer-valued:",
    all_integer
)

print(
    "Any missing values:",
    any_nan
)

assert all_nonnegative
assert all_integer
assert not any_nan


# ------------------------------------------------------------
# 5. Sample-level QC
# ------------------------------------------------------------

qc_rows = []

for _, row in download_manifest.iterrows():

    gsm = row["gsm"]

    counts = count_matrix[gsm]

    library_size = counts.sum()

    detected = (
        counts > 0
    ).sum()

    genes_ge10 = (
        counts >= 10
    ).sum()

    genes_ge100 = (
        counts >= 100
    ).sum()

    zero_fraction = (
        counts == 0
    ).mean()

    # fraction of total reads in top 10 expressed genes
    top10_fraction = (
        counts.nlargest(10).sum()
        / library_size
        if library_size > 0
        else np.nan
    )

    qc_rows.append({
        "gsm": gsm,
        "timepoint": row["timepoint"],
        "condition": row["condition"],

        "library_size": int(
            library_size
        ),

        "detected_genes": int(
            detected
        ),

        "genes_counts_ge10": int(
            genes_ge10
        ),

        "genes_counts_ge100": int(
            genes_ge100
        ),

        "zero_fraction": zero_fraction,

        "top10_read_fraction": top10_fraction
    })


sample_qc = pd.DataFrame(
    qc_rows
)

print("\n" + "=" * 100)
print("SAMPLE-LEVEL RAW COUNT QC")
print("=" * 100)

display(
    sample_qc.round(4)
)


# ------------------------------------------------------------
# 6. QC summaries by timepoint / condition
# ------------------------------------------------------------

qc_summary = (
    sample_qc
    .groupby(
        [
            "timepoint",
            "condition"
        ]
    )
    .agg(
        n_samples=(
            "gsm",
            "size"
        ),

        median_library_size=(
            "library_size",
            "median"
        ),

        min_library_size=(
            "library_size",
            "min"
        ),

        max_library_size=(
            "library_size",
            "max"
        ),

        median_detected_genes=(
            "detected_genes",
            "median"
        ),

        median_zero_fraction=(
            "zero_fraction",
            "median"
        )
    )
    .reset_index()
)

print("\n" + "=" * 100)
print("QC SUMMARY BY GROUP")
print("=" * 100)

display(
    qc_summary.round(4)
)


# ------------------------------------------------------------
# 7. Log-CPM ONLY for unsupervised QC
#
# This is not the inferential model.
# ------------------------------------------------------------

library_sizes = count_matrix.sum(
    axis=0
)

cpm = (
    count_matrix
    .divide(
        library_sizes,
        axis=1
    )
    * 1e6
)

logcpm = np.log2(
    cpm + 0.5
)


# ------------------------------------------------------------
# 8. Filter extremely uninformative genes for correlation QC
#
# Require >= 1 CPM in at least 3 samples.
# This is QC only.
# ------------------------------------------------------------

qc_gene_mask = (
    cpm >= 1
).sum(
    axis=1
) >= 3

logcpm_qc = logcpm.loc[
    qc_gene_mask
]

print("\nGenes used for correlation QC:",
      logcpm_qc.shape[0])


# ------------------------------------------------------------
# 9. Sample Spearman correlation
# ------------------------------------------------------------

corr = logcpm_qc.corr(
    method="spearman"
)

# median correlation of each sample to all other samples
median_corr = {}

for gsm in corr.columns:

    vals = corr.loc[
        corr.index != gsm,
        gsm
    ]

    median_corr[gsm] = vals.median()

sample_qc[
    "median_spearman_to_all"
] = sample_qc["gsm"].map(
    median_corr
)


# ------------------------------------------------------------
# 10. Within-group replicate correlations
# ------------------------------------------------------------

within_group_rows = []

for (
    tp,
    cond
), group in sample_qc.groupby(
    [
        "timepoint",
        "condition"
    ]
):

    gsms = group["gsm"].tolist()

    pair_values = []

    for i in range(
        len(gsms)
    ):

        for j in range(
            i + 1,
            len(gsms)
        ):

            pair_values.append(
                corr.loc[
                    gsms[i],
                    gsms[j]
                ]
            )

    within_group_rows.append({
        "timepoint": tp,
        "condition": cond,
        "n_samples": len(gsms),

        "median_pairwise_spearman": (
            np.median(pair_values)
            if len(pair_values) > 0
            else np.nan
        ),

        "min_pairwise_spearman": (
            np.min(pair_values)
            if len(pair_values) > 0
            else np.nan
        ),

        "max_pairwise_spearman": (
            np.max(pair_values)
            if len(pair_values) > 0
            else np.nan
        )
    })


within_group_corr = pd.DataFrame(
    within_group_rows
)

print("\n" + "=" * 100)
print("WITHIN-GROUP REPLICATE CORRELATIONS")
print("=" * 100)

display(
    within_group_corr.round(4)
)


# ------------------------------------------------------------
# 11. Lowest-correlating samples
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("LOWEST GLOBAL SAMPLE CORRELATIONS")
print("=" * 100)

display(
    sample_qc[
        [
            "gsm",
            "timepoint",
            "condition",
            "library_size",
            "detected_genes",
            "median_spearman_to_all"
        ]
    ]
    .sort_values(
        "median_spearman_to_all"
    )
    .head(8)
    .round(4)
)


# ------------------------------------------------------------
# 12. Library-size imbalance STZ / Sham at each timepoint
# ------------------------------------------------------------

depth_rows = []

for tp in [
    "1mo",
    "3mo",
    "6mo"
]:

    sub = sample_qc[
        sample_qc["timepoint"] == tp
    ]

    sham_median = sub.loc[
        sub["condition"] == "Sham",
        "library_size"
    ].median()

    stz_median = sub.loc[
        sub["condition"] == "STZ",
        "library_size"
    ].median()

    depth_rows.append({
        "timepoint": tp,
        "median_sham_library": sham_median,
        "median_STZ_library": stz_median,

        "STZ_to_Sham_depth_ratio": (
            stz_median / sham_median
            if sham_median > 0
            else np.nan
        )
    })


depth_audit = pd.DataFrame(
    depth_rows
)

print("\n" + "=" * 100)
print("LIBRARY DEPTH BALANCE")
print("=" * 100)

display(
    depth_audit.round(4)
)


# ------------------------------------------------------------
# 13. Save matrices + QC
# ------------------------------------------------------------

count_file = (
    OUTDIR /
    "GSE313176_retina_bulk_RAW_COUNTS.csv.gz"
)

qc_file = (
    OUTDIR /
    "GSE313176_retina_bulk_sample_QC.csv"
)

corr_file = (
    OUTDIR /
    "GSE313176_retina_bulk_sample_spearman.csv"
)

depth_file = (
    OUTDIR /
    "GSE313176_retina_bulk_depth_audit.csv"
)

count_matrix.to_csv(
    count_file,
    compression="gzip"
)

sample_qc.to_csv(
    qc_file,
    index=False
)

corr.to_csv(
    corr_file
)

depth_audit.to_csv(
    depth_file,
    index=False
)


print("\nSaved:")
print(count_file)
print(qc_file)
print(corr_file)
print(depth_file)

print(
    "\nSTOP HERE. "
    "Next step depends on whether all 18 libraries "
    "and replicates pass this QC audit."
)

In [ ]:
# ============================================================
# GSE313176 — CELL 4
# ENSEMBL → MOUSE GENE SYMBOL MAPPING + LOCKED SIGNATURE COVERAGE
#
# NO differential expression yet.
# NO signature testing yet.
#
# Goals:
# - map ENSMUSG IDs reproducibly
# - preserve Ensembl IDs as count-matrix identifiers
# - audit duplicated / missing symbols
# - determine coverage of the frozen GSE328966 signatures
# ============================================================

import sys
import subprocess
import json
import numpy as np
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. Install/import MyGene
# ------------------------------------------------------------

try:
    import mygene
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "mygene"
        ]
    )
    import mygene


# ------------------------------------------------------------
# 2. Prepare Ensembl IDs
# ------------------------------------------------------------

ensembl_ids = (
    count_matrix.index
    .astype(str)
    .str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
    .tolist()
)

print("=" * 100)
print("ENSEMBL → SYMBOL ANNOTATION")
print("=" * 100)

print(
    f"\nEnsembl IDs submitted: "
    f"{len(ensembl_ids):,}"
)

print(
    "Unique Ensembl IDs:",
    len(set(ensembl_ids))
)


# ------------------------------------------------------------
# 3. Query MyGene in one batch
#
# species=mouse
# scope=ensembl.gene
# ------------------------------------------------------------

mg = mygene.MyGeneInfo()

mapping_raw = mg.querymany(
    ensembl_ids,
    scopes="ensembl.gene",
    fields=[
        "symbol",
        "name",
        "entrezgene"
    ],
    species="mouse",
    as_dataframe=False,
    returnall=False,
    verbose=False
)


# ------------------------------------------------------------
# 4. Parse mapping carefully
#
# querymany can theoretically return multiple records for
# one query; keep first unique symbol per Ensembl ID but record
# ambiguity.
# ------------------------------------------------------------

records_by_id = {}

for rec in mapping_raw:

    query = str(
        rec.get("query", "")
    )

    if not query:
        continue

    if query not in records_by_id:
        records_by_id[query] = []

    records_by_id[query].append(
        rec
    )


mapping_rows = []

for eid in ensembl_ids:

    records = records_by_id.get(
        eid,
        []
    )

    valid = [
        r for r in records
        if not r.get("notfound", False)
    ]

    symbols = sorted(
        {
            str(r["symbol"])
            for r in valid
            if r.get("symbol")
        }
    )

    names = sorted(
        {
            str(r["name"])
            for r in valid
            if r.get("name")
        }
    )

    entrez = sorted(
        {
            str(r["entrezgene"])
            for r in valid
            if r.get("entrezgene") is not None
        }
    )

    mapping_rows.append({
        "ensembl_gene_id": eid,
        "symbol": (
            symbols[0]
            if len(symbols) >= 1
            else np.nan
        ),
        "gene_name": (
            names[0]
            if len(names) >= 1
            else np.nan
        ),
        "entrezgene": (
            entrez[0]
            if len(entrez) >= 1
            else np.nan
        ),
        "n_symbols_returned": len(
            symbols
        ),
        "symbol_ambiguous": (
            len(symbols) > 1
        )
    })


gene_annotation = pd.DataFrame(
    mapping_rows
).drop_duplicates(
    subset="ensembl_gene_id",
    keep="first"
).set_index(
    "ensembl_gene_id"
)


# ------------------------------------------------------------
# 5. Mapping QC
# ------------------------------------------------------------

n_total = len(
    gene_annotation
)

n_mapped = gene_annotation[
    "symbol"
].notna().sum()

n_unmapped = (
    n_total -
    n_mapped
)

n_ambiguous = gene_annotation[
    "symbol_ambiguous"
].sum()

# duplicated symbols among distinct Ensembl IDs
symbol_counts = (
    gene_annotation[
        "symbol"
    ]
    .dropna()
    .value_counts()
)

duplicated_symbols = (
    symbol_counts[
        symbol_counts > 1
    ]
)

print("\n" + "=" * 100)
print("ANNOTATION QC")
print("=" * 100)

print(
    f"Total Ensembl genes:       {n_total:,}"
)

print(
    f"Mapped to symbol:          {n_mapped:,} "
    f"({100*n_mapped/n_total:.2f}%)"
)

print(
    f"Unmapped:                  {n_unmapped:,} "
    f"({100*n_unmapped/n_total:.2f}%)"
)

print(
    f"Ambiguous Ensembl queries: {n_ambiguous:,}"
)

print(
    f"Symbols represented by >1 Ensembl ID: "
    f"{len(duplicated_symbols):,}"
)


# ------------------------------------------------------------
# 6. Expression-aware mapping QC
#
# More important than mapping all 54k annotation rows:
# what fraction of genes with meaningful counts are mapped?
# ------------------------------------------------------------

total_counts_per_gene = (
    count_matrix.sum(
        axis=1
    )
)

expressed_mask = (
    total_counts_per_gene > 0
)

expressed_ids = (
    count_matrix.index[
        expressed_mask
    ]
    .astype(str)
    .str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
)

expressed_annotation = (
    gene_annotation.reindex(
        expressed_ids
    )
)

expressed_mapped = (
    expressed_annotation[
        "symbol"
    ]
    .notna()
    .sum()
)

print("\nExpressed genes (>0 total counts):",
      f"{len(expressed_ids):,}")

print(
    "Expressed genes with symbol:",
    f"{expressed_mapped:,} "
    f"({100*expressed_mapped/len(expressed_ids):.2f}%)"
)


# ------------------------------------------------------------
# 7. Load the LOCKED discovery signatures
# ------------------------------------------------------------

locked_json = (
    ROOT /
    "signatures" /
    "GSE328966_LOCKED_discovery_signatures.json"
)

with open(
    locked_json,
    "r"
) as f:

    locked_signatures = json.load(
        f
    )

print("\n" + "=" * 100)
print("LOCKED SIGNATURE COVERAGE IN GSE313176")
print("=" * 100)


# ------------------------------------------------------------
# 8. Build symbol → Ensembl lookup
#
# Preserve possible one-to-many mappings.
# ------------------------------------------------------------

symbol_to_ensembl = {}

for eid, row in gene_annotation.iterrows():

    symbol = row["symbol"]

    if pd.isna(symbol):
        continue

    symbol = str(symbol)

    if symbol not in symbol_to_ensembl:
        symbol_to_ensembl[
            symbol
        ] = []

    symbol_to_ensembl[
        symbol
    ].append(
        eid
    )


# ------------------------------------------------------------
# 9. Signature coverage audit
#
# A signature gene counts as measurable only if:
# - it maps to >=1 Ensembl ID
# - at least one mapped ID has >0 total counts
# ------------------------------------------------------------

coverage_rows = []

signature_gene_rows = []

for signature_name, genes in locked_signatures.items():

    genes = list(
        genes
    )

    measurable = []
    absent = []

    for gene in genes:

        mapped_ids = symbol_to_ensembl.get(
            gene,
            []
        )

        expressed_mapped_ids = [
            eid
            for eid in mapped_ids
            if (
                eid in total_counts_per_gene.index
                and total_counts_per_gene.loc[
                    eid
                ] > 0
            )
        ]

        is_measurable = (
            len(
                expressed_mapped_ids
            ) > 0
        )

        if is_measurable:
            measurable.append(
                gene
            )
        else:
            absent.append(
                gene
            )

        signature_gene_rows.append({
            "signature": signature_name,
            "gene_symbol": gene,
            "mapped_ensembl_ids": ";".join(
                mapped_ids
            ),
            "expressed_ensembl_ids": ";".join(
                expressed_mapped_ids
            ),
            "measurable": is_measurable
        })

    coverage_rows.append({
        "signature": signature_name,
        "locked_n_genes": len(
            genes
        ),
        "measurable_n": len(
            measurable
        ),
        "coverage_pct": (
            100 * len(measurable) / len(genes)
            if len(genes) > 0
            else np.nan
        ),
        "missing_n": len(
            absent
        ),
        "missing_genes": ";".join(
            absent
        )
    })


signature_coverage = pd.DataFrame(
    coverage_rows
)

signature_gene_map = pd.DataFrame(
    signature_gene_rows
)

display(
    signature_coverage.round(2)
)


# ------------------------------------------------------------
# 10. Explicitly print PRIMARY/SECONDARY signatures
# ------------------------------------------------------------

important_signatures = [
    "early_IFN_alpha_beta",
    "early_retinoid_metabolism",
    "early_fat_soluble_vitamin_metabolism",
    "early_ECM_GAG_exploratory"
]

for sig in important_signatures:

    if sig not in locked_signatures:
        continue

    sub = signature_gene_map[
        signature_gene_map[
            "signature"
        ] == sig
    ]

    print("\n" + "-" * 100)
    print(sig)
    print("-" * 100)

    display(
        sub[
            [
                "gene_symbol",
                "mapped_ensembl_ids",
                "expressed_ensembl_ids",
                "measurable"
            ]
        ]
    )


# ------------------------------------------------------------
# 11. Check specific genes we already care about
# ------------------------------------------------------------

focus_genes = [
    "Ccl21a",
    "Rsad2",
    "Bst2",
    "Ccl19",
    "Gbp9",
    "Phgdh",
    "Ntn3",
    "Vegfc",
    "Aqp5",
    "Ncan",
    "Chsy3",
    "Kcnj10",
    "Glul",
    "Slc1a3",
    "Mt1",
    "Zfp36",
    "Slc14a1"
]

focus_rows = []

for gene in focus_genes:

    ids = symbol_to_ensembl.get(
        gene,
        []
    )

    expressed_ids = [
        eid for eid in ids
        if (
            eid in total_counts_per_gene.index
            and total_counts_per_gene.loc[eid] > 0
        )
    ]

    focus_rows.append({
        "gene": gene,
        "mapped_ensembl_ids": ";".join(
            ids
        ),
        "expressed_ensembl_ids": ";".join(
            expressed_ids
        ),
        "measurable": (
            len(expressed_ids) > 0
        ),
        "total_counts": (
            int(
                sum(
                    total_counts_per_gene.loc[eid]
                    for eid in expressed_ids
                )
            )
            if len(expressed_ids) > 0
            else 0
        )
    })


focus_mapping = pd.DataFrame(
    focus_rows
)

print("\n" + "=" * 100)
print("FOCUS GENE MAPPING")
print("=" * 100)

display(
    focus_mapping
)


# ------------------------------------------------------------
# 12. Save
# ------------------------------------------------------------

annotation_file = (
    OUTDIR /
    "GSE313176_Ensembl_to_symbol_annotation.csv"
)

coverage_file = (
    OUTDIR /
    "GSE313176_LOCKED_signature_coverage.csv"
)

signature_map_file = (
    OUTDIR /
    "GSE313176_LOCKED_signature_gene_mapping.csv"
)

focus_file = (
    OUTDIR /
    "GSE313176_focus_gene_mapping.csv"
)

gene_annotation.to_csv(
    annotation_file
)

signature_coverage.to_csv(
    coverage_file,
    index=False
)

signature_gene_map.to_csv(
    signature_map_file,
    index=False
)

focus_mapping.to_csv(
    focus_file,
    index=False
)

print("\nSaved:")
print(annotation_file)
print(coverage_file)
print(signature_map_file)
print(focus_file)

print(
    "\nSTOP HERE. "
    "Do not run DE or signature scoring until "
    "mapping/coverage is assessed."
)

In [ ]:
# ============================================================
# GSE313176 — CELL 5
# REPLICATED RETINA BULK DIFFERENTIAL EXPRESSION
#
# Three independent comparisons:
#   1 month: STZ vs Sham
#   3 months: STZ vs Sham
#   6 months: STZ vs Sham
#
# PyDESeq2 negative-binomial model.
#
# IMPORTANT:
# - Ensembl IDs remain the inferential unit.
# - Gene symbols are annotation only.
# - Locked signatures are NOT statistically tested yet.
# - This cell only establishes gene-level effects.
# ============================================================

import sys
import subprocess
import io
import contextlib
import numpy as np
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. Install/import PyDESeq2
# ------------------------------------------------------------

try:
    import pydeseq2
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "pydeseq2"
        ]
    )
    import pydeseq2

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

print("=" * 100)
print("GSE313176 — REPLICATED RETINA BULK DE")
print("=" * 100)

print(
    "\nPyDESeq2 version:",
    getattr(
        pydeseq2,
        "__version__",
        "unknown"
    )
)


# ------------------------------------------------------------
# 2. Sample metadata
# ------------------------------------------------------------

de_meta = (
    download_manifest[
        [
            "gsm",
            "timepoint",
            "condition"
        ]
    ]
    .copy()
    .set_index("gsm")
)

# Explicit categorical order
de_meta["condition"] = pd.Categorical(
    de_meta["condition"],
    categories=[
        "Sham",
        "STZ"
    ],
    ordered=True
)

de_meta["timepoint"] = pd.Categorical(
    de_meta["timepoint"],
    categories=[
        "1mo",
        "3mo",
        "6mo"
    ],
    ordered=True
)


# ------------------------------------------------------------
# 3. Fit one DESeq2 model per timepoint
#
# Prefilter:
# keep genes with >=10 TOTAL counts across the six samples.
#
# This is deliberately permissive.
# DESeq2 independent filtering remains enabled downstream.
# ------------------------------------------------------------

all_de_results = []

de_summary_rows = []

dds_objects = {}

for tp in [
    "1mo",
    "3mo",
    "6mo"
]:

    print(
        "\n" + "=" * 100
    )
    print(
        f"FITTING {tp}: STZ vs SHAM"
    )
    print("=" * 100)

    # ---------------- samples ----------------

    sample_ids = (
        de_meta.index[
            de_meta["timepoint"] == tp
        ]
        .tolist()
    )

    sub_meta = (
        de_meta.loc[
            sample_ids,
            [
                "condition"
            ]
        ]
        .copy()
    )

    print(
        "Samples:",
        len(sample_ids)
    )

    print(
        sub_meta["condition"]
        .value_counts()
        .sort_index()
    )

    # ---------------- counts ----------------
    # PyDESeq2 wants:
    # samples × genes

    sub_counts = (
        count_matrix[
            sample_ids
        ]
        .T
        .copy()
    )

    # permissive low-count filter
    gene_keep = (
        sub_counts.sum(
            axis=0
        ) >= 10
    )

    sub_counts_fit = (
        sub_counts.loc[
            :,
            gene_keep
        ]
        .astype(int)
    )

    print(
        "Genes before filtering:",
        sub_counts.shape[1]
    )

    print(
        "Genes retained (total count >=10):",
        sub_counts_fit.shape[1]
    )

    # ---------------- fit ----------------

    dds = DeseqDataSet(
        counts=sub_counts_fit,
        metadata=sub_meta,
        design="~condition",
        refit_cooks=True,
        n_cpus=4,
        quiet=True
    )

    dds.deseq2()

    stats = DeseqStats(
        dds,
        contrast=[
            "condition",
            "STZ",
            "Sham"
        ],
        alpha=0.05,
        cooks_filter=True,
        independent_filter=True,
        n_cpus=4,
        quiet=True
    )

    # summary() performs/finishes the statistical workflow
    # but suppress its huge console printout.
    with contextlib.redirect_stdout(
        io.StringIO()
    ):
        stats.summary()

    result = (
        stats.results_df
        .copy()
    )

    result.index = result.index.astype(
        str
    )

    result.index.name = (
        "ensembl_gene_id"
    )

    result["timepoint"] = tp

    # ---------------- annotation ----------------

    annotation_sub = (
        gene_annotation
        .reindex(
            result.index
        )
    )

    result["symbol"] = (
        annotation_sub[
            "symbol"
        ]
        .values
    )

    result["gene_name"] = (
        annotation_sub[
            "gene_name"
        ]
        .values
    )

    # ---------------- flags ----------------

    result["FDR_lt_0.05"] = (
        result["padj"] < 0.05
    )

    result["FDR_lt_0.10"] = (
        result["padj"] < 0.10
    )

    result["abs_LFC_ge_0.5"] = (
        result[
            "log2FoldChange"
        ].abs() >= 0.5
    )

    all_de_results.append(
        result.reset_index()
    )

    # ---------------- summary ----------------

    valid_padj = (
        result["padj"]
        .notna()
    )

    sig05 = (
        valid_padj &
        (
            result["padj"]
            < 0.05
        )
    )

    sig10 = (
        valid_padj &
        (
            result["padj"]
            < 0.10
        )
    )

    sig05_fc = (
        sig05 &
        (
            result[
                "log2FoldChange"
            ].abs() >= 0.5
        )
    )

    de_summary_rows.append({
        "timepoint": tp,
        "n_samples": len(
            sample_ids
        ),
        "genes_fitted": len(
            result
        ),
        "genes_with_padj": int(
            valid_padj.sum()
        ),
        "FDR_lt_0.05": int(
            sig05.sum()
        ),
        "FDR_lt_0.10": int(
            sig10.sum()
        ),
        "FDR_lt_0.05_and_absLFC_ge0.5": int(
            sig05_fc.sum()
        ),
        "up_FDR05": int(
            (
                sig05 &
                (
                    result[
                        "log2FoldChange"
                    ] > 0
                )
            ).sum()
        ),
        "down_FDR05": int(
            (
                sig05 &
                (
                    result[
                        "log2FoldChange"
                    ] < 0
                )
            ).sum()
        )
    })

    # keep object available if needed later
    dds_objects[tp] = dds


# ------------------------------------------------------------
# 4. Combine all three DE result tables
# ------------------------------------------------------------

de_results = pd.concat(
    all_de_results,
    ignore_index=True
)

de_summary = pd.DataFrame(
    de_summary_rows
)


print(
    "\n" + "=" * 100
)
print(
    "DIFFERENTIAL EXPRESSION SUMMARY"
)
print("=" * 100)

display(
    de_summary
)


# ------------------------------------------------------------
# 5. Top genes at each timepoint
# ------------------------------------------------------------

for tp in [
    "1mo",
    "3mo",
    "6mo"
]:

    sub = (
        de_results[
            de_results[
                "timepoint"
            ] == tp
        ]
        .copy()
    )

    # require non-null adjusted p-value
    sub = sub[
        sub["padj"].notna()
    ]

    print(
        "\n" + "=" * 100
    )

    print(
        f"{tp} — TOP STZ-UP GENES"
    )

    print("=" * 100)

    display(
        sub.sort_values(
            [
                "padj",
                "log2FoldChange"
            ],
            ascending=[
                True,
                False
            ]
        )[
            [
                "symbol",
                "baseMean",
                "log2FoldChange",
                "lfcSE",
                "stat",
                "pvalue",
                "padj"
            ]
        ]
        .head(15)
        .round(5)
    )

    print(
        f"\n{tp} — TOP STZ-DOWN GENES"
    )

    display(
        sub.sort_values(
            [
                "padj",
                "log2FoldChange"
            ],
            ascending=[
                True,
                True
            ]
        )[
            [
                "symbol",
                "baseMean",
                "log2FoldChange",
                "lfcSE",
                "stat",
                "pvalue",
                "padj"
            ]
        ]
        .head(15)
        .round(5)
    )


# ------------------------------------------------------------
# 6. Build locked/focus gene audit
#
# This is gene-level inspection ONLY.
# No signature-level statistical claim yet.
# ------------------------------------------------------------

important_signature_names = [
    "early_IFN_alpha_beta",
    "early_retinoid_metabolism",
    "early_ECM_GAG_exploratory"
]

important_genes = set()

for sig in important_signature_names:

    important_genes.update(
        locked_signatures.get(
            sig,
            []
        )
    )

# Add previously discussed individual genes
important_genes.update(
    [
        "Ccl21a",
        "Rsad2",
        "Bst2",
        "Ccl19",
        "Gbp9",
        "Phgdh",
        "Ntn3",
        "Vegfc",
        "Aqp5",
        "Ncan",
        "Chsy3",
        "Kcnj10",
        "Glul",
        "Slc1a3",
        "Mt1",
        "Zfp36",
        "Slc14a1"
    ]
)


focus_de_rows = []

for gene in sorted(
    important_genes
):

    mapped_ids = (
        symbol_to_ensembl
        .get(
            gene,
            []
        )
    )

    for tp in [
        "1mo",
        "3mo",
        "6mo"
    ]:

        found = False

        for eid in mapped_ids:

            hit = de_results[
                (
                    de_results[
                        "ensembl_gene_id"
                    ] == eid
                )
                &
                (
                    de_results[
                        "timepoint"
                    ] == tp
                )
            ]

            if len(hit) == 0:
                continue

            found = True

            r = hit.iloc[0]

            focus_de_rows.append({
                "gene": gene,
                "ensembl_gene_id": eid,
                "timepoint": tp,

                "baseMean": r[
                    "baseMean"
                ],

                "log2FC_STZ_vs_Sham": r[
                    "log2FoldChange"
                ],

                "lfcSE": r[
                    "lfcSE"
                ],

                "wald_stat": r[
                    "stat"
                ],

                "pvalue": r[
                    "pvalue"
                ],

                "padj": r[
                    "padj"
                ],

                "status": (
                    "tested"
                    if pd.notna(
                        r["pvalue"]
                    )
                    else "not_testable"
                )
            })

        if not found:

            focus_de_rows.append({
                "gene": gene,
                "ensembl_gene_id": (
                    ";".join(
                        mapped_ids
                    )
                ),
                "timepoint": tp,
                "baseMean": np.nan,
                "log2FC_STZ_vs_Sham": np.nan,
                "lfcSE": np.nan,
                "wald_stat": np.nan,
                "pvalue": np.nan,
                "padj": np.nan,
                "status": (
                    "filtered_or_absent"
                )
            })


focus_de = pd.DataFrame(
    focus_de_rows
)


# ------------------------------------------------------------
# 7. Print primary-signature/focus trajectories
# ------------------------------------------------------------

print(
    "\n" + "=" * 100
)
print(
    "LOCKED / FOCUS GENES — STZ vs SHAM ACROSS TIME"
)
print("=" * 100)

focus_pivot = (
    focus_de
    .pivot_table(
        index="gene",
        columns="timepoint",
        values="log2FC_STZ_vs_Sham",
        aggfunc="first"
    )
    .reindex(
        columns=[
            "1mo",
            "3mo",
            "6mo"
        ]
    )
)

focus_pivot.columns = [
    "log2FC_1mo",
    "log2FC_3mo",
    "log2FC_6mo"
]

display(
    focus_pivot.round(4)
)


print(
    "\n" + "=" * 100
)
print(
    "DETAILED LOCKED / FOCUS GENE RESULTS"
)
print("=" * 100)

display(
    focus_de[
        [
            "gene",
            "timepoint",
            "baseMean",
            "log2FC_STZ_vs_Sham",
            "lfcSE",
            "pvalue",
            "padj",
            "status"
        ]
    ]
    .sort_values(
        [
            "gene",
            "timepoint"
        ]
    )
    .round(5)
)


# ------------------------------------------------------------
# 8. Save
# ------------------------------------------------------------

combined_file = (
    OUTDIR /
    "GSE313176_retina_bulk_DESeq2_all_timepoints.csv.gz"
)

summary_file = (
    OUTDIR /
    "GSE313176_retina_bulk_DESeq2_summary.csv"
)

focus_file = (
    OUTDIR /
    "GSE313176_retina_bulk_LOCKED_focus_gene_DE.csv"
)

de_results.to_csv(
    combined_file,
    index=False,
    compression="gzip"
)

de_summary.to_csv(
    summary_file,
    index=False
)

focus_de.to_csv(
    focus_file,
    index=False
)

print("\nSaved:")
print(combined_file)
print(summary_file)
print(focus_file)

print(
    "\nSTOP HERE. "
    "These are gene-level replicated effects only. "
    "Do not yet call a locked signature validated."
)

In [ ]:
# ============================================================
# GSE313176 — CELL 6
# PRESPECIFIED LOCKED-SIGNATURE VALIDATION
#
# Tests:
#   PRIMARY
#     - early_IFN_alpha_beta
#     - early_retinoid_metabolism
#
#   SECONDARY
#     - early_ECM_GAG_exploratory
#
#   DESCRIPTIVE
#     - early_up_transient_all
#
# Method:
# - DESeq2 size-factor-normalized counts
# - log2(normalized count + 1)
# - gene-wise z scoring within each 6-sample timepoint
# - equal-weight mean module score
# - exact 3-vs-3 permutation test (20 possible allocations)
#
# Expected direction is UP because these signatures were
# frozen from early-up genes before seeing GSE313176.
# ============================================================

import numpy as np
import pandas as pd
from itertools import combinations


# ------------------------------------------------------------
# 1. Signatures to test
#
# Fat-soluble-vitamin signature is deliberately NOT tested
# separately because it is identical to retinoid metabolism.
# ------------------------------------------------------------

validation_signatures = {
    "early_IFN_alpha_beta":
        locked_signatures["early_IFN_alpha_beta"],

    "early_retinoid_metabolism":
        locked_signatures["early_retinoid_metabolism"],

    "early_ECM_GAG_exploratory":
        locked_signatures["early_ECM_GAG_exploratory"],

    "early_up_transient_all":
        locked_signatures["early_up_transient_all"]
}

evidence_family = {
    "early_IFN_alpha_beta":
        "PRIMARY",

    "early_retinoid_metabolism":
        "PRIMARY",

    "early_ECM_GAG_exploratory":
        "SECONDARY",

    "early_up_transient_all":
        "DESCRIPTIVE"
}


# ------------------------------------------------------------
# 2. BH helper
# ------------------------------------------------------------

def bh_adjust(pvalues):

    p = np.asarray(
        pvalues,
        dtype=float
    )

    m = len(p)

    order = np.argsort(p)
    ranked = p[order]

    q = (
        ranked *
        m /
        np.arange(
            1,
            m + 1
        )
    )

    q = np.minimum.accumulate(
        q[::-1]
    )[::-1]

    q = np.clip(
        q,
        0,
        1
    )

    out = np.empty(m)
    out[order] = q

    return out


# ------------------------------------------------------------
# 3. Hedges g
# ------------------------------------------------------------

def hedges_g(
    stz,
    sham
):

    stz = np.asarray(
        stz,
        dtype=float
    )

    sham = np.asarray(
        sham,
        dtype=float
    )

    n1 = len(stz)
    n0 = len(sham)

    pooled_var = (
        (
            (n1 - 1) *
            np.var(
                stz,
                ddof=1
            )
        )
        +
        (
            (n0 - 1) *
            np.var(
                sham,
                ddof=1
            )
        )
    ) / (
        n1 + n0 - 2
    )

    if pooled_var <= 0:
        return np.nan

    d = (
        np.mean(stz)
        -
        np.mean(sham)
    ) / np.sqrt(
        pooled_var
    )

    df = (
        n1 + n0 - 2
    )

    correction = (
        1 -
        3 /
        (
            4 * df - 1
        )
    )

    return (
        correction *
        d
    )


# ------------------------------------------------------------
# 4. Exact permutation test
#
# With 3 STZ / 3 Sham:
# choose 3 of 6 = 20 exact assignments.
#
# Primary directional test:
#   STZ score > Sham score
# ------------------------------------------------------------

def exact_3v3_test(
    scores,
    observed_stz_mask
):

    scores = np.asarray(
        scores,
        dtype=float
    )

    observed_stz_mask = np.asarray(
        observed_stz_mask,
        dtype=bool
    )

    observed_delta = (
        scores[
            observed_stz_mask
        ].mean()
        -
        scores[
            ~observed_stz_mask
        ].mean()
    )

    n = len(scores)

    assert n == 6
    assert observed_stz_mask.sum() == 3

    perm_deltas = []

    for selected in combinations(
        range(n),
        3
    ):

        mask = np.zeros(
            n,
            dtype=bool
        )

        mask[
            list(selected)
        ] = True

        delta = (
            scores[
                mask
            ].mean()
            -
            scores[
                ~mask
            ].mean()
        )

        perm_deltas.append(
            delta
        )

    perm_deltas = np.asarray(
        perm_deltas
    )

    # exact one-sided P:
    # prespecified direction = STZ > Sham
    p_one = np.mean(
        perm_deltas >= (
            observed_delta
            - 1e-12
        )
    )

    # also report two-sided exact P
    p_two = np.mean(
        np.abs(
            perm_deltas
        ) >= (
            abs(
                observed_delta
            )
            - 1e-12
        )
    )

    return (
        observed_delta,
        p_one,
        p_two,
        perm_deltas
    )


# ------------------------------------------------------------
# 5. Convert signature symbols → Ensembl IDs
# ------------------------------------------------------------

def signature_ensembl_ids(
    genes,
    available_ids
):

    rows = []

    available_ids = set(
        available_ids
    )

    for gene in genes:

        mapped = (
            symbol_to_ensembl
            .get(
                gene,
                []
            )
        )

        usable = [
            eid
            for eid in mapped
            if eid in available_ids
        ]

        for eid in usable:

            rows.append(
                (
                    gene,
                    eid
                )
            )

    return rows


# ------------------------------------------------------------
# 6. Score each signature separately at each timepoint
# ------------------------------------------------------------

result_rows = []
sample_score_rows = []
gene_direction_rows = []

for tp in [
    "1mo",
    "3mo",
    "6mo"
]:

    dds = dds_objects[
        tp
    ]

    # --------------------------------------------------------
    # DESeq2 normalized counts from the fitted object
    # --------------------------------------------------------

    norm = pd.DataFrame(
        np.asarray(
            dds.layers[
                "normed_counts"
            ]
        ),
        index=dds.obs_names.astype(
            str
        ),
        columns=dds.var_names.astype(
            str
        )
    )

    sample_ids = norm.index.tolist()

    conditions = (
        de_meta
        .loc[
            sample_ids,
            "condition"
        ]
        .astype(str)
    )

    observed_stz = (
        conditions.values
        == "STZ"
    )

    # log normalized expression
    lognorm = np.log2(
        norm + 1
    )

    # --------------------------------------------------------
    # each locked signature
    # --------------------------------------------------------

    for signature_name, locked_genes in (
        validation_signatures.items()
    ):

        mapped = signature_ensembl_ids(
            locked_genes,
            available_ids=lognorm.columns
        )

        # Ensembl IDs available after DE count filtering
        usable_eids = [
            eid
            for gene, eid in mapped
        ]

        usable_symbols = [
            gene
            for gene, eid in mapped
        ]

        # remove accidental duplicate Ensembl IDs
        unique_pairs = []

        seen_eids = set()

        for gene, eid in mapped:

            if eid in seen_eids:
                continue

            seen_eids.add(
                eid
            )

            unique_pairs.append(
                (
                    gene,
                    eid
                )
            )

        usable_symbols = [
            x[0]
            for x in unique_pairs
        ]

        usable_eids = [
            x[1]
            for x in unique_pairs
        ]

        # ----------------------------------------------------
        # Remove genes with zero variance across 6 samples
        # ----------------------------------------------------

        retained_symbols = []
        retained_eids = []

        for gene, eid in zip(
            usable_symbols,
            usable_eids
        ):

            values = (
                lognorm[
                    eid
                ]
                .to_numpy()
            )

            if np.std(
                values,
                ddof=1
            ) > 0:

                retained_symbols.append(
                    gene
                )

                retained_eids.append(
                    eid
                )

        if len(
            retained_eids
        ) < 2:

            print(
                f"WARNING: {tp} | "
                f"{signature_name} has "
                f"<2 usable genes."
            )

            continue

        # ----------------------------------------------------
        # Gene-wise z score across all six samples.
        #
        # This transformation is label-independent.
        # Each gene gets equal weight.
        # ----------------------------------------------------

        X = (
            lognorm[
                retained_eids
            ]
            .copy()
        )

        Xz = (
            X
            -
            X.mean(
                axis=0
            )
        ) / X.std(
            axis=0,
            ddof=1
        )

        module_score = (
            Xz.mean(
                axis=1
            )
        )

        # ----------------------------------------------------
        # Exact permutation inference
        # ----------------------------------------------------

        (
            delta,
            p_one,
            p_two,
            perm_deltas
        ) = exact_3v3_test(
            scores=module_score.values,
            observed_stz_mask=observed_stz
        )

        stz_scores = (
            module_score.values[
                observed_stz
            ]
        )

        sham_scores = (
            module_score.values[
                ~observed_stz
            ]
        )

        effect_g = hedges_g(
            stz_scores,
            sham_scores
        )

        # ----------------------------------------------------
        # Gene-level direction concordance
        # ----------------------------------------------------

        gene_lfcs = []

        for gene, eid in zip(
            retained_symbols,
            retained_eids
        ):

            hit = de_results[
                (
                    de_results[
                        "ensembl_gene_id"
                    ] == eid
                )
                &
                (
                    de_results[
                        "timepoint"
                    ] == tp
                )
            ]

            if len(hit) == 0:
                continue

            lfc = hit.iloc[0][
                "log2FoldChange"
            ]

            if pd.notna(
                lfc
            ):

                gene_lfcs.append(
                    (
                        gene,
                        float(
                            lfc
                        )
                    )
                )

        lfc_values = np.asarray(
            [
                x[1]
                for x in gene_lfcs
            ],
            dtype=float
        )

        n_positive = int(
            np.sum(
                lfc_values > 0
            )
        )

        n_negative = int(
            np.sum(
                lfc_values < 0
            )
        )

        # ----------------------------------------------------
        # summary
        # ----------------------------------------------------

        result_rows.append({
            "timepoint": tp,
            "signature": signature_name,
            "evidence_family":
                evidence_family[
                    signature_name
                ],

            "locked_n_genes":
                len(
                    locked_genes
                ),

            "scored_n_genes":
                len(
                    retained_eids
                ),

            "scored_genes":
                ";".join(
                    retained_symbols
                ),

            "mean_score_STZ":
                np.mean(
                    stz_scores
                ),

            "mean_score_Sham":
                np.mean(
                    sham_scores
                ),

            "delta_STZ_minus_Sham":
                delta,

            "hedges_g":
                effect_g,

            "exact_p_one_sided_up":
                p_one,

            "exact_p_two_sided":
                p_two,

            "n_gene_LFC_positive":
                n_positive,

            "n_gene_LFC_negative":
                n_negative,

            "fraction_gene_LFC_positive":
                (
                    n_positive /
                    len(
                        lfc_values
                    )
                    if len(
                        lfc_values
                    ) > 0
                    else np.nan
                ),

            "median_gene_log2FC":
                (
                    np.median(
                        lfc_values
                    )
                    if len(
                        lfc_values
                    ) > 0
                    else np.nan
                ),

            "mean_gene_log2FC":
                (
                    np.mean(
                        lfc_values
                    )
                    if len(
                        lfc_values
                    ) > 0
                    else np.nan
                )
        })

        # ----------------------------------------------------
        # sample-level output
        # ----------------------------------------------------

        for sample in sample_ids:

            sample_score_rows.append({
                "timepoint": tp,
                "signature": signature_name,
                "sample": sample,
                "condition":
                    str(
                        de_meta.loc[
                            sample,
                            "condition"
                        ]
                    ),
                "module_score":
                    module_score.loc[
                        sample
                    ]
            })

        # ----------------------------------------------------
        # gene-level output
        # ----------------------------------------------------

        for gene, lfc in gene_lfcs:

            gene_direction_rows.append({
                "timepoint": tp,
                "signature": signature_name,
                "gene": gene,
                "log2FC_STZ_vs_Sham": lfc
            })


# ------------------------------------------------------------
# 7. Results table
# ------------------------------------------------------------

signature_validation = pd.DataFrame(
    result_rows
)

sample_signature_scores = pd.DataFrame(
    sample_score_rows
)

signature_gene_directions = pd.DataFrame(
    gene_direction_rows
)


# ------------------------------------------------------------
# 8. Multiple-testing correction WITHIN evidence family
#
# PRIMARY family:
# 2 independent biological modules × 3 timepoints = 6 tests
#
# Retinoid and fat-soluble-vitamin are not double-counted.
# ------------------------------------------------------------

signature_validation[
    "BH_q_one_sided_within_family"
] = np.nan

for family in [
    "PRIMARY",
    "SECONDARY",
    "DESCRIPTIVE"
]:

    mask = (
        signature_validation[
            "evidence_family"
        ] == family
    )

    if mask.sum() == 0:
        continue

    signature_validation.loc[
        mask,
        "BH_q_one_sided_within_family"
    ] = bh_adjust(
        signature_validation.loc[
            mask,
            "exact_p_one_sided_up"
        ].values
    )


# ------------------------------------------------------------
# 9. Print core validation table
# ------------------------------------------------------------

print("=" * 110)
print("LOCKED SIGNATURE VALIDATION — EXACT SAMPLE-LEVEL TEST")
print("=" * 110)

display(
    signature_validation[
        [
            "timepoint",
            "signature",
            "evidence_family",
            "locked_n_genes",
            "scored_n_genes",
            "delta_STZ_minus_Sham",
            "hedges_g",
            "exact_p_one_sided_up",
            "BH_q_one_sided_within_family",
            "fraction_gene_LFC_positive",
            "median_gene_log2FC"
        ]
    ]
    .sort_values(
        [
            "evidence_family",
            "signature",
            "timepoint"
        ]
    )
    .round(4)
)


# ------------------------------------------------------------
# 10. Print PRIMARY modules clearly
# ------------------------------------------------------------

for signature_name in [
    "early_IFN_alpha_beta",
    "early_retinoid_metabolism"
]:

    print(
        "\n" + "=" * 110
    )

    print(
        signature_name.upper()
    )

    print("=" * 110)

    sub = (
        signature_validation[
            signature_validation[
                "signature"
            ] == signature_name
        ]
        .copy()
    )

    display(
        sub[
            [
                "timepoint",
                "scored_n_genes",
                "delta_STZ_minus_Sham",
                "hedges_g",
                "exact_p_one_sided_up",
                "BH_q_one_sided_within_family",
                "n_gene_LFC_positive",
                "n_gene_LFC_negative",
                "median_gene_log2FC",
                "mean_gene_log2FC"
            ]
        ]
        .round(4)
    )


# ------------------------------------------------------------
# 11. Print sample scores
# ------------------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "SAMPLE-LEVEL MODULE SCORES"
)

print("=" * 110)

display(
    sample_signature_scores
    .pivot_table(
        index=[
            "timepoint",
            "sample",
            "condition"
        ],
        columns="signature",
        values="module_score"
    )
    .round(4)
)


# ------------------------------------------------------------
# 12. Save
# ------------------------------------------------------------

validation_file = (
    OUTDIR /
    "GSE313176_LOCKED_signature_validation_exact_permutation.csv"
)

score_file = (
    OUTDIR /
    "GSE313176_LOCKED_signature_sample_scores.csv"
)

direction_file = (
    OUTDIR /
    "GSE313176_LOCKED_signature_gene_directions.csv"
)

signature_validation.to_csv(
    validation_file,
    index=False
)

sample_signature_scores.to_csv(
    score_file,
    index=False
)

signature_gene_directions.to_csv(
    direction_file,
    index=False
)

print("\nSaved:")
print(validation_file)
print(score_file)
print(direction_file)

print(
    "\nIMPORTANT:"
    "\nThe smallest possible one-sided exact P with "
    "3-vs-3 samples is 0.05."
    "\nInterpret effect size, sample separation, "
    "gene-direction concordance and exact P together."
)

In [ ]:
# ============================================================
# DR MÜLLER PROJECT — FRESH-COLAB RESUME / BOOTSTRAP CELL
#
# Run this FIRST whenever Colab has been closed/restarted.
#
# Restores the variables needed to continue from saved Drive
# files without rerunning previous analyses.
# ============================================================

from pathlib import Path
import json
import pandas as pd

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)


# ------------------------------------------------------------
# 2. Restore project paths
# ------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/DR_Muller_AdaptiveReactive"
)

OUTDIR = (
    ROOT /
    "04_GSE313176_mouse_longitudinal"
)

SCRNA_DIR = (
    OUTDIR /
    "scRNA_3mo"
)

SCRNA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 3. Restore GSE313176 metadata
# ------------------------------------------------------------

manifest_file = (
    OUTDIR /
    "GSE313176_sample_manifest.csv"
)

assert manifest_file.exists(), (
    f"Missing manifest:\n{manifest_file}"
)

meta313176 = pd.read_csv(
    manifest_file
)


# ------------------------------------------------------------
# 4. Restore LOCKED GSE328966 discovery signatures
# ------------------------------------------------------------

locked_signature_file = (
    ROOT /
    "signatures" /
    "GSE328966_LOCKED_discovery_signatures.json"
)

assert locked_signature_file.exists(), (
    f"Missing locked signature file:\n"
    f"{locked_signature_file}"
)

with open(
    locked_signature_file,
    "r"
) as f:

    locked_signatures = json.load(
        f
    )


# ------------------------------------------------------------
# 5. Restore useful GSE313176 bulk manifests/results paths
#
# We are NOT loading huge matrices unless needed.
# ------------------------------------------------------------

bulk_manifest_file = (
    OUTDIR /
    "GSE313176_retina_bulk_download_manifest.csv"
)

if bulk_manifest_file.exists():

    download_manifest = pd.read_csv(
        bulk_manifest_file
    )

else:

    download_manifest = None


# ------------------------------------------------------------
# 6. Sanity check
# ------------------------------------------------------------

print("=" * 100)
print("PROJECT RESUME CHECK")
print("=" * 100)

print("\nROOT:")
print(ROOT)

print("\nOUTDIR:")
print(OUTDIR)

print("\nSCRNA_DIR:")
print(SCRNA_DIR)

print(
    "\nGSE313176 manifest samples:",
    len(meta313176)
)

print(
    "scRNA libraries:",
    (
        meta313176["assay"]
        == "scRNA"
    ).sum()
)

print(
    "bulk libraries:",
    (
        meta313176["assay"]
        == "bulk_RNA"
    ).sum()
)

print(
    "\nLocked signatures restored:",
    len(locked_signatures)
)

for name, genes in locked_signatures.items():

    print(
        f"  {name}: {len(genes)} genes"
    )

print(
    "\nBulk download manifest restored:",
    download_manifest is not None
)

print("\n" + "=" * 100)
print("RESUME STATUS")
print("=" * 100)

print(
    "READY — essential project variables restored."
)

print(
    "\nNext analysis:"
    "\nGSE313176 3-month scRNA download + raw audit."
)

In [ ]:
# ============================================================
# GSE313176 scRNA — SAVE COMPLETED RAW AUDIT BEFORE RESTART
#
# Avoids pandas pivot/unstack entirely.
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# 1. Print Müller-marker coverage WITHOUT pivot()
# ------------------------------------------------------------

print("=" * 80)
print("MÜLLER MARKER COVERAGE")
print("=" * 80)

print(
    f"{'Gene':<12}"
    f"{'Sham':<10}"
    f"{'STZ':<10}"
)

print("-" * 32)

for gene in muller_markers:

    sham_present = False
    stz_present = False

    for _, r in marker_audit.iterrows():

        if r["gene"] != gene:
            continue

        if r["condition"] == "Sham":
            sham_present = bool(
                r["present"]
            )

        elif r["condition"] == "STZ":
            stz_present = bool(
                r["present"]
            )

    print(
        f"{gene:<12}"
        f"{str(sham_present):<10}"
        f"{str(stz_present):<10}"
    )


# ------------------------------------------------------------
# 2. Save all completed audit tables
# ------------------------------------------------------------

feature_file = (
    SCRNA_DIR /
    "GSE313176_scRNA_feature_audit.csv"
)

raw_qc_file = (
    SCRNA_DIR /
    "GSE313176_scRNA_raw_QC_audit.csv"
)

sig_file = (
    SCRNA_DIR /
    "GSE313176_scRNA_locked_signature_coverage.csv"
)

marker_file = (
    SCRNA_DIR /
    "GSE313176_scRNA_Muller_marker_coverage.csv"
)


feature_audit.to_csv(
    feature_file,
    index=False
)

raw_audit.to_csv(
    raw_qc_file,
    index=False
)

signature_coverage_scrna.to_csv(
    sig_file,
    index=False
)

marker_audit.to_csv(
    marker_file,
    index=False
)


# ------------------------------------------------------------
# 3. Save checkpoint note
# ------------------------------------------------------------

checkpoint = (
    SCRNA_DIR /
    "CHECKPOINT_raw_audit_complete.txt"
)

with open(
    checkpoint,
    "w"
) as f:

    f.write(
        "GSE313176 3mo scRNA raw audit complete.\n"
        "Raw 10x files downloaded for Sham and STZ.\n"
        "No filtering, normalization, clustering, or DE performed.\n"
        "Next step: restart runtime, bootstrap project, verify "
        "pandas/scanpy/anndata versions, then derive QC thresholds.\n"
    )


print("\n" + "=" * 80)
print("SAVED SUCCESSFULLY")
print("=" * 80)

print(feature_file)
print(raw_qc_file)
print(sig_file)
print(marker_file)
print(checkpoint)

print(
    "\nNow restart the Colab session."
)

In [ ]:
# ============================================================
# DR MÜLLER PROJECT — POST-RESTART BOOTSTRAP + ENVIRONMENT CHECK
#
# Purpose:
# - remount Drive
# - restore project variables
# - confirm raw scRNA checkpoint exists
# - inspect pandas / anndata / scanpy versions
# - test the exact operations that previously failed
#
# NO data analysis.
# NO package installation.
# ============================================================

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)


# ------------------------------------------------------------
# 2. Restore project paths
# ------------------------------------------------------------

ROOT = Path(
    "/content/drive/MyDrive/DR_Muller_AdaptiveReactive"
)

OUTDIR = (
    ROOT /
    "04_GSE313176_mouse_longitudinal"
)

SCRNA_DIR = (
    OUTDIR /
    "scRNA_3mo"
)


# ------------------------------------------------------------
# 3. Restore metadata
# ------------------------------------------------------------

meta313176 = pd.read_csv(
    OUTDIR /
    "GSE313176_sample_manifest.csv"
)

with open(
    ROOT /
    "signatures" /
    "GSE328966_LOCKED_discovery_signatures.json",
    "r"
) as f:

    locked_signatures = json.load(f)


# ------------------------------------------------------------
# 4. Restore completed scRNA audit tables
# ------------------------------------------------------------

feature_audit = pd.read_csv(
    SCRNA_DIR /
    "GSE313176_scRNA_feature_audit.csv"
)

raw_audit = pd.read_csv(
    SCRNA_DIR /
    "GSE313176_scRNA_raw_QC_audit.csv"
)

signature_coverage_scrna = pd.read_csv(
    SCRNA_DIR /
    "GSE313176_scRNA_locked_signature_coverage.csv"
)

marker_audit = pd.read_csv(
    SCRNA_DIR /
    "GSE313176_scRNA_Muller_marker_coverage.csv"
)

checkpoint = (
    SCRNA_DIR /
    "CHECKPOINT_raw_audit_complete.txt"
)


# ------------------------------------------------------------
# 5. Environment versions
# ------------------------------------------------------------

print("=" * 100)
print("POST-RESTART ENVIRONMENT CHECK")
print("=" * 100)

print("\nPython:")
print(sys.version)

print("\npandas:")
print(pd.__version__)

print("\nnumpy:")
print(np.__version__)


# ------------------------------------------------------------
# 6. Import anndata + scanpy WITHOUT installing anything
# ------------------------------------------------------------

anndata_ok = False
scanpy_ok = False

try:

    import anndata as ad

    print("\nanndata:")
    print(ad.__version__)

    anndata_ok = True

except Exception as e:

    print("\nANNDATA IMPORT FAILED:")
    print(type(e).__name__, str(e))


try:

    import scanpy as sc

    print("\nscanpy:")
    print(sc.__version__)

    scanpy_ok = True

except Exception as e:

    print("\nSCANPY IMPORT FAILED:")
    print(type(e).__name__, str(e))


# ------------------------------------------------------------
# 7. Test pandas StringDtype operation implicated previously
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("PANDAS COMPATIBILITY TEST")
print("=" * 100)

try:

    test_dtype = pd.StringDtype(
        na_value=np.nan
    )

    print(
        "pd.StringDtype(na_value=...) : PASS"
    )

except Exception as e:

    print(
        "pd.StringDtype(na_value=...) : FAIL"
    )

    print(
        type(e).__name__,
        str(e)
    )


# ------------------------------------------------------------
# 8. Test pandas pivot/unstack
# ------------------------------------------------------------

try:

    tiny = pd.DataFrame({
        "gene": [
            "Rlbp1",
            "Rlbp1",
            "Glul",
            "Glul"
        ],
        "condition": [
            "Sham",
            "STZ",
            "Sham",
            "STZ"
        ],
        "present": [
            True,
            True,
            True,
            True
        ]
    })

    tiny_pivot = tiny.pivot(
        index="gene",
        columns="condition",
        values="present"
    )

    print(
        "pandas pivot/unstack        : PASS"
    )

except Exception as e:

    print(
        "pandas pivot/unstack        : FAIL"
    )

    print(
        type(e).__name__,
        str(e)
    )


# ------------------------------------------------------------
# 9. Test tiny AnnData construction
#
# This directly probes the code path that failed before.
# ------------------------------------------------------------

if anndata_ok:

    try:

        from scipy.sparse import csr_matrix

        X_test = csr_matrix(
            np.array(
                [
                    [1, 0, 3],
                    [0, 2, 1]
                ],
                dtype=np.int32
            )
        )

        tiny_adata = ad.AnnData(
            X=X_test
        )

        tiny_adata.obs_names = [
            "cell1",
            "cell2"
        ]

        tiny_adata.var_names = [
            "GeneA",
            "GeneB",
            "GeneC"
        ]

        print(
            "AnnData construction       : PASS"
        )

    except Exception as e:

        print(
            "AnnData construction       : FAIL"
        )

        print(
            type(e).__name__,
            str(e)
        )


# ------------------------------------------------------------
# 10. Check saved project state
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("PROJECT STATE")
print("=" * 100)

print(
    "Checkpoint exists:",
    checkpoint.exists()
)

print(
    "GSE313176 samples:",
    len(meta313176)
)

print(
    "scRNA libraries:",
    (
        meta313176["assay"]
        == "scRNA"
    ).sum()
)

print(
    "Raw audit rows:",
    len(raw_audit)
)

print(
    "Locked signatures:",
    len(locked_signatures)
)

print(
    "Müller marker audit rows:",
    len(marker_audit)
)


# ------------------------------------------------------------
# 11. Final status
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("STATUS")
print("=" * 100)

if (
    anndata_ok
    and scanpy_ok
):

    print(
        "Imports succeeded."
    )

else:

    print(
        "Environment still requires repair."
    )

print(
    "\nSTOP HERE."
)

print(
    "Do not run filtering/clustering yet."
)

In [ ]:
# ============================================================
# GSE313176 — scRNA CELL 2
# QC DISTRIBUTIONS + THRESHOLD SENSITIVITY
#
# PURPOSE:
# - reload both raw 10x libraries with clean Scanpy environment
# - save RAW AnnData checkpoints
# - quantify QC distributions
# - derive robust candidate bounds
# - test sensitivity to mitochondrial cutoffs
#
# IMPORTANT:
# NO CELLS ARE FILTERED IN THIS CELL.
# ============================================================

import numpy as np
import pandas as pd
import scanpy as sc

from pathlib import Path
from scipy.stats import median_abs_deviation


# ------------------------------------------------------------
# 1. Restore scRNA metadata and sample directories
# ------------------------------------------------------------

scrna_meta = (
    meta313176[
        meta313176["assay"] == "scRNA"
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(scrna_meta) == 2


sample_dirs = {}

for _, row in scrna_meta.iterrows():

    gsm = row["gsm"]
    condition = row["condition"]

    sample_dir = (
        SCRNA_DIR /
        f"{gsm}_{condition}"
    )

    assert sample_dir.exists()

    sample_dirs[gsm] = sample_dir


# ------------------------------------------------------------
# 2. Read raw 10x matrices with Scanpy
# ------------------------------------------------------------

adatas_raw = {}

for _, row in scrna_meta.iterrows():

    gsm = row["gsm"]
    condition = row["condition"]

    print(
        f"Reading {gsm} | {condition} ..."
    )

    ad = sc.read_10x_mtx(
        sample_dirs[gsm],
        var_names="gene_symbols",
        make_unique=True,
        cache=False
    )

    # preserve original barcode
    ad.obs["barcode"] = (
        ad.obs_names.astype(str)
    )

    ad.obs["gsm"] = gsm
    ad.obs["condition"] = condition
    ad.obs["timepoint"] = "3mo"

    # globally unique cell ID
    ad.obs_names = [
        f"{gsm}_{bc}"
        for bc in ad.obs["barcode"]
    ]

    # mitochondrial annotation
    ad.var["mt"] = (
        ad.var_names
        .str.upper()
        .str.startswith("MT-")
    )

    sc.pp.calculate_qc_metrics(
        ad,
        qc_vars=["mt"],
        percent_top=None,
        log1p=False,
        inplace=True
    )

    adatas_raw[gsm] = ad


# ------------------------------------------------------------
# 3. Verify against yesterday's manual audit
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SCANPY ↔ MANUAL AUDIT CONSISTENCY")
print("=" * 100)

consistency_rows = []

for _, row in scrna_meta.iterrows():

    gsm = row["gsm"]
    condition = row["condition"]

    ad = adatas_raw[gsm]

    previous = raw_audit[
        raw_audit["gsm"] == gsm
    ].iloc[0]

    consistency_rows.append({
        "gsm": gsm,
        "condition": condition,

        "scanpy_cells":
            ad.n_obs,

        "manual_cells":
            int(
                previous["n_cells"]
            ),

        "scanpy_genes":
            ad.n_vars,

        "manual_genes":
            int(
                previous["n_genes"]
            ),

        "scanpy_median_UMI":
            ad.obs[
                "total_counts"
            ].median(),

        "manual_median_UMI":
            previous[
                "median_UMI"
            ],

        "scanpy_median_genes":
            ad.obs[
                "n_genes_by_counts"
            ].median(),

        "manual_median_genes":
            previous[
                "median_genes"
            ],

        "scanpy_median_mt":
            ad.obs[
                "pct_counts_mt"
            ].median(),

        "manual_median_mt":
            previous[
                "median_pct_mt"
            ]
    })


consistency = pd.DataFrame(
    consistency_rows
)

display(
    consistency.round(3)
)


# ------------------------------------------------------------
# 4. Save RAW AnnData checkpoints
#
# These are untouched raw count matrices.
# ------------------------------------------------------------

for _, row in scrna_meta.iterrows():

    gsm = row["gsm"]
    condition = row["condition"]

    raw_file = (
        SCRNA_DIR /
        f"{gsm}_{condition}_RAW.h5ad"
    )

    adatas_raw[gsm].write(
        raw_file
    )

    print(
        "Saved:",
        raw_file
    )


# ------------------------------------------------------------
# 5. Detailed QC quantiles
# ------------------------------------------------------------

quantiles = [
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99
]

qc_quantile_rows = []


for _, row in scrna_meta.iterrows():

    gsm = row["gsm"]
    condition = row["condition"]

    obs = adatas_raw[
        gsm
    ].obs

    metrics = {
        "total_counts":
            obs["total_counts"],

        "n_genes_by_counts":
            obs["n_genes_by_counts"],

        "pct_counts_mt":
            obs["pct_counts_mt"]
    }

    for metric_name, values in metrics.items():

        q = values.quantile(
            quantiles
        )

        out = {
            "gsm": gsm,
            "condition": condition,
            "metric": metric_name
        }

        for qq in quantiles:

            out[
                f"q{int(qq*100):02d}"
            ] = q.loc[qq]

        out["max"] = values.max()

        qc_quantile_rows.append(
            out
        )


qc_quantiles = pd.DataFrame(
    qc_quantile_rows
)

print("\n" + "=" * 100)
print("DETAILED QC DISTRIBUTIONS")
print("=" * 100)

display(
    qc_quantiles.round(3)
)


# ------------------------------------------------------------
# 6. Robust MAD-based candidate bounds
#
# We are NOT applying these.
#
# Counts / genes:
#   calculated on log1p scale
#   low  = median - 3 MAD
#   high = median + 4 MAD
#
# mt%:
#   median + 3 MAD on original scale
# ------------------------------------------------------------

def robust_log_bounds(
    values,
    low_nmads=3,
    high_nmads=4
):

    values = np.asarray(
        values,
        dtype=float
    )

    y = np.log1p(
        values
    )

    med = np.median(
        y
    )

    mad = median_abs_deviation(
        y,
        scale=1.0,
        nan_policy="omit"
    )

    low = np.expm1(
        med -
        low_nmads * mad
    )

    high = np.expm1(
        med +
        high_nmads * mad
    )

    return (
        max(
            0,
            low
        ),
        high,
        mad
    )


threshold_rows = []


for _, row in scrna_meta.iterrows():

    gsm = row["gsm"]
    condition = row["condition"]

    obs = adatas_raw[
        gsm
    ].obs

    gene_low, gene_high, gene_mad = (
        robust_log_bounds(
            obs[
                "n_genes_by_counts"
            ]
        )
    )

    umi_low, umi_high, umi_mad = (
        robust_log_bounds(
            obs[
                "total_counts"
            ]
        )
    )

    mt_values = obs[
        "pct_counts_mt"
    ].to_numpy()

    mt_median = np.median(
        mt_values
    )

    mt_mad = median_abs_deviation(
        mt_values,
        scale=1.0,
        nan_policy="omit"
    )

    mt_3mad = (
        mt_median +
        3 * mt_mad
    )

    # counts flagged by each candidate boundary
    n = len(obs)

    threshold_rows.append({
        "gsm": gsm,
        "condition": condition,
        "n_cells": n,

        "genes_low_3MAD":
            gene_low,

        "genes_high_4MAD":
            gene_high,

        "cells_below_gene_low":
            int(
                (
                    obs[
                        "n_genes_by_counts"
                    ] < gene_low
                ).sum()
            ),

        "cells_above_gene_high":
            int(
                (
                    obs[
                        "n_genes_by_counts"
                    ] > gene_high
                ).sum()
            ),

        "UMI_low_3MAD":
            umi_low,

        "UMI_high_4MAD":
            umi_high,

        "cells_below_UMI_low":
            int(
                (
                    obs[
                        "total_counts"
                    ] < umi_low
                ).sum()
            ),

        "cells_above_UMI_high":
            int(
                (
                    obs[
                        "total_counts"
                    ] > umi_high
                ).sum()
            ),

        "mt_median":
            mt_median,

        "mt_MAD":
            mt_mad,

        "mt_high_3MAD":
            mt_3mad,

        "cells_above_mt_3MAD":
            int(
                (
                    obs[
                        "pct_counts_mt"
                    ] > mt_3mad
                ).sum()
            )
    })


candidate_thresholds = pd.DataFrame(
    threshold_rows
)

print("\n" + "=" * 100)
print("ROBUST CANDIDATE QC BOUNDS — NOT YET APPLIED")
print("=" * 100)

display(
    candidate_thresholds.round(2)
)


# ------------------------------------------------------------
# 7. Mitochondrial cutoff sensitivity
#
# Shows exactly how much each global cutoff removes.
# ------------------------------------------------------------

mt_caps = [
    5,
    10,
    15,
    20,
    25,
    30
]

mt_sensitivity_rows = []


for _, row in scrna_meta.iterrows():

    gsm = row["gsm"]
    condition = row["condition"]

    obs = adatas_raw[
        gsm
    ].obs

    n = len(obs)

    for cap in mt_caps:

        kept = (
            obs[
                "pct_counts_mt"
            ] < cap
        )

        mt_sensitivity_rows.append({
            "gsm": gsm,
            "condition": condition,
            "mt_cutoff_pct": cap,

            "cells_total": n,

            "cells_retained":
                int(
                    kept.sum()
                ),

            "cells_removed":
                int(
                    (
                        ~kept
                    ).sum()
                ),

            "retained_pct":
                100 *
                kept.mean(),

            "removed_pct":
                100 *
                (
                    ~kept
                ).mean()
        })


mt_sensitivity = pd.DataFrame(
    mt_sensitivity_rows
)

print("\n" + "=" * 100)
print("MITOCHONDRIAL CUTOFF SENSITIVITY")
print("=" * 100)

display(
    mt_sensitivity.round(2)
)


# ------------------------------------------------------------
# 8. Minimal lower-quality filter sensitivity
#
# Still NOT applying.
#
# Fixed floor:
#   >=200 genes
#   >=500 UMIs
#
# Then vary mt cutoff.
# ------------------------------------------------------------

joint_rows = []


for _, row in scrna_meta.iterrows():

    gsm = row["gsm"]
    condition = row["condition"]

    obs = adatas_raw[
        gsm
    ].obs

    base = (
        (
            obs[
                "n_genes_by_counts"
            ] >= 200
        )
        &
        (
            obs[
                "total_counts"
            ] >= 500
        )
    )

    for cap in [
        10,
        15,
        20,
        25
    ]:

        keep = (
            base
            &
            (
                obs[
                    "pct_counts_mt"
                ] < cap
            )
        )

        joint_rows.append({
            "gsm": gsm,
            "condition": condition,

            "criteria":
                f"genes>=200;UMI>=500;mt<{cap}%",

            "cells_total":
                len(obs),

            "cells_retained":
                int(
                    keep.sum()
                ),

            "cells_removed":
                int(
                    (
                        ~keep
                    ).sum()
                ),

            "retained_pct":
                100 *
                keep.mean()
        })


joint_sensitivity = pd.DataFrame(
    joint_rows
)

print("\n" + "=" * 100)
print("JOINT MINIMAL-QC SENSITIVITY — NOT APPLIED")
print("=" * 100)

display(
    joint_sensitivity.round(2)
)


# ------------------------------------------------------------
# 9. Save audit tables
# ------------------------------------------------------------

qc_quantiles.to_csv(
    SCRNA_DIR /
    "GSE313176_scRNA_QC_quantiles.csv",
    index=False
)

candidate_thresholds.to_csv(
    SCRNA_DIR /
    "GSE313176_scRNA_QC_candidate_MAD_thresholds.csv",
    index=False
)

mt_sensitivity.to_csv(
    SCRNA_DIR /
    "GSE313176_scRNA_QC_mt_sensitivity.csv",
    index=False
)

joint_sensitivity.to_csv(
    SCRNA_DIR /
    "GSE313176_scRNA_QC_joint_sensitivity.csv",
    index=False
)


print("\n" + "=" * 100)
print("STOP")
print("=" * 100)

print(
    "QC distributions calculated."
)

print(
    "NO cells have been filtered."
)

print(
    "Next step: choose defensible QC thresholds "
    "from these distributions, then run doublet detection."
)

In [ ]:
# ============================================================
# GSE313176 — scRNA CELL 3
# APPLY PRIMARY QC + SCRUBLET DOUBLET SCORING
#
# Primary QC:
#   n_genes >= 200
#   total_counts >= 500
#   pct_counts_mt < 20
#
# IMPORTANT:
# - No upper UMI/gene cutoff
# - Scrublet run separately per library
# - Predicted doublets are SCORED but NOT REMOVED yet
# ============================================================

import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path


# ------------------------------------------------------------
# 1. Fixed primary QC definition
# ------------------------------------------------------------

MIN_GENES = 200
MIN_UMI = 500
MAX_MT = 20.0

EXPECTED_DOUBLET_RATE = 0.05
RANDOM_STATE = 42


# ------------------------------------------------------------
# 2. Restore scRNA metadata
# ------------------------------------------------------------

scrna_meta = (
    meta313176[
        meta313176["assay"] == "scRNA"
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(scrna_meta) == 2


qc_summary_rows = []
scrublet_summary_rows = []
scrublet_score_rows = []

adatas_scrublet = {}


# ------------------------------------------------------------
# 3. Process each library independently
# ------------------------------------------------------------

for _, row in scrna_meta.iterrows():

    gsm = row["gsm"]
    condition = row["condition"]

    print("\n" + "=" * 100)
    print(f"{gsm} | {condition}")
    print("=" * 100)

    raw_file = (
        SCRNA_DIR /
        f"{gsm}_{condition}_RAW.h5ad"
    )

    assert raw_file.exists(), (
        f"Missing raw AnnData: {raw_file}"
    )

    ad = sc.read_h5ad(
        raw_file
    )

    n_raw = ad.n_obs


    # --------------------------------------------------------
    # 4. Primary QC mask
    # --------------------------------------------------------

    pass_genes = (
        ad.obs["n_genes_by_counts"]
        >= MIN_GENES
    )

    pass_umi = (
        ad.obs["total_counts"]
        >= MIN_UMI
    )

    pass_mt = (
        ad.obs["pct_counts_mt"]
        < MAX_MT
    )

    qc_pass = (
        pass_genes
        &
        pass_umi
        &
        pass_mt
    )


    # --------------------------------------------------------
    # 5. Record reasons for exclusion
    # --------------------------------------------------------

    ad.obs["qc_pass_genes"] = pass_genes
    ad.obs["qc_pass_umi"] = pass_umi
    ad.obs["qc_pass_mt"] = pass_mt
    ad.obs["qc_pass_primary"] = qc_pass

    qc_summary_rows.append({
        "gsm": gsm,
        "condition": condition,

        "raw_cells":
            n_raw,

        "fail_genes_lt200":
            int(
                (~pass_genes).sum()
            ),

        "fail_umi_lt500":
            int(
                (~pass_umi).sum()
            ),

        "fail_mt_ge20":
            int(
                (~pass_mt).sum()
            ),

        "cells_removed_any_QC":
            int(
                (~qc_pass).sum()
            ),

        "cells_retained_QC":
            int(
                qc_pass.sum()
            ),

        "retained_pct":
            100 *
            qc_pass.mean()
    })


    # --------------------------------------------------------
    # 6. Subset QC-passing cells
    # --------------------------------------------------------

    ad_qc = (
        ad[
            qc_pass
        ]
        .copy()
    )

    print(
        f"Raw cells:       {n_raw:,}"
    )

    print(
        f"QC-pass cells:   {ad_qc.n_obs:,} "
        f"({100*ad_qc.n_obs/n_raw:.2f}%)"
    )


    # --------------------------------------------------------
    # 7. Scrublet
    #
    # Run independently per library.
    #
    # Use software-default-style prior 5%.
    # We will inspect the resulting predicted rate before
    # removing anything.
    # --------------------------------------------------------

    sc.pp.scrublet(
        ad_qc,
        expected_doublet_rate=
            EXPECTED_DOUBLET_RATE,
        sim_doublet_ratio=2.0,
        n_prin_comps=30,
        random_state=RANDOM_STATE,
        copy=False
    )


    # --------------------------------------------------------
    # 8. Extract Scrublet results
    # --------------------------------------------------------

    assert "doublet_score" in ad_qc.obs.columns
    assert "predicted_doublet" in ad_qc.obs.columns

    scores = (
        ad_qc.obs[
            "doublet_score"
        ]
        .astype(float)
    )

    predicted = (
        ad_qc.obs[
            "predicted_doublet"
        ]
        .astype(bool)
    )

    scrublet_uns = (
        ad_qc.uns.get(
            "scrublet",
            {}
        )
    )

    threshold = (
        scrublet_uns.get(
            "threshold",
            np.nan
        )
        if isinstance(
            scrublet_uns,
            dict
        )
        else np.nan
    )


    scrublet_summary_rows.append({
        "gsm": gsm,
        "condition": condition,

        "qc_cells":
            ad_qc.n_obs,

        "expected_doublet_rate":
            EXPECTED_DOUBLET_RATE,

        "scrublet_threshold":
            threshold,

        "predicted_doublets":
            int(
                predicted.sum()
            ),

        "predicted_doublet_pct":
            100 *
            predicted.mean(),

        "score_q50":
            scores.quantile(
                0.50
            ),

        "score_q90":
            scores.quantile(
                0.90
            ),

        "score_q95":
            scores.quantile(
                0.95
            ),

        "score_q99":
            scores.quantile(
                0.99
            ),

        "score_max":
            scores.max()
    })


    # --------------------------------------------------------
    # 9. Save cell-level Scrublet scores
    # --------------------------------------------------------

    for cell in ad_qc.obs_names:

        scrublet_score_rows.append({
            "cell_id":
                cell,

            "gsm":
                gsm,

            "condition":
                condition,

            "total_counts":
                ad_qc.obs.loc[
                    cell,
                    "total_counts"
                ],

            "n_genes_by_counts":
                ad_qc.obs.loc[
                    cell,
                    "n_genes_by_counts"
                ],

            "pct_counts_mt":
                ad_qc.obs.loc[
                    cell,
                    "pct_counts_mt"
                ],

            "doublet_score":
                ad_qc.obs.loc[
                    cell,
                    "doublet_score"
                ],

            "predicted_doublet":
                bool(
                    ad_qc.obs.loc[
                        cell,
                        "predicted_doublet"
                    ]
                )
        })


    # --------------------------------------------------------
    # 10. Save scored AnnData
    #
    # Predicted doublets are still present.
    # --------------------------------------------------------

    scored_file = (
        SCRNA_DIR /
        f"{gsm}_{condition}_QC20_SCRUBLET_SCORED.h5ad"
    )

    ad_qc.write(
        scored_file
    )

    print(
        "Saved scored AnnData:"
    )

    print(
        scored_file
    )

    adatas_scrublet[gsm] = ad_qc


# ------------------------------------------------------------
# 11. Summaries
# ------------------------------------------------------------

qc_summary = pd.DataFrame(
    qc_summary_rows
)

scrublet_summary = pd.DataFrame(
    scrublet_summary_rows
)

scrublet_scores = pd.DataFrame(
    scrublet_score_rows
)


print("\n" + "=" * 100)
print("PRIMARY QC SUMMARY")
print("=" * 100)

display(
    qc_summary.round(2)
)


print("\n" + "=" * 100)
print("SCRUBLET SUMMARY")
print("=" * 100)

display(
    scrublet_summary.round(4)
)


# ------------------------------------------------------------
# 12. Doublet-vs-singlet QC characteristics
# ------------------------------------------------------------

doublet_qc_summary = (
    scrublet_scores
    .groupby(
        [
            "condition",
            "predicted_doublet"
        ]
    )
    .agg(
        n_cells=(
            "cell_id",
            "size"
        ),

        median_UMI=(
            "total_counts",
            "median"
        ),

        median_genes=(
            "n_genes_by_counts",
            "median"
        ),

        median_mt=(
            "pct_counts_mt",
            "median"
        ),

        median_doublet_score=(
            "doublet_score",
            "median"
        )
    )
    .reset_index()
)


print("\n" + "=" * 100)
print("PREDICTED DOUBLET QC CHARACTERISTICS")
print("=" * 100)

display(
    doublet_qc_summary.round(3)
)


# ------------------------------------------------------------
# 13. Highest-scoring cells
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TOP SCRUBLET SCORES — SHAM")
print("=" * 100)

display(
    scrublet_scores[
        scrublet_scores[
            "condition"
        ] == "Sham"
    ]
    .sort_values(
        "doublet_score",
        ascending=False
    )
    .head(20)
    .round(4)
)


print("\n" + "=" * 100)
print("TOP SCRUBLET SCORES — STZ")
print("=" * 100)

display(
    scrublet_scores[
        scrublet_scores[
            "condition"
        ] == "STZ"
    ]
    .sort_values(
        "doublet_score",
        ascending=False
    )
    .head(20)
    .round(4)
)


# ------------------------------------------------------------
# 14. Save outputs
# ------------------------------------------------------------

qc_file = (
    SCRNA_DIR /
    "GSE313176_scRNA_PRIMARY_QC_summary.csv"
)

scrublet_summary_file = (
    SCRNA_DIR /
    "GSE313176_scRNA_SCRUBLET_summary.csv"
)

scrublet_scores_file = (
    SCRNA_DIR /
    "GSE313176_scRNA_SCRUBLET_cell_scores.csv.gz"
)

doublet_qc_file = (
    SCRNA_DIR /
    "GSE313176_scRNA_SCRUBLET_doublet_QC_summary.csv"
)


qc_summary.to_csv(
    qc_file,
    index=False
)

scrublet_summary.to_csv(
    scrublet_summary_file,
    index=False
)

scrublet_scores.to_csv(
    scrublet_scores_file,
    index=False,
    compression="gzip"
)

doublet_qc_summary.to_csv(
    doublet_qc_file,
    index=False
)


print("\nSaved:")
print(qc_file)
print(scrublet_summary_file)
print(scrublet_scores_file)
print(doublet_qc_file)


print("\n" + "=" * 100)
print("STOP")
print("=" * 100)

print(
    "Primary QC applied."
)

print(
    "Scrublet scores calculated."
)

print(
    "Predicted doublets have NOT been removed."
)

In [ ]:
# ============================================================
# GSE313176 — scRNA CELL 4
# COMBINE + NORMALIZE + UNSUPERVISED CLUSTERING
# + BROAD RETINAL CELL-TYPE AUDIT
#
# IMPORTANT:
# - Uses QC20 cells
# - Removes ZERO cells based on Scrublet
# - Top 1% Scrublet scores are flagged only as candidates
# - No batch correction is performed
# - No condition DE is performed
# ============================================================

import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc


# ------------------------------------------------------------
# 1. Ensure igraph is available for Leiden
# ------------------------------------------------------------

try:
    import igraph
except ImportError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "igraph"
        ]
    )
    import igraph


# ------------------------------------------------------------
# 2. Restore scRNA metadata
# ------------------------------------------------------------

scrna_meta = (
    meta313176[
        meta313176["assay"] == "scRNA"
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(scrna_meta) == 2


# ------------------------------------------------------------
# 3. Load QC20 + Scrublet-scored libraries
# ------------------------------------------------------------

adatas = []

candidate_rows = []

for _, row in scrna_meta.iterrows():

    gsm = row["gsm"]
    condition = row["condition"]

    scored_file = (
        SCRNA_DIR /
        f"{gsm}_{condition}_QC20_SCRUBLET_SCORED.h5ad"
    )

    assert scored_file.exists(), scored_file

    a = sc.read_h5ad(
        scored_file
    )

    # --------------------------------------------------------
    # Scrublet automatic prediction = keep for provenance,
    # but DO NOT use it for filtering.
    # --------------------------------------------------------

    score = (
        a.obs["doublet_score"]
        .astype(float)
    )

    q99 = float(
        score.quantile(0.99)
    )

    a.obs[
        "scrublet_highscore_candidate"
    ] = (
        score >= q99
    )

    # Also flag extreme-complexity cells descriptively
    umi_q99 = float(
        a.obs[
            "total_counts"
        ].quantile(0.99)
    )

    gene_q99 = float(
        a.obs[
            "n_genes_by_counts"
        ].quantile(0.99)
    )

    a.obs[
        "high_complexity_candidate"
    ] = (
        (
            a.obs[
                "total_counts"
            ] >= umi_q99
        )
        |
        (
            a.obs[
                "n_genes_by_counts"
            ] >= gene_q99
        )
    )

    candidate_rows.append({
        "gsm": gsm,
        "condition": condition,

        "n_cells":
            a.n_obs,

        "scrublet_q99":
            q99,

        "scrublet_highscore_n":
            int(
                a.obs[
                    "scrublet_highscore_candidate"
                ].sum()
            ),

        "UMI_q99":
            umi_q99,

        "genes_q99":
            gene_q99,

        "high_complexity_n":
            int(
                a.obs[
                    "high_complexity_candidate"
                ].sum()
            )
    })

    adatas.append(
        a
    )


candidate_summary = pd.DataFrame(
    candidate_rows
)

print("=" * 100)
print("DOUBLET-CANDIDATE FLAGS — NO CELLS REMOVED")
print("=" * 100)

display(
    candidate_summary.round(4)
)


# ------------------------------------------------------------
# 4. Concatenate libraries
#
# Cell names were already prefixed by GSM, so they should
# already be globally unique.
# ------------------------------------------------------------

adata = ad.concat(
    adatas,
    join="inner",
    merge="same",
    index_unique=None
)

assert adata.obs_names.is_unique

print("\nCombined cells:", adata.n_obs)
print("Combined genes:", adata.n_vars)

print("\nCondition counts:")
print(
    adata.obs[
        "condition"
    ].value_counts()
)


# ------------------------------------------------------------
# 5. Preserve raw-count provenance
#
# We do NOT create a duplicate counts layer because raw
# individual h5ad files are already frozen on Drive.
# ------------------------------------------------------------

adata.obs[
    "condition"
] = adata.obs[
    "condition"
].astype("category")

adata.obs[
    "gsm"
] = adata.obs[
    "gsm"
].astype("category")


# ------------------------------------------------------------
# 6. Normalize + log transform
# ------------------------------------------------------------

sc.pp.normalize_total(
    adata,
    target_sum=1e4
)

sc.pp.log1p(
    adata
)


# ------------------------------------------------------------
# 7. Batch-aware HVG SELECTION ONLY
#
# batch_key is used only when choosing robust HVGs.
# We are NOT integrating / Harmony-correcting the libraries.
# ------------------------------------------------------------

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    flavor="seurat",
    batch_key="gsm",
    inplace=True
)

print(
    "\nHighly variable genes:",
    int(
        adata.var[
            "highly_variable"
        ].sum()
    )
)


# ------------------------------------------------------------
# 8. PCA / neighbors / UMAP / Leiden
# ------------------------------------------------------------

sc.tl.pca(
    adata,
    n_comps=50,
    mask_var="highly_variable",
    svd_solver="arpack",
    random_state=42
)

sc.pp.neighbors(
    adata,
    n_neighbors=15,
    n_pcs=30,
    random_state=42
)

sc.tl.umap(
    adata,
    random_state=42
)

sc.tl.leiden(
    adata,
    resolution=0.6,
    key_added="leiden",
    flavor="igraph",
    n_iterations=2,
    directed=False,
    random_state=42
)


# ------------------------------------------------------------
# 9. Broad retinal marker panels
#
# These are for cluster-identification diagnostics only.
# They are NOT inferential signatures.
# ------------------------------------------------------------

marker_panels = {

    "Muller": [
        "Rlbp1",
        "Glul",
        "Slc1a3",
        "Aqp4",
        "Kcnj10",
        "Mlc1",
        "Sox9",
        "Vim"
    ],

    "Rod": [
        "Rho",
        "Nrl",
        "Gnat1",
        "Pde6a",
        "Sag",
        "Rom1"
    ],

    "Cone": [
        "Arr3",
        "Gnat2",
        "Opn1sw",
        "Opn1mw",
        "Pde6c"
    ],

    "Bipolar": [
        "Vsx2",
        "Grm6",
        "Prkca",
        "Trpm1"
    ],

    "RGC": [
        "Rbpms",
        "Pou4f1",
        "Pou4f2",
        "Slc17a6"
    ],

    "Amacrine": [
        "Gad1",
        "Gad2",
        "Tfap2a",
        "Slc6a1"
    ],

    "Horizontal": [
        "Lhx1",
        "Calb1",
        "Onecut1",
        "Prox1"
    ],

    "Microglia": [
        "P2ry12",
        "Cx3cr1",
        "Aif1",
        "C1qa",
        "C1qb"
    ],

    "Astrocyte": [
        "Gfap",
        "Aldoc",
        "S100b"
    ],

    "Endothelial": [
        "Pecam1",
        "Kdr",
        "Emcn",
        "Cldn5",
        "Esam"
    ],

    "Pericyte": [
        "Pdgfrb",
        "Rgs5",
        "Cspg4",
        "Des",
        "Abcc9"
    ],

    "RPE": [
        "Rpe65",
        "Best1",
        "Tyrp1",
        "Mitf"
    ]
}


# ------------------------------------------------------------
# 10. Marker scoring
# ------------------------------------------------------------

score_columns = []

for celltype, genes in marker_panels.items():

    present = [
        g
        for g in genes
        if g in adata.var_names
    ]

    print(
        f"{celltype:12s}: "
        f"{len(present)}/{len(genes)} markers present"
    )

    if len(present) < 2:
        continue

    score_col = (
        f"score_{celltype}"
    )

    sc.tl.score_genes(
        adata,
        gene_list=present,
        score_name=score_col,
        random_state=42,
        use_raw=False
    )

    score_columns.append(
        score_col
    )


# ------------------------------------------------------------
# 11. Cluster × condition counts
# ------------------------------------------------------------

cluster_condition = pd.crosstab(
    adata.obs[
        "leiden"
    ],
    adata.obs[
        "condition"
    ]
)

cluster_condition[
    "total"
] = cluster_condition.sum(
    axis=1
)

for condition in [
    "Sham",
    "STZ"
]:

    if condition not in cluster_condition.columns:

        cluster_condition[
            condition
        ] = 0

cluster_condition[
    "Sham_pct"
] = (
    100
    *
    cluster_condition[
        "Sham"
    ]
    /
    cluster_condition[
        "total"
    ]
)

cluster_condition[
    "STZ_pct"
] = (
    100
    *
    cluster_condition[
        "STZ"
    ]
    /
    cluster_condition[
        "total"
    ]
)


# ------------------------------------------------------------
# 12. Mean marker scores by cluster
# ------------------------------------------------------------

cluster_scores = (
    adata.obs
    .groupby(
        "leiden",
        observed=True
    )[
        score_columns
    ]
    .mean()
)

cluster_scores.columns = [
    x.replace(
        "score_",
        ""
    )
    for x in cluster_scores.columns
]


# ------------------------------------------------------------
# 13. Scrublet candidate enrichment by cluster
# ------------------------------------------------------------

candidate_by_cluster = (
    adata.obs
    .groupby(
        "leiden",
        observed=True
    )
    .agg(
        scrublet_candidate_pct=(
            "scrublet_highscore_candidate",
            lambda x:
                100 *
                np.mean(
                    x.astype(bool)
                )
        ),

        high_complexity_pct=(
            "high_complexity_candidate",
            lambda x:
                100 *
                np.mean(
                    x.astype(bool)
                )
        ),

        median_UMI=(
            "total_counts",
            "median"
        ),

        median_genes=(
            "n_genes_by_counts",
            "median"
        ),

        median_mt=(
            "pct_counts_mt",
            "median"
        )
    )
)


# ------------------------------------------------------------
# 14. Combined cluster summary
# ------------------------------------------------------------

cluster_summary = (
    cluster_condition
    .join(
        cluster_scores,
        how="left"
    )
    .join(
        candidate_by_cluster,
        how="left"
    )
)

marker_names = list(
    cluster_scores.columns
)

cluster_summary[
    "highest_marker_score"
] = (
    cluster_scores[
        marker_names
    ]
    .idxmax(
        axis=1
    )
)

cluster_summary[
    "highest_score_value"
] = (
    cluster_scores[
        marker_names
    ]
    .max(
        axis=1
    )
)


print("\n" + "=" * 100)
print("CLUSTER SUMMARY")
print("=" * 100)

display(
    cluster_summary.round(3)
)


# ------------------------------------------------------------
# 15. Clusters ranked specifically by Müller score
# ------------------------------------------------------------

muller_ranked = (
    cluster_summary
    .sort_values(
        "Muller",
        ascending=False
    )
)

print("\n" + "=" * 100)
print("CLUSTERS RANKED BY MÜLLER SCORE")
print("=" * 100)

display(
    muller_ranked[
        [
            "Sham",
            "STZ",
            "total",
            "Sham_pct",
            "STZ_pct",
            "Muller",
            "Rod",
            "Cone",
            "Bipolar",
            "RGC",
            "Amacrine",
            "Microglia",
            "Astrocyte",
            "Endothelial",
            "Pericyte",
            "scrublet_candidate_pct",
            "high_complexity_pct",
            "median_UMI",
            "median_genes",
            "median_mt",
            "highest_marker_score"
        ]
    ]
    .round(3)
)


# ------------------------------------------------------------
# 16. Save cluster assignment / UMAP coordinates
# ------------------------------------------------------------

cluster_obs = (
    adata.obs[
        [
            "gsm",
            "condition",
            "total_counts",
            "n_genes_by_counts",
            "pct_counts_mt",
            "doublet_score",
            "predicted_doublet",
            "scrublet_highscore_candidate",
            "high_complexity_candidate",
            "leiden"
        ]
        +
        score_columns
    ]
    .copy()
)

cluster_obs[
    "UMAP1"
] = adata.obsm[
    "X_umap"
][:, 0]

cluster_obs[
    "UMAP2"
] = adata.obsm[
    "X_umap"
][:, 1]


obs_file = (
    SCRNA_DIR /
    "GSE313176_scRNA_QC20_cluster_assignments.csv.gz"
)

summary_file = (
    SCRNA_DIR /
    "GSE313176_scRNA_QC20_cluster_summary.csv"
)

cluster_obs.to_csv(
    obs_file,
    compression="gzip"
)

cluster_summary.to_csv(
    summary_file
)


# ------------------------------------------------------------
# 17. Save compact clustered HVG checkpoint
#
# Only 3000 HVGs are stored to avoid duplicating the full
# 32k-gene matrices already safely preserved on Drive.
# ------------------------------------------------------------

adata_hvg = (
    adata[
        :,
        adata.var[
            "highly_variable"
        ]
    ]
    .copy()
)

checkpoint_file = (
    SCRNA_DIR /
    "GSE313176_scRNA_QC20_clustered_HVG3000.h5ad"
)

adata_hvg.write(
    checkpoint_file,
    compression="gzip"
)


print("\nSaved:")
print(obs_file)
print(summary_file)
print(checkpoint_file)

print("\n" + "=" * 100)
print("STOP")
print("=" * 100)

print(
    "No cells were removed based on Scrublet."
)

print(
    "Next step: identify the Müller cluster(s) from "
    "the cluster-marker pattern before any "
    "condition-level Müller analysis."
)

In [ ]:
# ============================================================
# GSE313176 — scRNA CELL 5
# FOCUSED MÜLLER CLUSTER AUDIT
#
# Goal:
#   determine whether Leiden 10 and Leiden 4 represent:
#
#   - clean Müller glia
#   - stressed / low-quality Müller glia
#   - mixed-lineage / doublet populations
#
# IMPORTANT:
# - NO cells removed
# - NO condition-level disease DE
# - rank_genes_groups used ONLY for cluster annotation
# ============================================================

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc


# ------------------------------------------------------------
# 1. Reload QC20 scored matrices
# ------------------------------------------------------------

scrna_meta = (
    meta313176[
        meta313176["assay"] == "scRNA"
    ]
    .copy()
    .reset_index(drop=True)
)

reload_adatas = []

for _, row in scrna_meta.iterrows():

    gsm = row["gsm"]
    condition = row["condition"]

    f = (
        SCRNA_DIR /
        f"{gsm}_{condition}_QC20_SCRUBLET_SCORED.h5ad"
    )

    assert f.exists(), f

    x = sc.read_h5ad(
        f
    )

    reload_adatas.append(
        x
    )


adata_audit = ad.concat(
    reload_adatas,
    join="inner",
    merge="same",
    index_unique=None
)

assert adata_audit.obs_names.is_unique

print(
    "Reloaded cells:",
    adata_audit.n_obs
)

print(
    "Reloaded genes:",
    adata_audit.n_vars
)


# ------------------------------------------------------------
# 2. Restore frozen cluster assignments from previous cell
# ------------------------------------------------------------

assignment_file = (
    SCRNA_DIR /
    "GSE313176_scRNA_QC20_cluster_assignments.csv.gz"
)

assignments = pd.read_csv(
    assignment_file,
    index_col=0
)

assert set(
    adata_audit.obs_names
) == set(
    assignments.index
), (
    "Cell identities do not match saved clustering."
)

assignments = assignments.loc[
    adata_audit.obs_names
]

adata_audit.obs[
    "leiden"
] = (
    assignments[
        "leiden"
    ]
    .astype(str)
    .values
)

adata_audit.obs[
    "leiden"
] = adata_audit.obs[
    "leiden"
].astype("category")


# restore descriptive candidate flags
for col in [
    "scrublet_highscore_candidate",
    "high_complexity_candidate"
]:

    adata_audit.obs[
        col
    ] = (
        assignments[
            col
        ]
        .astype(bool)
        .values
    )


# ------------------------------------------------------------
# 3. Normalize + log1p
#
# Needed only for expression/marker annotation.
# Raw source remains untouched on Drive.
# ------------------------------------------------------------

sc.pp.normalize_total(
    adata_audit,
    target_sum=1e4
)

sc.pp.log1p(
    adata_audit
)


# ------------------------------------------------------------
# 4. Marker panels
# ------------------------------------------------------------

marker_panels = {

    "Muller": [
        "Rlbp1",
        "Glul",
        "Slc1a3",
        "Aqp4",
        "Kcnj10",
        "Mlc1",
        "Sox9",
        "Vim"
    ],

    "Rod": [
        "Rho",
        "Nrl",
        "Gnat1",
        "Pde6a",
        "Sag",
        "Rom1"
    ],

    "Cone": [
        "Arr3",
        "Gnat2",
        "Opn1sw",
        "Opn1mw",
        "Pde6c"
    ],

    "Bipolar": [
        "Vsx2",
        "Grm6",
        "Prkca",
        "Trpm1"
    ],

    "RGC": [
        "Rbpms",
        "Pou4f1",
        "Pou4f2",
        "Slc17a6"
    ],

    "Amacrine": [
        "Gad1",
        "Gad2",
        "Tfap2a",
        "Slc6a1"
    ],

    "Microglia": [
        "P2ry12",
        "Cx3cr1",
        "Aif1",
        "C1qa",
        "C1qb"
    ],

    "Astrocyte": [
        "Gfap",
        "Aldoc",
        "S100b"
    ],

    "Endothelial": [
        "Pecam1",
        "Kdr",
        "Emcn",
        "Cldn5",
        "Esam"
    ],

    "Pericyte": [
        "Pdgfrb",
        "Rgs5",
        "Cspg4",
        "Des",
        "Abcc9"
    ]
}


# ------------------------------------------------------------
# 5. Direct gene-level detection/expression
#
# Focus:
#   cluster 10
#   cluster 4
#   all remaining cells as reference
# ------------------------------------------------------------

target_clusters = [
    "10",
    "4"
]

direct_rows = []


for group_name in [
    "10",
    "4",
    "REST"
]:

    if group_name == "REST":

        mask = (
            ~adata_audit.obs[
                "leiden"
            ]
            .isin(
                target_clusters
            )
        ).to_numpy()

    else:

        mask = (
            adata_audit.obs[
                "leiden"
            ].astype(str)
            == group_name
        ).to_numpy()

    n_cells = int(
        mask.sum()
    )

    for lineage, genes in marker_panels.items():

        for gene in genes:

            if gene not in adata_audit.var_names:
                continue

            gene_idx = int(
                adata_audit.var_names.get_loc(
                    gene
                )
            )

            values = (
                adata_audit[
                    mask,
                    gene_idx
                ]
                .X
            )

            if hasattr(
                values,
                "toarray"
            ):

                values = (
                    values
                    .toarray()
                    .ravel()
                )

            else:

                values = (
                    np.asarray(
                        values
                    )
                    .ravel()
                )

            direct_rows.append({
                "group":
                    group_name,

                "n_cells":
                    n_cells,

                "lineage":
                    lineage,

                "gene":
                    gene,

                "detect_pct":
                    100 *
                    np.mean(
                        values > 0
                    ),

                "mean_logexpr":
                    np.mean(
                        values
                    ),

                "median_logexpr":
                    np.median(
                        values
                    )
            })


direct_marker_audit = pd.DataFrame(
    direct_rows
)


# ------------------------------------------------------------
# 6. Print Müller markers specifically
# ------------------------------------------------------------

print(
    "\n" + "=" * 110
)

print(
    "DIRECT MÜLLER MARKER EXPRESSION"
)

print("=" * 110)

display(
    direct_marker_audit[
        direct_marker_audit[
            "lineage"
        ] == "Muller"
    ]
    .pivot_table(
        index="gene",
        columns="group",
        values=[
            "detect_pct",
            "mean_logexpr"
        ]
    )
    .round(3)
)


# ------------------------------------------------------------
# 7. Direct rod / astrocyte contamination audit
# ------------------------------------------------------------

for lineage in [
    "Rod",
    "Astrocyte",
    "Endothelial",
    "Microglia"
]:

    print(
        "\n" + "=" * 110
    )

    print(
        f"{lineage.upper()} MARKER EXPRESSION"
    )

    print("=" * 110)

    display(
        direct_marker_audit[
            direct_marker_audit[
                "lineage"
            ] == lineage
        ]
        .pivot_table(
            index="gene",
            columns="group",
            values=[
                "detect_pct",
                "mean_logexpr"
            ]
        )
        .round(3)
    )


# ------------------------------------------------------------
# 8. Per-cell lineage marker counts
#
# Counts how many genes from each canonical panel
# are detected in each cell.
# ------------------------------------------------------------

panel_count_columns = []


for lineage, genes in marker_panels.items():

    present = [
        g
        for g in genes
        if g in adata_audit.var_names
    ]

    X = (
        adata_audit[
            :,
            present
        ]
        .X
    )

    detected = (
        X > 0
    )

    counts = np.asarray(
        detected.sum(
            axis=1
        )
    ).ravel()

    col = (
        f"n_{lineage}_markers"
    )

    adata_audit.obs[
        col
    ] = counts

    panel_count_columns.append(
        col
    )


# ------------------------------------------------------------
# 9. Cluster-level lineage support
#
# Strong-support thresholds are descriptive:
#
# Müller: >=3 of 8 markers
# Rod: >=3 of 6
# Other lineages: >=2 markers
# ------------------------------------------------------------

support_threshold = {
    "Muller": 3,
    "Rod": 3,
    "Cone": 2,
    "Bipolar": 2,
    "RGC": 2,
    "Amacrine": 2,
    "Microglia": 2,
    "Astrocyte": 2,
    "Endothelial": 2,
    "Pericyte": 2
}


support_rows = []


for cluster in [
    "10",
    "4"
]:

    mask = (
        adata_audit.obs[
            "leiden"
        ].astype(str)
        == cluster
    )

    obs = (
        adata_audit.obs.loc[
            mask
        ]
    )

    for lineage, threshold in (
        support_threshold.items()
    ):

        col = (
            f"n_{lineage}_markers"
        )

        support_rows.append({
            "cluster":
                cluster,

            "lineage":
                lineage,

            "n_cells":
                len(obs),

            "threshold_n_markers":
                threshold,

            "mean_markers_detected":
                obs[
                    col
                ].mean(),

            "median_markers_detected":
                obs[
                    col
                ].median(),

            "pct_cells_meeting_support":
                100 *
                (
                    obs[
                        col
                    ] >= threshold
                ).mean()
        })


panel_support = pd.DataFrame(
    support_rows
)

print(
    "\n" + "=" * 110
)

print(
    "LINEAGE SUPPORT WITHIN CANDIDATE MÜLLER CLUSTERS"
)

print("=" * 110)

display(
    panel_support
    .pivot_table(
        index="lineage",
        columns="cluster",
        values=[
            "mean_markers_detected",
            "pct_cells_meeting_support"
        ]
    )
    .round(2)
)


# ------------------------------------------------------------
# 10. QC / Scrublet profile by cluster AND condition
# ------------------------------------------------------------

qc_cluster_condition = (
    adata_audit.obs[
        adata_audit.obs[
            "leiden"
        ].astype(str)
        .isin(
            target_clusters
        )
    ]
    .groupby(
        [
            "leiden",
            "condition"
        ],
        observed=True
    )
    .agg(
        n_cells=(
            "gsm",
            "size"
        ),

        median_UMI=(
            "total_counts",
            "median"
        ),

        median_genes=(
            "n_genes_by_counts",
            "median"
        ),

        median_mt=(
            "pct_counts_mt",
            "median"
        ),

        median_scrublet=(
            "doublet_score",
            "median"
        ),

        scrublet_q99_candidate_pct=(
            "scrublet_highscore_candidate",
            lambda x:
                100 *
                np.mean(
                    x.astype(bool)
                )
        ),

        high_complexity_pct=(
            "high_complexity_candidate",
            lambda x:
                100 *
                np.mean(
                    x.astype(bool)
                )
        )
    )
    .reset_index()
)


print(
    "\n" + "=" * 110
)

print(
    "CLUSTER × CONDITION QC PROFILE"
)

print("=" * 110)

display(
    qc_cluster_condition.round(3)
)


# ------------------------------------------------------------
# 11. Mixed-lineage diagnostics
#
# Particularly:
#   Müller + Rod
#   Müller + Endothelial
#   Müller + Astrocyte
# ------------------------------------------------------------

mixed_rows = []


for cluster in [
    "10",
    "4"
]:

    obs = adata_audit.obs[
        adata_audit.obs[
            "leiden"
        ].astype(str)
        == cluster
    ]

    muller_positive = (
        obs[
            "n_Muller_markers"
        ] >= 3
    )

    for lineage in [
        "Rod",
        "Astrocyte",
        "Endothelial",
        "Microglia",
        "Pericyte"
    ]:

        threshold = (
            support_threshold[
                lineage
            ]
        )

        other_positive = (
            obs[
                f"n_{lineage}_markers"
            ] >= threshold
        )

        mixed_rows.append({
            "cluster":
                cluster,

            "comparison":
                f"Muller+{lineage}",

            "n_cells":
                len(obs),

            "Muller_supported_pct":
                100 *
                muller_positive.mean(),

            "other_supported_pct":
                100 *
                other_positive.mean(),

            "double_supported_pct":
                100 *
                (
                    muller_positive
                    &
                    other_positive
                ).mean()
        })


mixed_lineage = pd.DataFrame(
    mixed_rows
)

print(
    "\n" + "=" * 110
)

print(
    "MIXED-LINEAGE COEXPRESSION AUDIT"
)

print("=" * 110)

display(
    mixed_lineage.round(2)
)


# ------------------------------------------------------------
# 12. Cluster markers for annotation only
#
# CELL-LEVEL P values are NOT used for disease inference.
# We only inspect marker identity / effect direction.
# ------------------------------------------------------------

sc.tl.rank_genes_groups(
    adata_audit,
    groupby="leiden",
    groups=[
        "10",
        "4"
    ],
    reference="rest",
    method="wilcoxon",
    use_raw=False,
    pts=True,
    key_added="candidate_muller_markers"
)


marker_rows = []


for cluster in [
    "10",
    "4"
]:

    df = sc.get.rank_genes_groups_df(
        adata_audit,
        group=cluster,
        key="candidate_muller_markers"
    )

    df = df.head(
        40
    )

    df[
        "cluster"
    ] = cluster

    marker_rows.append(
        df
    )


candidate_cluster_markers = pd.concat(
    marker_rows,
    ignore_index=True
)


print(
    "\n" + "=" * 110
)

print(
    "TOP CLUSTER 10 MARKERS — ANNOTATION ONLY"
)

print("=" * 110)

display(
    candidate_cluster_markers[
        candidate_cluster_markers[
            "cluster"
        ] == "10"
    ][
        [
            "names",
            "scores",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
    .head(30)
    .round(4)
)


print(
    "\n" + "=" * 110
)

print(
    "TOP CLUSTER 4 MARKERS — ANNOTATION ONLY"
)

print("=" * 110)

display(
    candidate_cluster_markers[
        candidate_cluster_markers[
            "cluster"
        ] == "4"
    ][
        [
            "names",
            "scores",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
    .head(30)
    .round(4)
)


# ------------------------------------------------------------
# 13. Save
# ------------------------------------------------------------

direct_marker_audit.to_csv(
    SCRNA_DIR /
    "GSE313176_candidate_Muller_direct_marker_audit.csv",
    index=False
)

panel_support.to_csv(
    SCRNA_DIR /
    "GSE313176_candidate_Muller_lineage_support.csv",
    index=False
)

qc_cluster_condition.to_csv(
    SCRNA_DIR /
    "GSE313176_candidate_Muller_QC_by_condition.csv",
    index=False
)

mixed_lineage.to_csv(
    SCRNA_DIR /
    "GSE313176_candidate_Muller_mixed_lineage_audit.csv",
    index=False
)

candidate_cluster_markers.to_csv(
    SCRNA_DIR /
    "GSE313176_candidate_Muller_cluster_markers.csv",
    index=False
)


print(
    "\n" + "=" * 110
)

print(
    "STOP"
)

print("=" * 110)

print(
    "No cells removed."
)

print(
    "No Müller population frozen yet."
)

print(
    "Next step: decide whether cluster 10 alone, "
    "cluster 10 + cluster 4, or a quality-restricted "
    "subset should define the Müller compartment."
)

In [ ]:
# ============================================================
# GSE313176 — FREEZE PRIMARY MÜLLER POPULATION
# Primary = Leiden cluster 10
# Cluster 4 retained separately for sensitivity analysis
# ============================================================

import pandas as pd

assignments = pd.read_csv(
    SCRNA_DIR / "GSE313176_scRNA_QC20_cluster_assignments.csv.gz",
    index_col=0
)

muller = assignments[
    assignments["leiden"].astype(str) == "10"
].copy()

muller["primary_Muller"] = True

print("PRIMARY MÜLLER CELLS")
print("=" * 50)

print("\nTotal:", len(muller))
print("\nBy condition:")
print(muller["condition"].value_counts())

print("\nPercent of QC cells:")
for condition in ["Sham", "STZ"]:
    n_muller = (muller["condition"] == condition).sum()
    n_all = (assignments["condition"] == condition).sum()

    print(
        condition,
        f"{n_muller}/{n_all}",
        f"({100*n_muller/n_all:.2f}%)"
    )

out = (
    SCRNA_DIR /
    "GSE313176_PRIMARY_Muller_cluster10_cells.csv.gz"
)

muller.to_csv(
    out,
    compression="gzip"
)

print("\nSaved:")
print(out)

print("\nPRIMARY DEFINITION FROZEN = LEIDEN 10")
print("Cluster 4 reserved for sensitivity analysis.")

In [ ]:
# ============================================================
# GSE313176 — MÜLLER LOCKED-SIGNATURE RESPONSE
# Descriptive pseudobulk only: 1 Sham library vs 1 STZ library
# ============================================================

import json
import numpy as np
import pandas as pd
import scanpy as sc

# frozen Müller IDs
muller_ids = pd.read_csv(
    SCRNA_DIR / "GSE313176_PRIMARY_Muller_cluster10_cells.csv.gz",
    index_col=0
).index

# reload RAW counts
parts = []
for gsm, condition in [
    ("GSM9363367", "Sham"),
    ("GSM9363368", "STZ")
]:
    a = sc.read_h5ad(SCRNA_DIR / f"{gsm}_{condition}_RAW.h5ad")
    keep = a.obs_names.isin(muller_ids)
    a = a[keep].copy()
    a.obs["condition"] = condition
    parts.append(a)

# sum raw counts within each library
pb = {}

for a in parts:
    condition = a.obs["condition"].iloc[0]
    counts = np.asarray(a.X.sum(axis=0)).ravel()
    libsize = counts.sum()
    cpm = counts / libsize * 1e6

    pb[condition] = pd.Series(
        cpm,
        index=a.var_names
    )

effects = pd.DataFrame({
    "CPM_Sham": pb["Sham"],
    "CPM_STZ": pb["STZ"]
})

effects["log2FC_STZ_vs_Sham"] = np.log2(
    (effects["CPM_STZ"] + 0.5) /
    (effects["CPM_Sham"] + 0.5)
)

# unique locked signatures
sig_names = [
    "early_IFN_alpha_beta",
    "early_retinoid_metabolism",
    "early_ECM_GAG_exploratory",
    "early_up_transient_all",
    "persistent_up_all_descriptive",
    "up_to_down_reversal_all_descriptive"
]

rows = []

for sig in sig_names:
    genes = locked_signatures[sig]
    present = [g for g in genes if g in effects.index]
    x = effects.loc[present, "log2FC_STZ_vs_Sham"]

    rows.append({
        "signature": sig,
        "locked_genes": len(genes),
        "genes_present": len(present),
        "median_log2FC": x.median(),
        "mean_log2FC": x.mean(),
        "fraction_positive": (x > 0).mean()
    })

summary = pd.DataFrame(rows)

print("=" * 80)
print("MÜLLER PSEUDOBULK — LOCKED SIGNATURES")
print("=" * 80)
display(summary.round(3))

print("\nPRIMARY MODULE GENES")
display(
    effects.loc[
        sorted(set(
            locked_signatures["early_IFN_alpha_beta"]
            + locked_signatures["early_retinoid_metabolism"]
            + locked_signatures["early_ECM_GAG_exploratory"]
        ))
    ][["CPM_Sham", "CPM_STZ", "log2FC_STZ_vs_Sham"]]
    .round(3)
)

effects.to_csv(
    SCRNA_DIR / "GSE313176_PRIMARY_Muller_pseudobulk_gene_effects.csv"
)

summary.to_csv(
    SCRNA_DIR / "GSE313176_PRIMARY_Muller_locked_signature_summary.csv",
    index=False
)

print("\nSaved.")
print("STOP — descriptive only; no P-values because n=1 library/condition.")

In [ ]:
# ============================================================
# GSE313176 — MÜLLER SENSITIVITY
# Compare:
#   cluster 10 alone
#   cluster 4 alone
#   clusters 10 + 4
# ============================================================

import numpy as np
import pandas as pd
import scanpy as sc

assign = pd.read_csv(
    SCRNA_DIR / "GSE313176_scRNA_QC20_cluster_assignments.csv.gz",
    index_col=0
)

groups = {
    "cluster10_primary": ["10"],
    "cluster4_stressed": ["4"],
    "cluster10_plus4": ["10", "4"]
}

rows = []

for label, clusters in groups.items():

    ids = assign[
        assign["leiden"].astype(str).isin(clusters)
    ].index

    pb = {}

    for gsm, condition in [
        ("GSM9363367", "Sham"),
        ("GSM9363368", "STZ")
    ]:
        a = sc.read_h5ad(
            SCRNA_DIR / f"{gsm}_{condition}_RAW.h5ad"
        )

        a = a[a.obs_names.isin(ids)]

        counts = np.asarray(
            a.X.sum(axis=0)
        ).ravel()

        cpm = counts / counts.sum() * 1e6

        pb[condition] = pd.Series(
            cpm,
            index=a.var_names
        )

    fc = np.log2(
        (pb["STZ"] + 0.5) /
        (pb["Sham"] + 0.5)
    )

    for sig in [
        "early_IFN_alpha_beta",
        "early_retinoid_metabolism",
        "early_ECM_GAG_exploratory",
        "early_up_transient_all"
    ]:

        genes = [
            g for g in locked_signatures[sig]
            if g in fc.index
        ]

        x = fc.loc[genes]

        rows.append({
            "population": label,
            "signature": sig,
            "n_genes": len(x),
            "median_log2FC": x.median(),
            "mean_log2FC": x.mean(),
            "fraction_positive": (x > 0).mean()
        })

result = pd.DataFrame(rows)

print("=" * 80)
print("MÜLLER SENSITIVITY ANALYSIS")
print("=" * 80)

display(
    result.round(3)
)

result.to_csv(
    SCRNA_DIR /
    "GSE313176_Muller_cluster10_vs_cluster4_signature_sensitivity.csv",
    index=False
)

print("\nSaved.")
print("STOP.")

In [ ]:
# ============================================================
# GSE313176 — WHERE DO THE LOCKED PROGRAMS LOCALIZE AT 3 MONTHS?
# Descriptive pseudobulk STZ-vs-Sham within each Leiden cluster
# ============================================================

import numpy as np
import pandas as pd
import scanpy as sc

assign = pd.read_csv(
    SCRNA_DIR / "GSE313176_scRNA_QC20_cluster_assignments.csv.gz",
    index_col=0
)

cluster_summary = pd.read_csv(
    SCRNA_DIR / "GSE313176_scRNA_QC20_cluster_summary.csv",
    index_col=0
)

raw = {}
for gsm, condition in [
    ("GSM9363367", "Sham"),
    ("GSM9363368", "STZ")
]:
    raw[condition] = sc.read_h5ad(
        SCRNA_DIR / f"{gsm}_{condition}_RAW.h5ad"
    )

signatures = {
    "IFN": locked_signatures["early_IFN_alpha_beta"],
    "Retinoid": locked_signatures["early_retinoid_metabolism"],
    "ECM_GAG": locked_signatures["early_ECM_GAG_exploratory"],
    "Early184": locked_signatures["early_up_transient_all"]
}

rows = []

for cluster in sorted(assign["leiden"].astype(str).unique(), key=int):

    ids = assign[
        assign["leiden"].astype(str) == cluster
    ].index

    counts_by_condition = {}
    n_by_condition = {}

    for condition in ["Sham", "STZ"]:

        a = raw[condition]
        keep = a.obs_names.isin(ids)
        n_by_condition[condition] = int(keep.sum())

        if keep.sum() == 0:
            continue

        x = np.asarray(
            a[keep].X.sum(axis=0)
        ).ravel()

        counts_by_condition[condition] = pd.Series(
            x / x.sum() * 1e6,
            index=a.var_names
        )

    # require enough cells in BOTH conditions
    if min(n_by_condition.values()) < 20:
        continue

    fc = np.log2(
        (counts_by_condition["STZ"] + 0.5) /
        (counts_by_condition["Sham"] + 0.5)
    )

    label = (
        cluster_summary.loc[int(cluster), "highest_marker_score"]
        if int(cluster) in cluster_summary.index
        else "Unknown"
    )

    row = {
        "cluster": cluster,
        "annotation": label,
        "Sham_n": n_by_condition["Sham"],
        "STZ_n": n_by_condition["STZ"]
    }

    for name, genes in signatures.items():
        g = [x for x in genes if x in fc.index]
        row[name] = fc.loc[g].median()

    rows.append(row)

result = pd.DataFrame(rows)

print("3-MONTH CELL-COMPARTMENT SIGNATURE EFFECTS")
print("=" * 70)

display(
    result.sort_values("Retinoid", ascending=False).round(3)
)

result.to_csv(
    SCRNA_DIR / "GSE313176_3mo_cluster_signature_localization.csv",
    index=False
)

print("\nSaved. STOP.")

In [ ]:
# ============================================================
# GSE313176 — 3mo BULK vs scRNA WHOLE-RETINA CONCORDANCE
# ============================================================

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import spearmanr

# bulk DESeq2
bulk = pd.read_csv(
    OUTDIR / "GSE313176_retina_bulk_DESeq2_all_timepoints.csv.gz"
)

bulk = bulk[
    (bulk["timepoint"] == "3mo") &
    bulk["symbol"].notna()
].copy()

# keep one-to-one symbols only
unique_symbols = bulk["symbol"].value_counts()
bulk = bulk[
    bulk["symbol"].isin(
        unique_symbols[unique_symbols == 1].index
    )
]

bulk = bulk.set_index("symbol")

# scRNA whole-retina pseudobulk using QC-passing cells
assign = pd.read_csv(
    SCRNA_DIR / "GSE313176_scRNA_QC20_cluster_assignments.csv.gz",
    index_col=0
)

pb = {}

for gsm, condition in [
    ("GSM9363367", "Sham"),
    ("GSM9363368", "STZ")
]:
    a = sc.read_h5ad(
        SCRNA_DIR / f"{gsm}_{condition}_RAW.h5ad"
    )

    a = a[a.obs_names.isin(assign.index)]

    counts = np.asarray(a.X.sum(axis=0)).ravel()
    cpm = counts / counts.sum() * 1e6

    pb[condition] = pd.Series(cpm, index=a.var_names)

scrna_fc = np.log2(
    (pb["STZ"] + 0.5) /
    (pb["Sham"] + 0.5)
)

# global concordance
shared = bulk.index.intersection(scrna_fc.index)

comp = pd.DataFrame({
    "bulk_log2FC": bulk.loc[shared, "log2FoldChange"],
    "scrna_log2FC": scrna_fc.loc[shared],
    "baseMean": bulk.loc[shared, "baseMean"],
    "scRNA_maxCPM": np.maximum(
        pb["Sham"].reindex(shared),
        pb["STZ"].reindex(shared)
    )
}).dropna()

# avoid near-zero genes
comp = comp[
    (comp["baseMean"] >= 10) &
    (comp["scRNA_maxCPM"] >= 1)
]

rho, p = spearmanr(
    comp["bulk_log2FC"],
    comp["scrna_log2FC"]
)

print("GLOBAL 3mo CONCORDANCE")
print("=" * 60)
print("Genes:", len(comp))
print("Spearman rho:", round(rho, 3))
print("P:", f"{p:.3e}")

# locked signatures
rows = []

for sig in [
    "early_IFN_alpha_beta",
    "early_retinoid_metabolism",
    "early_ECM_GAG_exploratory",
    "early_up_transient_all"
]:
    genes = [
        g for g in locked_signatures[sig]
        if g in comp.index
    ]

    rows.append({
        "signature": sig,
        "n_genes": len(genes),
        "bulk_median_log2FC":
            comp.loc[genes, "bulk_log2FC"].median(),
        "scRNA_median_log2FC":
            comp.loc[genes, "scrna_log2FC"].median()
    })

sig_compare = pd.DataFrame(rows)

print("\nLOCKED SIGNATURE COMPARISON")
display(sig_compare.round(3))

comp.to_csv(
    SCRNA_DIR / "GSE313176_3mo_bulk_vs_scRNA_gene_concordance.csv"
)

sig_compare.to_csv(
    SCRNA_DIR / "GSE313176_3mo_bulk_vs_scRNA_signature_concordance.csv",
    index=False
)

print("\nSaved. STOP.")

In [ ]:
# ============================================================
# GSE313176 — SIGN AUDIT
# 1) Does DESeq2 sign agree with simple bulk CPM STZ/Sham?
# 2) Do our Müller cells reproduce published Glul/Slc1a3 loss?
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# ---------- bulk ----------
counts = pd.read_csv(
    OUTDIR / "GSE313176_retina_bulk_RAW_COUNTS.csv.gz",
    index_col=0
)

de = pd.read_csv(
    OUTDIR / "GSE313176_retina_bulk_DESeq2_all_timepoints.csv.gz"
)

d3 = de[de["timepoint"] == "3mo"].copy()

stz = ["GSM9363345", "GSM9363347", "GSM9363351"]
sham = ["GSM9363349", "GSM9363353", "GSM9363355"]

cpm = counts.div(counts.sum(axis=0), axis=1) * 1e6

simple_fc = np.log2(
    (cpm[stz].mean(axis=1) + 0.5) /
    (cpm[sham].mean(axis=1) + 0.5)
)

check = d3.set_index("ensembl_gene_id")
shared = check.index.intersection(simple_fc.index)

x = check.loc[shared, "log2FoldChange"]
y = simple_fc.loc[shared]

ok = x.notna() & y.notna()

rho = spearmanr(x[ok], y[ok]).statistic

print("BULK SIGN CHECK")
print("=" * 50)
print("DESeq2 vs direct CPM rho:", round(rho, 4))
print("Expected: strongly POSITIVE if STZ/Sham orientation is correct.")

# ---------- published Müller sanity check ----------
muller = pd.read_csv(
    SCRNA_DIR / "GSE313176_PRIMARY_Muller_pseudobulk_gene_effects.csv",
    index_col=0
)

print("\nPUBLISHED MÜLLER POSITIVE-CONTROL GENES")
print("=" * 50)

for gene in ["Glul", "Slc1a3", "Vim", "Rlbp1"]:
    if gene in muller.index:
        print(
            gene,
            "log2FC STZ/Sham =",
            round(muller.loc[gene, "log2FC_STZ_vs_Sham"], 3)
        )

print("\nSTOP.")

## Part III — Human Müller scRNA atlas translation

In [ ]:
# ============================================================
# HUMAN DR ATLAS — CELL 1
# Inspect Figshare files before downloading anything
# ============================================================

import requests
import pandas as pd

HUMAN_DIR = ROOT / "02_Human_scRNA_2026"
HUMAN_DIR.mkdir(exist_ok=True)

article_id = 29204750

j = requests.get(
    f"https://api.figshare.com/v2/articles/{article_id}"
).json()

files = pd.DataFrame([
    {
        "file_id": f["id"],
        "name": f["name"],
        "size_MB": round(f["size"] / 1024**2, 1),
        "download_url": f["download_url"]
    }
    for f in j["files"]
])

print("Title:", j["title"])
print("Total files:", len(files))
print("Total size GB:", round(files["size_MB"].sum()/1024, 2))

display(
    files.sort_values("size_MB", ascending=False)
)

files.to_csv(
    HUMAN_DIR / "Figshare_29204750_file_manifest.csv",
    index=False
)

print("\nSaved manifest. STOP.")

In [ ]:
# ============================================================
# HUMAN ATLAS — CELL 2
# Download ONLY README + Metadata.xlsx
# ============================================================

import requests, pandas as pd

small = files[files["name"].isin(["README", "Metadata.xlsx"])]

for _, r in small.iterrows():
    p = HUMAN_DIR / r["name"]
    p.write_bytes(requests.get(r["download_url"]).content)
    print("Saved:", p)

print("\nREADME")
print("=" * 70)
print((HUMAN_DIR / "README").read_text(errors="ignore")[:12000])

xls = pd.ExcelFile(HUMAN_DIR / "Metadata.xlsx")

print("\nEXCEL SHEETS:", xls.sheet_names)

for sheet in xls.sheet_names:
    print("\n", "=" * 70)
    print("SHEET:", sheet)
    print("=" * 70)
    display(pd.read_excel(xls, sheet_name=sheet).head(30))

In [ ]:
# HUMAN ATLAS — map Figshare triplets to Sample 1–20

m = pd.read_csv(
    HUMAN_DIR / "Figshare_29204750_file_manifest.csv"
).sort_values("file_id")

print(
    m[["file_id", "name", "size_MB"]]
    .to_string(index=False)
)

print("\nTotal:", len(m), "files")
print("Expected: 20 × 3 data files + README + Metadata = 62")

In [ ]:
# HUMAN ATLAS — inspect raw Figshare file metadata

import requests, pprint

j = requests.get(
    "https://api.figshare.com/v2/articles/29204750"
).json()

print("Available fields for first 6 data files:\n")

for f in j["files"][:6]:
    pprint.pp(f)
    print("-" * 60)

print("\nSTOP.")

In [ ]:
# HUMAN ATLAS — validate Sample 1→20 ordering using barcode counts

import requests, gzip, io
import pandas as pd

data = files[~files["name"].isin(["README", "Metadata.xlsx"])].sort_values("file_id")

# each consecutive triplet = one uploaded sample
triplets = []
for i in range(20):
    block = data.iloc[i*3:(i+1)*3]

    barcode = block[block["name"] == "barcodes.tsv.gz"].iloc[0]

    raw = requests.get(barcode["download_url"]).content
    n_cells = len(gzip.decompress(raw).splitlines())

    sample = i + 1
    status = "NON" if sample <= 7 else ("DM" if sample <= 12 else "DR")

    triplets.append({
        "sample": sample,
        "status": status,
        "n_cells": n_cells,
        "matrix_MB": block.loc[
            block["name"] == "matrix.mtx.gz", "size_MB"
        ].iloc[0]
    })

check = pd.DataFrame(triplets)

print("SAMPLE CELL COUNTS")
display(check)

print("\nGROUP TOTALS")
group = check.groupby("status")["n_cells"].sum()
display(pd.DataFrame({
    "cells": group,
    "percent": 100 * group / group.sum()
}).round(2))

print("\nTOTAL CELLS:", check["n_cells"].sum())
print("Published total: 297121")

check.to_csv(
    HUMAN_DIR / "Human_atlas_tentative_sample_mapping_cellcounts.csv",
    index=False
)

print("\nSTOP.")

In [ ]:
# HUMAN ATLAS — freeze sample/donor/file manifest

import pandas as pd

meta = pd.read_excel(HUMAN_DIR / "Metadata.xlsx")

# Known paired eyes from identical donor characteristics
donor_map = {
    1:"D01", 2:"D02", 3:"D03", 4:"D04", 5:"D05",
    6:"D06", 7:"D06",
    8:"D07",
    9:"D08", 10:"D08",
    11:"D09", 12:"D09",
    13:"D10", 14:"D10",
    15:"D11", 16:"D11",
    17:"D12", 18:"D12",
    19:"D13", 20:"D13"
}

data = files[
    ~files["name"].isin(["README", "Metadata.xlsx"])
].sort_values("file_id").reset_index(drop=True)

rows = []

for i in range(20):

    sample = i + 1
    block = data.iloc[i*3:(i+1)*3]

    r = meta.iloc[i]

    rows.append({
        "sample": sample,
        "donor": donor_map[sample],
        "status": r["Diabetic Status"],
        "age": r["Age"],
        "sex": r["Sex"],
        "eye": r["Left/Right Eye"],
        "matrix_id": int(block.loc[
            block["name"] == "matrix.mtx.gz", "file_id"
        ].iloc[0]),
        "features_id": int(block.loc[
            block["name"] == "features.tsv.gz", "file_id"
        ].iloc[0]),
        "barcodes_id": int(block.loc[
            block["name"] == "barcodes.tsv.gz", "file_id"
        ].iloc[0])
    })

human_manifest = pd.DataFrame(rows)

display(human_manifest)

print("\nDONORS BY GROUP")
print(
    human_manifest.groupby("status")["donor"].nunique()
)

print("\nTOTAL DONORS:", human_manifest["donor"].nunique())

human_manifest.to_csv(
    HUMAN_DIR / "Human_atlas_LOCKED_sample_donor_manifest.csv",
    index=False
)

print("\nSaved. STOP.")

In [ ]:
# HUMAN ATLAS — PILOT: 1 NON + 1 DM + 1 DR

import requests, pandas as pd, numpy as np, scanpy as sc
from pathlib import Path

manifest = pd.read_csv(HUMAN_DIR / "Human_atlas_LOCKED_sample_donor_manifest.csv")
fig = pd.read_csv(HUMAN_DIR / "Figshare_29204750_file_manifest.csv")

pilot_samples = [1, 8, 13]   # NON, DM, DR
muller_genes = ["RLBP1","GLUL","SLC1A3","AQP4","KCNJ10","MLC1","SOX9","VIM"]

rows = []

for s in pilot_samples:

    r = manifest[manifest["sample"] == s].iloc[0]

    folder = HUMAN_DIR / "pilot" / f"Sample{s}_{r['status']}"
    folder.mkdir(parents=True, exist_ok=True)

    # download triplet
    for kind, fid in [
        ("matrix.mtx.gz", r["matrix_id"]),
        ("features.tsv.gz", r["features_id"]),
        ("barcodes.tsv.gz", r["barcodes_id"])
    ]:
        out = folder / kind
        if not out.exists():
            url = fig.loc[fig["file_id"] == fid, "download_url"].iloc[0]
            with requests.get(url, stream=True) as resp:
                resp.raise_for_status()
                with open(out, "wb") as f:
                    for chunk in resp.iter_content(1024 * 1024):
                        f.write(chunk)

    # load
    a = sc.read_10x_mtx(
        folder,
        var_names="gene_symbols",
        make_unique=True
    )

    a.var["mt"] = a.var_names.str.upper().str.startswith("MT-")

    sc.pp.calculate_qc_metrics(
        a,
        qc_vars=["mt"],
        percent_top=None,
        inplace=True
    )

    # simple Müller-marker detection audit
    present = [g for g in muller_genes if g in a.var_names]

    X = a[:, present].X
    n_muller_markers = np.asarray((X > 0).sum(axis=1)).ravel()

    rows.append({
        "sample": s,
        "donor": r["donor"],
        "status": r["status"],
        "cells": a.n_obs,
        "median_UMI": a.obs["total_counts"].median(),
        "median_genes": a.obs["n_genes_by_counts"].median(),
        "median_mt": a.obs["pct_counts_mt"].median(),
        "q95_mt": a.obs["pct_counts_mt"].quantile(.95),
        "Muller_markers_present": len(present),
        "cells_with_3plus_Muller_markers": int((n_muller_markers >= 3).sum()),
        "pct_with_3plus_Muller_markers": 100 * (n_muller_markers >= 3).mean()
    })

pilot = pd.DataFrame(rows)

print("HUMAN PILOT QC + MÜLLER AUDIT")
print("=" * 70)
display(pilot.round(2))

pilot.to_csv(
    HUMAN_DIR / "Human_atlas_PILOT_QC_Muller_audit.csv",
    index=False
)

print("\nSaved. STOP.")

In [ ]:
# HUMAN ATLAS — PILOT CELL-TYPE CLUSTERING
# 1 NON + 1 DM + 1 DR
# Goal: identify genuine Müller cluster(s), not marker-threshold cells.

import numpy as np, pandas as pd, scanpy as sc, anndata as ad

pilot_samples = [(1,"NON"), (8,"DM"), (13,"DR")]
parts = []

for s, status in pilot_samples:
    folder = HUMAN_DIR / "pilot" / f"Sample{s}_{status}"
    a = sc.read_10x_mtx(folder, var_names="gene_symbols", make_unique=True)

    a.obs["sample"] = str(s)
    a.obs["status"] = status
    a.obs_names = [f"S{s}_{x}" for x in a.obs_names]

    a.var["mt"] = a.var_names.str.upper().str.startswith("MT-")
    sc.pp.calculate_qc_metrics(a, qc_vars=["mt"], percent_top=None, inplace=True)

    # conservative QC only
    a = a[
        (a.obs["n_genes_by_counts"] >= 200) &
        (a.obs["total_counts"] >= 500) &
        (a.obs["pct_counts_mt"] < 15)
    ].copy()

    parts.append(a)

adata = ad.concat(parts, join="inner")

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    flavor="seurat",
    batch_key="sample"
)

sc.tl.pca(adata, n_comps=40, mask_var="highly_variable")
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=25)

sc.tl.leiden(
    adata,
    resolution=0.6,
    flavor="igraph",
    n_iterations=2,
    directed=False,
    random_state=42
)

panels = {
    "Muller":["RLBP1","GLUL","SLC1A3","AQP4","KCNJ10","MLC1","SOX9","VIM"],
    "Rod":["RHO","NRL","GNAT1","PDE6A","SAG"],
    "Cone":["ARR3","GNAT2","OPN1SW","OPN1MW"],
    "Bipolar":["VSX2","GRM6","PRKCA","TRPM1"],
    "RGC":["RBPMS","POU4F1","POU4F2","SLC17A6"],
    "Amacrine":["GAD1","GAD2","TFAP2A"],
    "Microglia":["P2RY12","CX3CR1","AIF1","C1QA"],
    "Endothelial":["PECAM1","KDR","EMCN","CLDN5"],
    "Pericyte":["PDGFRB","RGS5","CSPG4","ABCC9"],
    "Astrocyte":["GFAP","ALDOC","S100B"],
    "RPE":["RPE65","BEST1","TYRP1","MITF"]
}

score_cols = []

for name, genes in panels.items():
    present = [g for g in genes if g in adata.var_names]
    sc.tl.score_genes(
        adata,
        present,
        score_name=f"score_{name}",
        random_state=42
    )
    score_cols.append(f"score_{name}")

summary = (
    adata.obs
    .groupby("leiden", observed=True)
    .agg(
        total=("sample","size"),
        NON=("status", lambda x: (x=="NON").sum()),
        DM=("status", lambda x: (x=="DM").sum()),
        DR=("status", lambda x: (x=="DR").sum()),
        **{c:(c,"mean") for c in score_cols}
    )
)

summary["best_identity"] = (
    summary[score_cols]
    .idxmax(axis=1)
    .str.replace("score_","", regex=False)
)

print("PILOT CLUSTERS RANKED BY MÜLLER SCORE")
print("=" * 70)

display(
    summary.sort_values("score_Muller", ascending=False).round(3)
)

summary.to_csv(
    HUMAN_DIR / "Human_atlas_PILOT_cluster_summary.csv"
)

print("\nSaved. STOP.")

In [ ]:
# ============================================================
# HUMAN PILOT — IDENTIFY MÜLLER CLUSTERS PER SAMPLE
# ============================================================

import numpy as np, pandas as pd, scanpy as sc

pilot_samples = [(1,"NON"), (8,"DM"), (13,"DR")]

muller = ["RLBP1","GLUL","SLC1A3","AQP4","KCNJ10","MLC1","SOX9","VIM"]
rivals = {
    "Rod":["RHO","NRL","GNAT1","SAG"],
    "Cone":["ARR3","GNAT2","OPN1SW","OPN1MW"],
    "Bipolar":["VSX2","GRM6","PRKCA","TRPM1"],
    "Microglia":["P2RY12","CX3CR1","AIF1","C1QA"],
    "Endothelial":["PECAM1","KDR","EMCN","CLDN5"],
    "Pericyte":["PDGFRB","RGS5","CSPG4","ABCC9"]
}

rows = []

for s, status in pilot_samples:

    a = sc.read_10x_mtx(
        HUMAN_DIR / "pilot" / f"Sample{s}_{status}",
        var_names="gene_symbols",
        make_unique=True
    )

    a.var["mt"] = a.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(a, qc_vars=["mt"], percent_top=None, inplace=True)

    a = a[
        (a.obs.n_genes_by_counts >= 200) &
        (a.obs.total_counts >= 500) &
        (a.obs.pct_counts_mt < 15)
    ].copy()

    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    sc.pp.highly_variable_genes(a, n_top_genes=2500, flavor="seurat")

    sc.tl.pca(a, n_comps=30, mask_var="highly_variable")
    sc.pp.neighbors(a, n_neighbors=15, n_pcs=20)
    sc.tl.leiden(
        a, resolution=0.6, flavor="igraph",
        n_iterations=2, directed=False, random_state=42
    )

    panels = {"Muller": muller, **rivals}

    for name, genes in panels.items():
        sc.tl.score_genes(
            a, [g for g in genes if g in a.var_names],
            score_name=name, random_state=42
        )

    for cl in a.obs["leiden"].cat.categories:

        x = a.obs[a.obs["leiden"] == cl]

        scores = {k: x[k].mean() for k in panels}
        rival = max(v for k,v in scores.items() if k != "Muller")

        rows.append({
            "sample": s,
            "status": status,
            "cluster": cl,
            "n_cells": len(x),
            "Muller_score": scores["Muller"],
            "best_rival": max(
                (k for k in rivals),
                key=lambda k: scores[k]
            ),
            "best_rival_score": rival,
            "Muller_margin": scores["Muller"] - rival
        })

res = pd.DataFrame(rows)

print("TOP MÜLLER-LIKE CLUSTERS WITHIN EACH SAMPLE")
print("=" * 75)

display(
    res.sort_values(
        ["sample","Muller_margin"],
        ascending=[True,False]
    ).groupby("sample").head(5).round(3)
)

res.to_csv(
    HUMAN_DIR / "Human_atlas_PILOT_per_sample_cluster_scores.csv",
    index=False
)

print("\nSaved. STOP.")

In [ ]:
# HUMAN PILOT — direct marker audit of candidate Müller clusters

import numpy as np, pandas as pd, scanpy as sc

candidates = {
    1: ["6"],
    8: ["0", "1"],
    13: ["2", "14"]
}

muller = ["RLBP1","GLUL","SLC1A3","AQP4","KCNJ10","MLC1","SOX9","VIM"]
rod = ["RHO","NRL","GNAT1","PDE6A","SAG"]

rows = []

for s, status in [(1,"NON"), (8,"DM"), (13,"DR")]:

    a = sc.read_10x_mtx(
        HUMAN_DIR / "pilot" / f"Sample{s}_{status}",
        var_names="gene_symbols",
        make_unique=True
    )

    a.var["mt"] = a.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(a, qc_vars=["mt"], percent_top=None, inplace=True)

    a = a[
        (a.obs.n_genes_by_counts >= 200) &
        (a.obs.total_counts >= 500) &
        (a.obs.pct_counts_mt < 15)
    ].copy()

    raw = a.copy()

    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    sc.pp.highly_variable_genes(a, n_top_genes=2500, flavor="seurat")
    sc.tl.pca(a, n_comps=30, mask_var="highly_variable")
    sc.pp.neighbors(a, n_neighbors=15, n_pcs=20)
    sc.tl.leiden(
        a, resolution=0.6, flavor="igraph",
        n_iterations=2, directed=False, random_state=42
    )

    for cl in candidates[s]:

        keep = a.obs["leiden"].astype(str) == cl
        x = raw[keep]

        row = {
            "sample": s,
            "status": status,
            "cluster": cl,
            "n_cells": int(keep.sum())
        }

        for g in muller + rod:
            if g in x.var_names:
                v = np.asarray(x[:, g].X.toarray()).ravel()
                row[f"{g}_pct"] = 100 * (v > 0).mean()

        rows.append(row)

audit = pd.DataFrame(rows)

print("CANDIDATE MÜLLER MARKER DETECTION (%)")
print("=" * 80)
display(audit.round(1))

audit.to_csv(
    HUMAN_DIR / "Human_atlas_PILOT_candidate_Muller_marker_audit.csv",
    index=False
)

print("\nSaved. STOP.")

In [ ]:
# HUMAN PILOT — top defining genes of candidate Müller clusters

import pandas as pd, scanpy as sc

candidates = {
    1:  ["6"],
    8:  ["0", "1"],
    13: ["2", "14"]
}

rows = []

for s, status in [(1,"NON"), (8,"DM"), (13,"DR")]:

    a = sc.read_10x_mtx(
        HUMAN_DIR / "pilot" / f"Sample{s}_{status}",
        var_names="gene_symbols",
        make_unique=True
    )

    a.var["mt"] = a.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(a, qc_vars=["mt"], percent_top=None, inplace=True)

    a = a[
        (a.obs.n_genes_by_counts >= 200) &
        (a.obs.total_counts >= 500) &
        (a.obs.pct_counts_mt < 15)
    ].copy()

    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    sc.pp.highly_variable_genes(a, n_top_genes=2500, flavor="seurat")
    sc.tl.pca(a, n_comps=30, mask_var="highly_variable")
    sc.pp.neighbors(a, n_neighbors=15, n_pcs=20)
    sc.tl.leiden(
        a, resolution=0.6, flavor="igraph",
        n_iterations=2, directed=False, random_state=42
    )

    for cl in candidates[s]:

        sc.tl.rank_genes_groups(
            a,
            groupby="leiden",
            groups=[cl],
            reference="rest",
            method="wilcoxon"
        )

        top = sc.get.rank_genes_groups_df(a, group=cl).head(20)

        rows.append({
            "sample": s,
            "status": status,
            "cluster": cl,
            "top20_genes": ", ".join(top["names"])
        })

out = pd.DataFrame(rows)

print("CANDIDATE CLUSTER DEFINING GENES")
print("=" * 80)
display(out)

out.to_csv(
    HUMAN_DIR / "Human_atlas_PILOT_candidate_cluster_topmarkers.csv",
    index=False
)

print("\nSTOP.")

In [ ]:
# ============================================================
# HUMAN ATLAS — FULL 20-SAMPLE MÜLLER SCREEN
# ============================================================

import requests, numpy as np, pandas as pd, scanpy as sc

manifest = pd.read_csv(HUMAN_DIR / "Human_atlas_LOCKED_sample_donor_manifest.csv")

markers = ["RLBP1","GLUL","SLC1A3","AQP4","KCNJ10","MLC1","SOX9","VIM"]
FULL = HUMAN_DIR / "all20"
FULL.mkdir(exist_ok=True)

rows = []

for _, r in manifest.iterrows():

    s, status = int(r["sample"]), r["status"]
    folder = FULL / f"Sample{s}_{status}"
    folder.mkdir(exist_ok=True)

    # download if missing
    for name, col in [
        ("matrix.mtx.gz","matrix_id"),
        ("features.tsv.gz","features_id"),
        ("barcodes.tsv.gz","barcodes_id")
    ]:
        p = folder / name
        if not p.exists():
            with requests.get(
                f"https://ndownloader.figshare.com/files/{int(r[col])}",
                stream=True
            ) as resp:
                resp.raise_for_status()
                with open(p, "wb") as f:
                    for chunk in resp.iter_content(1024*1024):
                        f.write(chunk)

    a = sc.read_10x_mtx(folder, var_names="gene_symbols", make_unique=True)

    a.var["mt"] = a.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(a, qc_vars=["mt"], percent_top=None, inplace=True)

    a = a[
        (a.obs.n_genes_by_counts >= 200) &
        (a.obs.total_counts >= 500) &
        (a.obs.pct_counts_mt < 15)
    ].copy()

    raw = a.copy()

    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    sc.pp.highly_variable_genes(a, n_top_genes=2500, flavor="seurat")
    sc.tl.pca(a, n_comps=30, mask_var="highly_variable")
    sc.pp.neighbors(a, n_neighbors=15, n_pcs=20)
    sc.tl.leiden(
        a, resolution=0.6, flavor="igraph",
        n_iterations=2, directed=False, random_state=42
    )

    # save cell→cluster mapping
    a.obs[["leiden"]].to_csv(
        folder / "cluster_assignments.csv.gz"
    )

    for cl in a.obs["leiden"].cat.categories:

        keep = a.obs["leiden"] == cl
        x = raw[keep]

        pct = {}
        for g in markers:
            v = np.asarray(x[:, g].X.toarray()).ravel()
            pct[g] = 100*(v > 0).mean()

        pass_rule = (
            len(x) >= 30 and
            pct["RLBP1"] >= 20 and
            pct["GLUL"] >= 20 and
            pct["SLC1A3"] >= 20 and
            sum(v >= 20 for v in pct.values()) >= 5
        )

        rows.append({
            "sample": s,
            "donor": r["donor"],
            "status": status,
            "cluster": str(cl),
            "n_cells": len(x),
            **{f"{g}_pct": pct[g] for g in markers},
            "n_markers_ge20": sum(v >= 20 for v in pct.values()),
            "candidate_Muller": pass_rule
        })

screen = pd.DataFrame(rows)

print("MÜLLER CANDIDATES — ALL 20 SAMPLES")
print("=" * 80)

display(
    screen[screen["candidate_Muller"]]
    .sort_values(["sample","cluster"])
    .round(1)
)

print("\nCANDIDATE CLUSTERS PER SAMPLE")
display(
    screen.groupby("sample")["candidate_Muller"]
    .sum()
    .rename("n_candidate_clusters")
    .to_frame()
)

screen.to_csv(
    HUMAN_DIR / "Human_atlas_ALL20_Muller_cluster_screen.csv",
    index=False
)

print("\nSaved. STOP.")

In [ ]:
# ============================================================
# HUMAN ATLAS — FINAL MÜLLER LINEAGE-PURITY AUDIT
# Rod signal is descriptive, NOT an exclusion criterion
# ============================================================

import numpy as np, pandas as pd, scanpy as sc

FULL = HUMAN_DIR / "all20"
screen = pd.read_csv(
    HUMAN_DIR / "Human_atlas_ALL20_Muller_cluster_screen.csv"
)
cand = screen[screen["candidate_Muller"]].copy()

muller = ["RLBP1","GLUL","SLC1A3","AQP4","SOX9"]

panels = {
    "RPE":         ["RPE65","BEST1","RDH5","TYRP1","LRAT"],
    "Astrocyte":   ["GFAP","ALDH1L1","S100B"],
    "Microglia":   ["P2RY12","C1QA","C1QB","AIF1","CX3CR1"],
    "Endothelial": ["PECAM1","EMCN","CLDN5","KDR","VWF"],
    "Pericyte":    ["RGS5","PDGFRB","CSPG4","ABCC9"],
    "Bipolar":     ["VSX2","TRPM1","GRM6","PRKCA"],
    "Amacrine":    ["GAD1","GAD2","TFAP2A","SLC6A1"],
    "RGC":         ["RBPMS","POU4F1","POU4F2","SNCG"],
    "Cone":        ["ARR3","GNAT2","OPN1MW","OPN1SW"]
}

rod = ["RHO","NRL","GNAT1","PDE6A","SAG"]

def median_detection(a, keep, genes):
    g = [x for x in genes if x in a.var_names]
    X = a[keep, g].X
    return float(np.median(np.asarray((X > 0).mean(axis=0)).ravel() * 100))

rows = []

for s in sorted(cand["sample"].unique()):

    r = cand[cand["sample"] == s].iloc[0]
    folder = FULL / f"Sample{int(s)}_{r['status']}"

    a = sc.read_10x_mtx(folder, var_names="gene_symbols", make_unique=True)

    a.var["mt"] = a.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(a, qc_vars=["mt"], percent_top=None, inplace=True)
    a = a[
        (a.obs.n_genes_by_counts >= 200) &
        (a.obs.total_counts >= 500) &
        (a.obs.pct_counts_mt < 15)
    ].copy()

    cl = pd.read_csv(folder / "cluster_assignments.csv.gz", index_col=0)
    a.obs["leiden"] = cl.reindex(a.obs_names)["leiden"].astype(str)

    for cluster in cand.loc[cand["sample"] == s, "cluster"].astype(str):

        keep = a.obs["leiden"] == cluster

        M = median_detection(a, keep, muller)
        rivals = {k: median_detection(a, keep, v) for k,v in panels.items()}
        rival_name = max(rivals, key=rivals.get)
        rival_score = rivals[rival_name]

        rows.append({
            "sample": int(s),
            "donor": r["donor"],
            "status": r["status"],
            "cluster": cluster,
            "n_cells": int(keep.sum()),
            "Muller_core_pct": M,
            "best_nonRod_rival": rival_name,
            "rival_pct": rival_score,
            "margin": M - rival_score,
            "Rod_pct": median_detection(a, keep, rod)
        })

audit = pd.DataFrame(rows)

print("FINAL MÜLLER LINEAGE AUDIT")
print("=" * 75)
print("Candidate clusters:", len(audit))
print("Positive Müller-vs-nonRod margin:", (audit["margin"] > 0).sum())
print("Non-positive margin:", (audit["margin"] <= 0).sum())

print("\nLOWEST 20 MARGINS")
display(
    audit.sort_values("margin").head(20).round(1)
)

audit.to_csv(
    HUMAN_DIR / "Human_atlas_ALL20_Muller_lineage_purity_audit.csv",
    index=False
)

print("\nSaved. STOP.")

In [ ]:
# ============================================================
# HUMAN ATLAS — FREEZE MÜLLER CELLS + BUILD PSEUDOBULK COUNTS
# Primary rule: candidate Müller AND lineage-purity margin > 0
# ============================================================

import numpy as np, pandas as pd, scanpy as sc

FULL = HUMAN_DIR / "all20"

audit = pd.read_csv(
    HUMAN_DIR / "Human_atlas_ALL20_Muller_lineage_purity_audit.csv"
)
manifest = pd.read_csv(
    HUMAN_DIR / "Human_atlas_LOCKED_sample_donor_manifest.csv"
)

primary = audit[audit["margin"] > 0].copy()
primary.to_csv(
    HUMAN_DIR / "Human_atlas_FROZEN_Muller_clusters.csv",
    index=False
)

eye_counts, cell_rows, gene_index = {}, [], None

for _, r in manifest.iterrows():

    s, status = int(r["sample"]), r["status"]
    folder = FULL / f"Sample{s}_{status}"

    a = sc.read_10x_mtx(folder, var_names="gene_symbols", make_unique=True)

    a.var["mt"] = a.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(a, qc_vars=["mt"], percent_top=None, inplace=True)

    a = a[
        (a.obs.n_genes_by_counts >= 200) &
        (a.obs.total_counts >= 500) &
        (a.obs.pct_counts_mt < 15)
    ].copy()

    cl = pd.read_csv(folder / "cluster_assignments.csv.gz", index_col=0)
    a.obs["leiden"] = cl.reindex(a.obs_names)["leiden"].astype(str)

    keep_clusters = set(
        primary.loc[primary["sample"] == s, "cluster"].astype(str)
    )
    keep = a.obs["leiden"].isin(keep_clusters)

    m = a[keep]

    if gene_index is None:
        gene_index = m.var_names
    else:
        assert gene_index.equals(m.var_names)

    eye_counts[f"S{s}"] = np.asarray(m.X.sum(axis=0)).ravel()

    cell_rows.extend([
        {
            "cell": f"S{s}_{bc}",
            "sample": s,
            "donor": r["donor"],
            "status": status,
            "cluster": c
        }
        for bc, c in zip(m.obs_names, m.obs["leiden"])
    ])

# genes × eyes
eye_pb = pd.DataFrame(eye_counts, index=gene_index)

# genes × donors
donor_pb = {}
for donor in manifest["donor"].unique():
    samples = manifest.loc[manifest["donor"] == donor, "sample"]
    cols = [f"S{x}" for x in samples]
    donor_pb[donor] = eye_pb[cols].sum(axis=1)

donor_pb = pd.DataFrame(donor_pb)

pd.DataFrame(cell_rows).to_csv(
    HUMAN_DIR / "Human_atlas_FROZEN_Muller_cells.csv.gz",
    index=False
)

eye_pb.to_csv(
    HUMAN_DIR / "Human_atlas_FROZEN_Muller_eye_pseudobulk_counts.csv.gz"
)

donor_pb.to_csv(
    HUMAN_DIR / "Human_atlas_FROZEN_Muller_donor_pseudobulk_counts.csv.gz"
)

summary = (
    pd.DataFrame(cell_rows)
    .groupby(["sample","donor","status"])
    .size()
    .rename("Muller_cells")
    .reset_index()
)

print("FROZEN HUMAN MÜLLER POPULATION")
print("=" * 70)
print("Primary clusters:", len(primary))
print("Excluded clusters:", len(audit) - len(primary))
print("Samples retained:", summary["sample"].nunique())
print("Donors retained:", summary["donor"].nunique())
print("Total Müller cells:", summary["Muller_cells"].sum())

print("\nMÜLLER CELLS PER EYE")
display(summary)

print("\nMÜLLER CELLS PER DONOR")
display(
    summary.groupby(["donor","status"])["Muller_cells"]
    .sum()
    .to_frame()
)

print("\nSaved. STOP.")

In [ ]:
# ============================================================
# HUMAN ATLAS — RAM-SAFE FREEZE + PSEUDOBULK
# No Scanpy / no full atlas in memory
# ============================================================

from pathlib import Path
import gzip, gc
import numpy as np
import pandas as pd
from scipy.io import mmread

ROOT = Path("/content/drive/MyDrive/DR_Muller_AdaptiveReactive")
HUMAN_DIR = ROOT / "02_Human_scRNA_2026"
FULL = HUMAN_DIR / "all20"

manifest = pd.read_csv(
    HUMAN_DIR / "Human_atlas_LOCKED_sample_donor_manifest.csv"
)

audit = pd.read_csv(
    HUMAN_DIR / "Human_atlas_ALL20_Muller_lineage_purity_audit.csv"
)

# Freeze pre-specified primary rule
primary = audit[audit["margin"] > 0].copy()
primary.to_csv(
    HUMAN_DIR / "Human_atlas_FROZEN_Muller_clusters.csv",
    index=False
)

eye_counts = {}
cell_tables = []

for _, r in manifest.iterrows():

    s = int(r["sample"])
    status = r["status"]
    folder = FULL / f"Sample{s}_{status}"

    cl = pd.read_csv(
        folder / "cluster_assignments.csv.gz",
        index_col=0
    )
    cl["leiden"] = cl["leiden"].astype(str)

    keep_clusters = set(
        primary.loc[
            primary["sample"] == s, "cluster"
        ].astype(str)
    )

    keep_barcodes = cl.index[
        cl["leiden"].isin(keep_clusters)
    ]

    # tiny annotation files
    barcodes = pd.read_csv(
        folder / "barcodes.tsv.gz",
        header=None
    )[0].astype(str)

    features = pd.read_csv(
        folder / "features.tsv.gz",
        sep="\t", header=None
    )
    genes = features[1].astype(str)

    # locate selected cells
    pos = pd.Series(
        np.arange(len(barcodes)),
        index=barcodes
    )

    cols = (
        pos.reindex(keep_barcodes)
        .dropna()
        .astype(int)
        .to_numpy()
    )

    # load ONE sparse matrix only
    with gzip.open(folder / "matrix.mtx.gz", "rb") as f:
        X = mmread(f).tocsr()

    assert X.shape == (len(genes), len(barcodes))

    counts = np.asarray(
        X[:, cols].sum(axis=1)
    ).ravel()

    # collapse any duplicate gene symbols safely
    eye_counts[f"S{s}"] = (
        pd.Series(counts, index=genes)
        .groupby(level=0)
        .sum()
    )

    picked = cl.loc[keep_barcodes, ["leiden"]].copy()
    picked["cell"] = [f"S{s}_{x}" for x in picked.index]
    picked["sample"] = s
    picked["donor"] = r["donor"]
    picked["status"] = status
    cell_tables.append(picked.reset_index(drop=True))

    print(f"Sample {s:02d}: {len(cols):,} frozen Müller cells")

    del X, counts, cl, barcodes, features, pos, cols
    gc.collect()

# genes × eyes
eye_pb = pd.concat(eye_counts, axis=1).fillna(0).astype("int64")

# genes × donors
donor_pb = pd.DataFrame({
    donor: eye_pb[
        [f"S{x}" for x in
         manifest.loc[manifest["donor"] == donor, "sample"]]
    ].sum(axis=1)
    for donor in manifest["donor"].unique()
})

cells = pd.concat(cell_tables, ignore_index=True)

eye_pb.to_csv(
    HUMAN_DIR / "Human_atlas_FROZEN_Muller_eye_pseudobulk_counts.csv.gz"
)

donor_pb.to_csv(
    HUMAN_DIR / "Human_atlas_FROZEN_Muller_donor_pseudobulk_counts.csv.gz"
)

cells.to_csv(
    HUMAN_DIR / "Human_atlas_FROZEN_Muller_cells.csv.gz",
    index=False
)

summary = (
    cells.groupby(["sample","donor","status"])
    .size()
    .rename("Muller_cells")
    .reset_index()
)

print("\nFROZEN HUMAN MÜLLER POPULATION")
print("=" * 60)
print("Primary clusters:", len(primary))
print("Excluded clusters:", len(audit) - len(primary))
print("Samples retained:", summary["sample"].nunique())
print("Donors retained:", summary["donor"].nunique())
print("Total Müller cells:", len(cells))

print("\nMÜLLER CELLS PER DONOR")
display(
    summary.groupby(["donor","status"])["Muller_cells"]
    .sum()
    .to_frame()
)

print("\nSaved. STOP.")

In [ ]:
# ============================================================
# HUMAN ATLAS — LOCK MOUSE→HUMAN ORTHOLOG MAPPING
# ============================================================

import requests, pandas as pd, json, time

SIGDIR = ROOT / "signatures"

with open(SIGDIR / "GSE328966_LOCKED_discovery_signatures.json") as f:
    locked_signatures = json.load(f)

mouse_genes = sorted(set(
    g for genes in locked_signatures.values() for g in genes
))

# Mouse symbol -> HomoloGene
mouse_hits = {}

for g in mouse_genes:
    j = requests.get(
        "https://mygene.info/v3/query",
        params={
            "q": f"symbol:{g}",
            "species": "mouse",
            "fields": "symbol,entrezgene,homologene",
            "size": 1
        }
    ).json()

    hits = j.get("hits", [])
    if hits:
        mouse_hits[g] = hits[0]

# collect human Entrez ortholog IDs
pairs = []

for g, hit in mouse_hits.items():
    hg = hit.get("homologene", {})
    human_ids = [
        str(x[1]) for x in hg.get("genes", [])
        if x[0] == 9606
    ]

    for hid in human_ids:
        pairs.append((g, hid))

# human Entrez -> human symbol
human_symbol = {}

for _, hid in pairs:
    j = requests.get(
        f"https://mygene.info/v3/gene/{hid}",
        params={"fields": "symbol"}
    ).json()

    if "symbol" in j:
        human_symbol[hid] = j["symbol"]

orth = pd.DataFrame([
    {
        "mouse_symbol": m,
        "human_entrez": h,
        "human_symbol": human_symbol.get(h)
    }
    for m, h in pairs
]).dropna()

orth.to_csv(
    HUMAN_DIR / "Mouse_to_Human_orthologs_LOCKED.csv",
    index=False
)

# signature coverage in frozen human Müller pseudobulk
pb = pd.read_csv(
    HUMAN_DIR / "Human_atlas_FROZEN_Muller_donor_pseudobulk_counts.csv.gz",
    index_col=0
)

rows = []

for name, genes in locked_signatures.items():

    mapped = orth[
        orth["mouse_symbol"].isin(genes)
    ]["human_symbol"].drop_duplicates()

    present = [g for g in mapped if g in pb.index]

    rows.append({
        "signature": name,
        "mouse_genes": len(genes),
        "human_orthologs": len(mapped),
        "present_in_human_Muller": len(present),
        "coverage_pct": 100 * len(present) / max(len(genes), 1)
    })

coverage = pd.DataFrame(rows)

print("LOCKED MOUSE→HUMAN SIGNATURE COVERAGE")
print("=" * 70)
display(coverage.round(1))

coverage.to_csv(
    HUMAN_DIR / "Human_Muller_locked_signature_ortholog_coverage.csv",
    index=False
)

print("\nMapped mouse genes:", orth["mouse_symbol"].nunique(), "/", len(mouse_genes))
print("Saved. STOP.")

In [ ]:
# ============================================================
# HUMAN MÜLLER — LOCKED SIGNATURE TEST
# Donor-level; expected pattern: DM > NON and DM > DR
# ============================================================

import numpy as np, pandas as pd
from itertools import combinations

pb = pd.read_csv(
    HUMAN_DIR / "Human_atlas_FROZEN_Muller_donor_pseudobulk_counts.csv.gz",
    index_col=0
)
orth = pd.read_csv(HUMAN_DIR / "Mouse_to_Human_orthologs_LOCKED.csv")
manifest = pd.read_csv(HUMAN_DIR / "Human_atlas_LOCKED_sample_donor_manifest.csv")

status = (
    manifest[["donor","status"]]
    .drop_duplicates()
    .set_index("donor")["status"]
)

# log-CPM
cpm = pb.div(pb.sum(axis=0), axis=1) * 1e6
logcpm = np.log2(cpm + 0.5)

# gene-wise z across donors
z = logcpm.sub(logcpm.mean(axis=1), axis=0)
z = z.div(logcpm.std(axis=1, ddof=1).replace(0, np.nan), axis=0)

sigs = {
    "IFN": locked_signatures["early_IFN_alpha_beta"],
    "Retinoid": locked_signatures["early_retinoid_metabolism"],
    "ECM_GAG": locked_signatures["early_ECM_GAG_exploratory"],
    "Early184": locked_signatures["early_up_transient_all"],
    "Persistent50": locked_signatures["persistent_up_all_descriptive"],
    "Reversal19": locked_signatures["up_to_down_reversal_all_descriptive"]
}

# equal weight per original mouse gene
scores = {}
for name, mouse_genes in sigs.items():
    gene_scores = []
    for mg in mouse_genes:
        hg = orth.loc[orth.mouse_symbol == mg, "human_symbol"].unique()
        hg = [g for g in hg if g in z.index]
        if hg:
            gene_scores.append(z.loc[hg].mean(axis=0))
    scores[name] = pd.concat(gene_scores, axis=1).mean(axis=1)

scores = pd.DataFrame(scores)
scores["status"] = status

def exact_greater(a, b):
    a, b = np.asarray(a), np.asarray(b)
    obs = a.mean() - b.mean()
    x = np.r_[a, b]
    n = len(a)
    vals = []
    for idx in combinations(range(len(x)), n):
        mask = np.zeros(len(x), bool)
        mask[list(idx)] = True
        vals.append(x[mask].mean() - x[~mask].mean())
    return obs, np.mean(np.asarray(vals) >= obs - 1e-12)

rows = []

for sig in sigs:
    non = scores.loc[scores.status=="NON", sig]
    dm  = scores.loc[scores.status=="DM", sig]
    dr  = scores.loc[scores.status=="DR", sig]

    d1, p1 = exact_greater(dm, non)  # DM > NON
    d2, p2 = exact_greater(dm, dr)   # DM > DR

    rows.append({
        "signature": sig,
        "NON_mean": non.mean(),
        "DM_mean": dm.mean(),
        "DR_mean": dr.mean(),
        "DM_minus_NON": d1,
        "p_DM_gt_NON": p1,
        "DM_minus_DR": d2,
        "p_DM_gt_DR": p2
    })

res = pd.DataFrame(rows)

print("DONOR-LEVEL HUMAN MÜLLER SIGNATURE SCORES")
print("=" * 70)
display(
    scores.sort_values(["status"]).round(3)
)

print("\nLOCKED HUMAN TRANSLATION TEST")
print("=" * 70)
display(res.round(4))

scores.to_csv(
    HUMAN_DIR / "Human_Muller_LOCKED_signature_donor_scores.csv"
)
res.to_csv(
    HUMAN_DIR / "Human_Muller_LOCKED_signature_exact_tests.csv",
    index=False
)

print("\nExact minimum p-values:")
print("DM vs NON =", 1/84)
print("DM vs DR  =", 1/35)
print("\nSaved. STOP.")

In [ ]:
# ============================================================
# HUMAN ATLAS — FINAL ROBUSTNESS
# Equal-eye donor weighting + leave-one-DM-out sensitivity
# ============================================================

import json, numpy as np, pandas as pd
from itertools import combinations

eye = pd.read_csv(
    HUMAN_DIR / "Human_atlas_FROZEN_Muller_eye_pseudobulk_counts.csv.gz",
    index_col=0
)
orth = pd.read_csv(HUMAN_DIR / "Mouse_to_Human_orthologs_LOCKED.csv")
meta = pd.read_csv(HUMAN_DIR / "Human_atlas_LOCKED_sample_donor_manifest.csv")

with open(ROOT / "signatures/GSE328966_LOCKED_discovery_signatures.json") as f:
    locked_signatures = json.load(f)

sigs = {
    "IFN": locked_signatures["early_IFN_alpha_beta"],
    "Retinoid": locked_signatures["early_retinoid_metabolism"],
    "ECM_GAG": locked_signatures["early_ECM_GAG_exploratory"],
    "Early184": locked_signatures["early_up_transient_all"],
    "Persistent50": locked_signatures["persistent_up_all_descriptive"],
    "Reversal19": locked_signatures["up_to_down_reversal_all_descriptive"]
}

# normalize each eye first, then average paired eyes equally
logcpm = np.log2(eye.div(eye.sum(axis=0), axis=1) * 1e6 + 0.5)

donor_expr = {}
status = {}

for donor, m in meta.groupby("donor"):
    cols = [f"S{x}" for x in m["sample"]]
    donor_expr[donor] = logcpm[cols].mean(axis=1)
    status[donor] = m["status"].iloc[0]

donor_expr = pd.DataFrame(donor_expr)

z = donor_expr.sub(donor_expr.mean(axis=1), axis=0)
z = z.div(donor_expr.std(axis=1, ddof=1).replace(0, np.nan), axis=0)

scores = {}

for name, mouse_genes in sigs.items():
    gs = []
    for mg in mouse_genes:
        hg = orth.loc[orth.mouse_symbol == mg, "human_symbol"].unique()
        hg = [g for g in hg if g in z.index]
        if hg:
            gs.append(z.loc[hg].mean(axis=0))
    scores[name] = pd.concat(gs, axis=1).mean(axis=1)

scores = pd.DataFrame(scores)
scores["status"] = pd.Series(status)

def exact_gt(a, b):
    a, b = np.asarray(a), np.asarray(b)
    obs = a.mean() - b.mean()
    x = np.r_[a, b]
    vals = []
    for idx in combinations(range(len(x)), len(a)):
        mask = np.zeros(len(x), bool)
        mask[list(idx)] = True
        vals.append(x[mask].mean() - x[~mask].mean())
    return obs, np.mean(np.asarray(vals) >= obs - 1e-12)

rows = []

for sig in sigs:
    non = scores.loc[scores.status=="NON", sig]
    dm  = scores.loc[scores.status=="DM", sig]
    dr  = scores.loc[scores.status=="DR", sig]

    d1,p1 = exact_gt(dm, non)
    d2,p2 = exact_gt(dm, dr)

    # remove each DM donor once
    loo_non, loo_dr = [], []
    for d in ["D07","D08","D09"]:
        x = dm.drop(d)
        loo_non.append(x.mean() - non.mean())
        loo_dr.append(x.mean() - dr.mean())

    rows.append({
        "signature": sig,
        "NON": non.mean(),
        "DM": dm.mean(),
        "DR": dr.mean(),
        "DM-NON": d1,
        "p_DM>NON": p1,
        "DM-DR": d2,
        "p_DM>DR": p2,
        "LOO_DM-NON_min": min(loo_non),
        "LOO_DM-NON_max": max(loo_non),
        "LOO_DM-DR_min": min(loo_dr),
        "LOO_DM-DR_max": max(loo_dr)
    })

robust = pd.DataFrame(rows)

print("EQUAL-EYE + LEAVE-ONE-DM-OUT ROBUSTNESS")
print("=" * 80)
display(robust.round(3))

robust.to_csv(
    HUMAN_DIR / "Human_Muller_LOCKED_signature_robustness.csv",
    index=False
)

print("\nSaved. STOP.")